In [ ]:
# === Cell 1
import os, re, json, logging, hashlib, difflib
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import pandas as pd
import numpy as np
from openai import OpenAI, RateLimitError
import backoff
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpx2").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

# === GPT Model Selection ===  Default model used by all GPT calls (Stage 1 & 2).
OPENAI_MODEL = "gpt-6-luna"
OPENAI_REASONING_EFFORT = "low"
os.environ["OPENAI_MODEL"] = OPENAI_MODEL
print(f"[config] GPT model set to: {OPENAI_MODEL}")

# === OpenAI API Key Selection ===
# If OPENAI_API_KEY_PROJECT is set, use it (e.g., your shared lab/project key). Otherwise, fall back to OPENAI_API_KEY (user's personal/business key).
PROJECT_OPENAI_KEY = os.getenv("OPENAI_API_KEY_PROJECT")
if PROJECT_OPENAI_KEY:
    client = OpenAI(api_key=PROJECT_OPENAI_KEY)
    print("[config] Using OPENAI_API_KEY_PROJECT (project/shared key).")
else:     # Optional: warn if API key missing
    if not os.getenv("OPENAI_API_KEY"):
        print("[warning] OPENAI_API_KEY not found — GPT calls will fail unless it’s set.")
    client = OpenAI()
    print("[config] Using OPENAI_API_KEY (default key).")

# Set variables: base_dir = the notebook's folder (Jupyter runs notebooks there). factors.csv, foodcategories.json and item_weights.json are read from it.
base_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()
# ---- SET LOCAL_INPUT TO THE NAME OF THE PROCUREMENT SPREADSHEET YOU ARE ASSESSING !!!!!!!!!!!!!!!!!!!!!!!!----
# On the web service (GitHub/Docker) the uploaded spreadsheet is always saved as input.csv (results are named after the uploaded file: see base_filename),
# so input.csv is used whenever it exists. Running locally, set LOCAL_INPUT.
LOCAL_INPUT = "usfoods.csv"   # a .csv, or a workbook (.xlsx: its "Data" sheet) such as usfoodsplus.xlsx (<input name>plus.xlsx) from add_certifier_dropdowns.ipynb
# LOCAL_INPUT = "northcoastseafood.csv"
input_filename = base_dir / "input.csv"
if not input_filename.exists():
    input_filename = base_dir / LOCAL_INPUT
print(f"[config] Input file: {input_filename}")
# Base name for outputs: the input file's name, or on the web service the uploaded file's name (BASE_NAME, set by the workflow)
base_filename = (re.sub(r"[^A-Za-z0-9_-]", "_", os.environ.get("BASE_NAME", ""))[:60] if input_filename.name == "input.csv" else "") or Path(input_filename).stem
# All result files go here; the web service returns the contents of this folder.
output_dir = base_dir / "output"
output_dir.mkdir(exist_ok=True)

# Set the input worksheet file type. See File Preparation section of documentation
# Choose: "auto" | "type01" | "type02" | "type03"
file_type = "auto"

# Units used by the purchase worksheet. "auto" uses whichever unit most invoice lines are in
# (kg / g / L / ml  vs  lb / oz / gal / qt). Stages 1-2 then keep weights in that unit (the
# ingredients and categories files have columns ending in _kg or _lbs) and Stage 3 converts
# only where needed. A type02 worksheet whose weight column has no unit is read as kg if the
# column heading mentions kg (e.g. "product_weight (kg)"), otherwise as lbs -- unless a unit
# is set explicitly here, in which case that unit is assumed.
# Choose: "auto" | "lb" | "kg"
INPUT_WEIGHT_UNIT = "auto"

# Which impact files Stage 3 writes (both are always calculated).
# Choose: "both" | "metric" | "imperial"
OUTPUT_FILES = "both"

# START_MODE SWITCH to run full Notebook or resume part way through the process.
# So, for example, you could review the ingredients and/or category output file, make edits and then
# restart the process using your edited file, if you set START_MODE to "after_ingredients" or "after_categories"
# Start at: "full" | "after_ingredients" | "after_categories"
START_MODE = "full"

# Optionally adjustable to eliminate ingredients that are a very minor percent of total ingredient weight
CATEGORY_PERCENT_MIN = 2.1

# factors.csv is the file with all the conversion factors from weight to impacts; item_weights.json has typical item weights
# foodcategories.json is a file that contains prior ingredient:category combinations
# that can be checked for use, avoiding need for calls to AI if the ingredient is already present
factors_filename    = base_dir / "factors.csv"
categories_filename = base_dir / "foodcategories.json"
item_weights_filename = base_dir / "item_weights.json"
# Session epoch for overwrite policy (v100g)
import time
_V100_SESSION_START = globals().get("_V100_SESSION_START")
if _V100_SESSION_START is None:
    _V100_SESSION_START = time.time()
    globals()["_V100_SESSION_START"] = _V100_SESSION_START
    
# === Cell 1: Purpose: Set core filenames, modes, and global switches; verify required assets.
# Inputs:
#   - input_filename (user-specified CSV file), factors.csv (for ALLOWED_CATEGORIES), foodcategories.json.
#   - constants and dictionaries (UOM_TO_LBS, CATEGORY_WEIGHT_MULTIPLIERS, etc.)
# Outputs:
#   - Global variables: BASE, START_MODE, CATEGORY_SOURCE, USE_CATEGORY_WEIGHT_MULTIPLIERS, etc.
# Outputs: In-memory globals (e.g., ALLOWED_CATEGORIES, FOODCATS) used by later cells.
#   - Logging configuration and printout of active settings.
# Key processes:
# • Imports core libraries and configures logging (INFO by default).
# • Reads/sets: input_filename, file_type ('auto'|'type01'|'type02'|'type03'), and base_filename.
# • Sets high-level run control START_MODE ('full' | 'after_ingredients' | 'after_categories').
# • Loads foodcategories.json (exact name → category map) preserving insertion order.
# • Loads factors.csv and composes ALLOWED_CATEGORIES (the “closed set” used by GPT).
#
# Notes:
# • For OpenAI to be used later, the environment variable OPENAI_API_KEY must be set before running Cell 5/6.
# • Changing input_filename or START_MODE here controls what the final Runner (Cell 9) will do.

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# some different files being used for input_filename in testing
# "hh100g.csv"
# "HH1.csv"
# "testfoods80.csv"
# "usfoods.csv"
# "usfoods_veg.csv  just a few vegan items
# "in1b_fpoint02.csv"      # 213k
# "in2b_usfoods02.csv"     # 226k
# "in2b_usfoods26.csv"     # 15k
# "in3_jpaige01.csv"       # 7k
# "in4b_jpolep01.csv"      # 19k
# "in5b_ncsea01.csv"       # 15k
# "in6b_jamix01.csv"       # 23k
# "baldor100a.csv"         # 40k
# "HH112in.csv"
# baldor103f.csv

def _load_foodcategories(json_path: Path) -> Dict[str, str]:
    if not Path(json_path).exists():
        logging.warning("No foodcategories.json found — using empty mapping.")
        return {}
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    mapping = {}
    if isinstance(data, dict):
        for k,v in data.items():
            mapping[str(k).strip().lower()] = str(v).strip()
    elif isinstance(data, list):
        for rule in data:
            if isinstance(rule, dict) and 'from' in rule and 'to' in rule:
                cat = str(rule['to']).strip()
                for src in rule['from']:
                    mapping[str(src).strip().lower()] = cat
    return mapping

FOODCATS = _load_foodcategories(categories_filename)

# Load category list from factors.csv
if not Path(factors_filename).exists():
    raise FileNotFoundError(f"factors file not found: {factors_filename}")
_factors_df = pd.read_csv(factors_filename, encoding="utf-8")
if "category" not in _factors_df.columns:
    raise ValueError("factors.csv must include a 'category' column")
FACTORS_CATEGORY_SET = set(str(x).strip() for x in _factors_df["category"].dropna().unique())

# Designed to help ensure that Food Categories selected for ingredients matches factors in factors.csv
ALLOWED_CATEGORIES = sorted(FACTORS_CATEGORY_SET)

# If worksheet does NOT show real weights (lbs or kg) for products with quantities expressed in fluid volumes such as fl oz 
# or pint, quart, gallon, ml, liter then set this to True in order for weights to be properly calculated. The multipliers
# here convert from the default, weight of water, to weight of products that may be lighter or heavier than water.
# --- Configuration Toggles and Data ---
USE_CATEGORY_WEIGHT_MULTIPLIERS = False

# These values represent the approximate specific gravity of the food, 
# which is the ratio of the food's density to the density of water (1.0).
CATEGORY_WEIGHT_MULTIPLIERS = {
    "skim milk": 1.034,  # Heavier due to solids, less fat
    "dairy milk": 1.030, # Slightly lighter than skim milk
    "buttermilk": 1.032, # Similar to skim milk
    "soy milk": 1.020,
    "oat milk": 1.015,
    "almond milk": 1.010,
    "rice milk": 1.008,
    "cream": 0.995,      # Lighter than water due to high fat content
    "olive oil": 0.915,  # All pure oils are less dense than water
    "palm oil": 0.890,
    "rapeseed/canola oil": 0.915,
    "soybean oil": 0.920,
    "sunflower oil": 0.918,
    "vegetable oils": 0.925  # General range for mixed oils
}

print(f"Updated multipliers to reflect Specific Gravity (Weight relative to Water = 1.0).")

# === Configuration: foodcategories.json Update Mode ===
# Controls how newly discovered ingredient-category pairs are handled

# Options:
#   "append"    - Add new pairs to foodcategories.json
#   "disabled"  - Don't store new pairs anywhere (pure lookup mode)
#   "separate"  - Add new pairs to newcategories.json (review before merging)

FOODCAT_UPDATE_MODE = "separate"  # Change to "disabled" or "append" as needed

# Validation
if FOODCAT_UPDATE_MODE not in ["append", "disabled", "separate"]:
    raise ValueError(f"FOODCAT_UPDATE_MODE must be 'append', 'disabled', or 'separate', got: {FOODCAT_UPDATE_MODE}")

if INPUT_WEIGHT_UNIT not in ("auto", "lb", "kg"):
    raise ValueError(f"INPUT_WEIGHT_UNIT must be 'auto', 'lb' or 'kg', got: {INPUT_WEIGHT_UNIT}")
if OUTPUT_FILES not in ("both", "metric", "imperial"):
    raise ValueError(f"OUTPUT_FILES must be 'both', 'metric' or 'imperial', got: {OUTPUT_FILES}")

print(f"[Config] foodcategories.json update mode: {FOODCAT_UPDATE_MODE}")
print(f"[Config] INPUT_WEIGHT_UNIT={INPUT_WEIGHT_UNIT}  OUTPUT_FILES={OUTPUT_FILES}")
if FOODCAT_UPDATE_MODE == "separate":
    print(f"[Config] New categories will be written to: {base_dir / 'newcategories.json'}")

# Shares of purchased weight that are certified, plant-based, local or from social impact suppliers
# (Cell 8b, shares_report_*.txt). Certifications, Local, Social Impact Supplier, Plant Based and
# Explanation are read from the columns add_certifier_dropdowns.ipynb adds to a worksheet; plant-based is
# also judged for every product in Stage 1. The worksheet's headings decide the scoring: AW / Env / Labor
# columns = Anchors in Action (GFPP levels from the AIA sheet of certifiers.ods; Program year and Nutrition
# checklist points (%) from the worksheet's last two columns); AASHE Organic / Animal Welfare / Environment / Labor /
# Small Producers / Seafood columns = AASHE STARS OP-07 (qualifying options from the STARS sheet).
# The first of these files that exists is used.
CERTIFIERS_FILES = [base_dir / "certifiers.ods", Path.home() / "certifiers.ods"]
# Plant-based products are graded by processing: 1 = whole or minimally processed, 2 = plant cooking
# ingredients (oils, sugars, syrups, starches, vinegar), 3 = simple processed foods and meat / dairy
# alternatives, 4 = ultra-processed. "Plant-based" in the shares report counts categories 1 up to this
# number (the report also shows each cutoff): 3 matches AASHE STARS v3.0; 4 counts every all-plant product.
PLANT_BASED_MAX_CATEGORY = 3


In [ ]:
# === Cell 2: Utilities ===
# Purpose: Provide shared helper functions used throughout the FoodImpacts workflow.
# Includes:
#   - read_input_csv / safe_read_csv: fault-tolerant CSV loading with encoding, delimiter
#     (comma / semicolon / tab) and decimal-comma detection; Excel / ODS workbooks are read too, and
#     certification columns are set aside in INPUT_ATTRIBUTES
#   - parse_number: numbers written as 2.5, 2,5, 1,234.5 or 1.234,5
#   - parse_qty_unit: extract numeric quantity and unit of measure from product strings
#   - result_out / result_in: result files are written to (and read back from) the output folder
#   - UOM_BASE / to_weight: convert quantity-unit pairs to the run's weight unit (kg or lb);
#     resolve_weight_unit: decide that unit (INPUT_WEIGHT_UNIT in Cell 1)
#   - _norm: text normalization helper for category matching
#   - cooked_split: canned / cooked beans, rice and pasta as their dry equivalent plus water
# Inputs: function arguments (varies)
# Outputs: function return values; no direct file writes.
# Behavior:
#   Supports robust data ingestion, normalization, and measurement conversion.

LB_TO_KG = 0.45359237
KG_TO_LB = 1.0 / LB_TO_KG

# Every unit -> (amount, system). Metric units are given in kg and US units in lb, so a line
# is only converted when its system differs from the run's weight unit (WEIGHT_UNIT below).
# Volumes assume the density of water (1 L = 1 kg; 1 US gal = 8.345 lb); see
# USE_CATEGORY_WEIGHT_MULTIPLIERS in Cell 1 for liquids lighter or heavier than water.
UOM_BASE = {}
def _add_units(names, amount, system):
    for n in names:
        UOM_BASE[n] = (amount, system)

# --- US weight ---
_add_units(["LB", "LBS", "POUND", "POUNDS"], 1.0, "lb")
_add_units(["OZ", "OUNCE", "OUNCES"], 1/16.0, "lb")
# --- US volume (water density) ---
_add_units(["GAL", "GALLON", "GALLONS"], 8.345, "lb")                 # 1 US gal water = 8.345 lb
_add_units(["QT", "QUART", "QUARTS"], 2.086, "lb")                    # 8.345 / 4
_add_units(["PT", "PINT", "PINTS"], 1.043, "lb")                      # 8.345 / 8
_add_units(["FLUID OZ", "FL OZ", "FLOZ"], 0.065, "lb")                # 8.345 / 128
# --- metric weight ---
_add_units(["KG", "KGS", "KILO", "KILOS", "KILOGRAM", "KILOGRAMS", "KILOGRAMME", "KILOGRAMMES"], 1.0, "kg")
_add_units(["G", "GR", "GRS", "GRAM", "GRAMS", "GRAMME", "GRAMMES"], 0.001, "kg")
# --- metric volume (water density) ---
_add_units(["L", "LT", "LTR", "LTRS", "LITER", "LITERS", "LITRE", "LITRES"], 1.0, "kg")
_add_units(["DL"], 0.1, "kg")
_add_units(["CL"], 0.01, "kg")
_add_units(["ML", "MILLILITER", "MILLILITERS", "MILLILITRE", "MILLILITRES"], 0.001, "kg")

# Legacy lookup tables (same units, expressed entirely in lb or entirely in kg)
UOM_TO_LBS = {u: a * (1.0 if s == "lb" else KG_TO_LB) for u, (a, s) in UOM_BASE.items()}
UOM_TO_KG  = {u: a * (1.0 if s == "kg" else LB_TO_KG) for u, (a, s) in UOM_BASE.items()}

# The run's weight unit: "lb" or "kg". Set by resolve_weight_unit() when Stage 1 reads the worksheet.
WEIGHT_UNIT = None

def weight_suffix(unit: Optional[str] = None) -> str:
    """Column suffix for a weight unit: 'kg' or 'lbs'."""
    return "kg" if (unit or WEIGHT_UNIT) == "kg" else "lbs"

def _uom_key(uom) -> str:
    k = str(uom).strip().upper().rstrip(".")
    return re.sub(r"[\s.]+", " ", k)

def uom_system(uom) -> Optional[str]:
    """'kg' for metric units, 'lb' for US units, None if not a known weight/volume unit."""
    if uom is None or (isinstance(uom, float) and np.isnan(uom)):
        return None
    hit = UOM_BASE.get(_uom_key(uom))
    return hit[1] if hit else None

def convert_weight(value, from_unit: str, to_unit: Optional[str] = None):
    """Convert a weight between 'lb' and 'kg' (no-op when the units match)."""
    to_unit = to_unit or WEIGHT_UNIT or "lb"
    if value is None:
        return None
    if from_unit == to_unit:
        return value
    return value * (LB_TO_KG if from_unit == "lb" else KG_TO_LB)

def to_weight(qty, uom, unit: Optional[str] = None):
    """Weight of qty x uom in `unit` (default: the run's WEIGHT_UNIT). None if the unit is unknown."""
    if qty is None or uom is None or pd.isna(qty) or pd.isna(uom):
        return None
    hit = UOM_BASE.get(_uom_key(uom))
    if hit is None:
        return None
    amount, system = hit
    return convert_weight(float(qty) * amount, system, unit or WEIGHT_UNIT or "lb")

def to_lbs(qty: float, uom: str):
    """Kept for compatibility with older code: weight in pounds."""
    return to_weight(qty, uom, "lb")

# Unit words written next to a number in product descriptions ("6x1 kg", "12/32 OZ", "#10")
_METRIC_MENTION_RX = re.compile(
    r"\d\s*(?:KGS?|KILOS?|KILOGRAMS?|G|GR|GRS|GRAMS?|L|LT|LTR|LITRES?|LITERS?|ML|CL|DL)\b", re.I)
_US_MENTION_RX = re.compile(r"\d\s*(?:LBS?|POUNDS?|OZ|GAL|GALLONS?|QT|PT)\b|#\s?10\b", re.I)

def count_unit_mentions(texts) -> Tuple[int, int]:
    """(metric lines, US lines) among product descriptions; lines naming both are not counted."""
    n_metric = n_us = 0
    for t in texts:
        s = str(t)
        m = bool(_METRIC_MENTION_RX.search(s))
        u = bool(_US_MENTION_RX.search(s))
        n_metric += m and not u
        n_us += u and not m
    return n_metric, n_us

def resolve_weight_unit(n_metric: int, n_us: int, context: str = "") -> str:
    """Decide the run's weight unit from INPUT_WEIGHT_UNIT (Cell 1) and the line counts; sets WEIGHT_UNIT."""
    global WEIGHT_UNIT
    setting = str(globals().get("INPUT_WEIGHT_UNIT", "auto")).lower()
    where = f" in {context}" if context else ""
    if setting in ("lb", "kg"):
        unit = setting
        print(f"[units] Weight unit '{unit}' (set by INPUT_WEIGHT_UNIT); found {n_metric} metric "
              f"and {n_us} US-unit lines{where}.")
        other = n_us if unit == "kg" else n_metric
        if other > (n_metric + n_us) / 2:
            print(f"[units] WARNING: most lines{where} are in the other unit system -- "
                  f"check INPUT_WEIGHT_UNIT.")
    elif n_metric > n_us:
        unit = "kg"
        print(f"[units] Detected kg: {n_metric} metric lines, {n_us} US-unit lines{where}.")
    else:
        unit = "lb"
        if n_metric == 0 and n_us == 0:
            print(f"[units] WARNING: no units recognized{where}; using lb. "
                  f"Set INPUT_WEIGHT_UNIT in Cell 1 if the weights are kilograms.")
        elif n_metric == n_us:
            print(f"[units] WARNING: equal numbers of metric and US-unit lines{where} ({n_us}); using lb.")
        else:
            print(f"[units] Detected lb: {n_us} US-unit lines, {n_metric} metric lines{where}.")
    WEIGHT_UNIT = unit
    return unit

# Decimal separator of the most recently read input worksheet ("," for European-style files)
CSV_DECIMAL = "."

def parse_number(s) -> Optional[float]:
    """Parse 2.5, 2,5, 1,234.5, 1.234,5 or 1 234,5. A lone comma followed by exactly three
    digits ("1,000") is a thousands separator unless the worksheet uses decimal commas."""
    if s is None:
        return None
    if isinstance(s, (int, float, np.integer, np.floating)):
        return None if pd.isna(s) else float(s)
    t = str(s).strip().replace("\u00a0", "").replace(" ", "")
    if not t or t.lower() in ("nan", "none"):
        return None
    if "," in t and "." in t:
        if t.rfind(",") > t.rfind("."):
            t = t.replace(".", "").replace(",", ".")
        else:
            t = t.replace(",", "")
    elif "," in t:
        if CSV_DECIMAL != "," and re.fullmatch(r"-?\d{1,3}(?:,\d{3})+", t):
            t = t.replace(",", "")
        else:
            t = t.replace(",", ".")
    try:
        return float(t)
    except ValueError:
        return None

def num_series(s: pd.Series) -> pd.Series:
    """Numeric version of a column that may hold text numbers with decimal commas."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(s.map(parse_number), errors="coerce")

def parse_qty_unit(text: str):
    if not isinstance(text, str): return (None, None)
    m = re.search(r"(?P<qty>\d+(?:[\.,]\d+)*)\s*(?P<unit>[A-Za-z]+)\b", text)
    if not m: return (None, None)
    qty = parse_number(m.group('qty'))
    unit = m.group('unit').upper()
    return qty, unit

def norm_text(s: str) -> str:
    s = str(s or "").lower()
    s = re.sub(r"[^a-z0-9\s/\-]", " ", s)
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

def _normalize_colnames(cols):
    out = []
    for c in cols:
        c2 = str(c).replace("\u00A0"," ").strip().lower()
        c2 = re.sub(r"[^\w]+", "_", c2)
        c2 = re.sub(r"_+", "_", c2).strip("_")
        out.append(c2)
    return out

def _sniff_csv_format(path) -> Tuple[str, str]:
    """(delimiter, decimal) of a CSV file: delimiter , ; tab or |; decimal '.' or ','."""
    import csv
    raw = Path(path).read_bytes()[:131072]
    text = None
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            text = raw.decode(enc)
            break
        except Exception:
            continue
    lines = [l for l in (text or "").splitlines() if l.strip()][:60]
    if not lines:
        return ",", "."
    best, best_score = ",", 0.0
    for d in [",", ";", "\t", "|"]:
        counts = [len(next(csv.reader([l], delimiter=d))) - 1 for l in lines]
        mode = max(set(counts), key=counts.count)
        if mode <= 0:
            continue
        score = mode * counts.count(mode) / len(counts)
        if score > best_score:
            best, best_score = d, score
    decimal = "."
    if best != ",":
        n_comma = n_dot = 0
        for l in lines[1:]:
            for f in next(csv.reader([l], delimiter=best)):
                f = f.strip()
                if re.fullmatch(r"-?\d+,\d+", f):
                    n_comma += 1
                elif re.fullmatch(r"-?\d+\.\d+", f):
                    n_dot += 1
        if n_comma > n_dot:
            decimal = ","
    return best, decimal

# Columns that add_certifier_dropdowns.ipynb adds to a worksheet (normalized heading -> name in result files).
# They are set aside when the input worksheet is read (INPUT_ATTRIBUTES, same row index as the worksheet),
# so they cannot affect format detection; Stage 1 copies them onto each product's rows.
# Only these exact headings count, so a worksheet's own columns with similar names (e.g. a university's
# "Plant Based" or "Explanation") are left alone. A worksheet has either the Anchors in Action columns (AW food
# group, AW, Env, No routine antibiotics, Labor, Local (GFPP level), Social Impact Supplier) or the six AASHE
# columns (AASHE STARS, with Local Credit and Social Impact Supplier as y columns); a user may fill several
# AASHE columns for one product.
ATTRIBUTE_COLUMNS = {
    "aw_food_group": "aw_food_group", "aw": "aw", "env": "env", "labor": "labor",
    "no_routine_antibiotics_y_cert_or_y": "antibiotics",        # No routine antibiotics (y-cert or y)
    "aashe_organic": "aashe_organic", "aashe_animal_welfare": "aashe_animal_welfare",
    "aashe_environment": "aashe_environment", "aashe_labor": "aashe_labor",
    "aashe_small_producers": "aashe_small_producers", "aashe_seafood": "aashe_seafood",
    "local_gfpp_level": "local",                                # Local (GFPP level)        (Anchors in Action)
    "local_credit_enter_y_if_yes": "local",                     # Local Credit (enter y if yes)       (AASHE)
    "social_impact_supplier": "social_impact",                  # Social Impact Supplier    (Anchors in Action)
    "social_impact_supplier_enter_y_if_yes": "social_impact",   # Social Impact Supplier (enter y if yes) (AASHE)
    "plant_based_y_or_1_4": "plant_based_user",                 # Plant Based (y or 1-4)
    "explanation_optional": "explanation",                      # Explanation (optional)
    # whole-file settings (Anchors in Action), read by Cell 8b; set aside here only so they cannot affect
    # format detection, and not copied onto products
    "program_year": "program_year",                             # Program year
    "nutrition_checklist_points": "nutrition_points",           # Nutrition checklist points (%)
}
INPUT_ATTRIBUTES = None

# The web upload page sends the worksheet as a CSV without its blank and "Total" rows, and adds a
# "Worksheet row" column with each line's row number in the uploaded worksheet. It is set aside on reading
# (INPUT_SHEET_ROWS) so result files and the report give the row numbers the user sees.
SHEET_ROW_HEADING = "worksheet_row"
INPUT_SHEET_ROWS = None

def reset_sheet_rows():
    """Forget the Worksheet row column and heading row of a previous input (called before the input worksheet is read)."""
    global INPUT_SHEET_ROWS, INPUT_HEADING_ROW
    INPUT_SHEET_ROWS = None
    INPUT_HEADING_ROW = 0

def sheet_row(i) -> int:
    """Worksheet row number of input line i (the index of the worksheet as read): the Worksheet row column
    when there is one, else i + FI_ROW_OFFSET (headings in row 1) + the lines above the headings, if any."""
    if INPUT_SHEET_ROWS is not None:
        v = INPUT_SHEET_ROWS.get(i)
        if v is not None and pd.notna(v):
            return int(v)
    return int(i) + int(os.getenv("FI_ROW_OFFSET", "2")) + INPUT_HEADING_ROW

def src_row_no(src_row, i) -> int:
    """The worksheet line a standardized row came from (its __src_row), else i."""
    v = src_row.get("__src_row", i) if hasattr(src_row, "get") else i
    return int(i) if v is None or pd.isna(v) else int(v)

def _attribute_name(norm_heading: str) -> Optional[str]:
    """Name in result files for a normalized worksheet heading, or None if it is not an attribute column."""
    return ATTRIBUTE_COLUMNS.get(norm_heading)

def _split_attributes(df: pd.DataFrame) -> pd.DataFrame:
    """Move the attribute columns of the input worksheet to INPUT_ATTRIBUTES (blank cells -> ''); if two
    headings mean the same column, the first is used. A Worksheet row column goes to INPUT_SHEET_ROWS."""
    global INPUT_ATTRIBUTES, INPUT_SHEET_ROWS
    for c, n in zip(df.columns, _normalize_colnames(df.columns)):
        if n == SHEET_ROW_HEADING:
            INPUT_SHEET_ROWS = pd.to_numeric(df[c], errors="coerce")
            df = df.drop(columns=[c])
            break
    found = {}
    for c, n in zip(df.columns, _normalize_colnames(df.columns)):
        a = _attribute_name(n)
        if a and a not in found.values():
            found[c] = a
    if not found:
        INPUT_ATTRIBUTES = None
        return df
    attrs = df[list(found)].rename(columns=found)
    INPUT_ATTRIBUTES = attrs.apply(lambda s: s.map(lambda v: "" if pd.isna(v) else str(v).strip()))
    return df.drop(columns=list(found))

# Vendor reports often start with a title or date lines above the column headings. The heading row is the
# first of the first 30 rows that is filled across at least 60% of the widest row and is at most a quarter
# numbers (the same rule as the web upload page); the lines above it are skipped. Row numbers in result
# files still match the worksheet (INPUT_HEADING_ROW = lines skipped).
INPUT_HEADING_ROW = 0
_NUMBER_CELL_RX = re.compile(r"[-+(]?\s*[$\u20ac\u00a3]?\s*[\d][\d.,\s]*%?\)?")

def find_heading_row(rows) -> int:
    """Index of the column-heading row among the first rows of a sheet (lists of cells); 0 if unsure."""
    def txt(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return ""
        return str(v).strip()
    rows = [[txt(v) for v in r] for r in list(rows)[:60]]
    widest = max((sum(1 for c in r if c) for r in rows), default=0)
    for i, r in enumerate(rows[:30]):
        cells = [c for c in r if c]
        numeric = sum(1 for c in cells if _NUMBER_CELL_RX.fullmatch(c))
        if len(cells) >= max(2, int(np.ceil(widest * 0.6))) and numeric <= len(cells) * 0.25:
            return i
    return 0

def _csv_first_rows(path, sep: str, n: int = 60) -> list:
    """The first n lines of a CSV as lists of cells (blank lines kept, so positions are line numbers)."""
    import csv, itertools
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            with open(path, newline="", encoding=enc) as f:
                return list(itertools.islice(csv.reader(f, delimiter=sep), n))
        except Exception:
            continue
    return []

_HEADING_NOTED = set()

def _note_heading_row(path, h: int):
    global INPUT_HEADING_ROW
    INPUT_HEADING_ROW = int(h)
    if h and (str(path), h) not in _HEADING_NOTED:   # the input is read more than once; say it once
        _HEADING_NOTED.add((str(path), h))
        print(f"[input] {Path(path).name}: column headings found in row {h + 1}; "
              f"the {h} line{'s' if h != 1 else ''} above {'them' if h != 1 else 'it'} skipped.")

# "Total" lines of a vendor report are not products (the web upload page leaves them out too)
_TOTAL_LINE_RX = re.compile(r"(?:(?:grand|sub)\s*-?\s*)?totals?\s*:?|grand\s*total\b.*", re.I)

def is_total_line(text) -> bool:
    """True for a product cell reading Total, Totals, Subtotal, Grand Total ... (not 'TOTAL cereal')."""
    return bool(_TOTAL_LINE_RX.fullmatch(str(text).strip()))

def _read_workbook(path, header=0) -> pd.DataFrame:
    """The "Data" sheet of an Excel / ODS workbook (else its first sheet)."""
    engine = "odf" if Path(path).suffix.lower() == ".ods" else None
    sheets = pd.ExcelFile(path, engine=engine).sheet_names
    sheet = "Data" if "Data" in sheets else sheets[0]
    return pd.read_excel(path, sheet_name=sheet, engine=engine, header=header)

def read_input_csv(path, remember_format: bool = True) -> pd.DataFrame:
    """Read a CSV whose delimiter may be , ; tab or | and whose decimals may be commas, or an Excel / ODS
    workbook (.xlsx .xlsm .xls .ods). Column names are left as-is (see safe_read_csv for normalized names).
    For the input worksheet (remember_format=True) the ATTRIBUTE_COLUMNS are moved to INPUT_ATTRIBUTES."""
    global CSV_DECIMAL
    if Path(path).suffix.lower() in (".xlsx", ".xlsm", ".xls", ".ods"):
        h = find_heading_row(_read_workbook(path, header=None).head(60).values.tolist())
        df = _read_workbook(path, header=h)
        if remember_format:
            CSV_DECIMAL = "."
            _note_heading_row(path, h)
            df = _split_attributes(df)
        return df
    sep, dec = _sniff_csv_format(path)
    if remember_format:
        CSV_DECIMAL = dec
    kwargs = {"sep": sep, "decimal": dec}
    h = find_heading_row(_csv_first_rows(path, sep))
    if h:
        kwargs["skiprows"] = list(range(h))
    if remember_format:
        _note_heading_row(path, h)
    if dec == ",":
        kwargs["thousands"] = "."
    if sep != "," or dec != ".":
        print(f"[csv] {Path(path).name}: delimiter {sep!r}, decimal {dec!r}")
    df = None
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            break
        except Exception:
            continue
    if df is None:
        df = pd.read_csv(path, **kwargs)
    return _split_attributes(df) if remember_format else df

def safe_read_csv(path, remember_format: bool = True):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    df = read_input_csv(path, remember_format=remember_format)
    df.columns = _normalize_colnames(df.columns)
    return df

def result_out(name) -> Path:
    """Path for writing a result file: the output folder (output_dir in Cell 1), created if needed."""
    d = Path(globals().get("output_dir") or Path(globals().get("base_dir", ".")) / "output")
    d.mkdir(parents=True, exist_ok=True)
    return d / name

def result_in(name) -> Path:
    """Path for reading a result file (e.g. an edited ingredients/categories file for START_MODE):
    the output folder first, then the notebook's folder."""
    p = result_out(name)
    if p.exists():
        return p
    alt = Path(globals().get("base_dir", ".")) / name
    return alt if alt.exists() else p

def save_csv(df, path: str):
    df.to_csv(path, index=False, encoding="utf-8")
    logging.info(f"Wrote: {path} ({len(df):,} rows)")

def load_if_exists(path: str):
    p = Path(path)
    if p.exists():
        logging.info(f"Loading existing file: {p}")
        return safe_read_csv(p)
    return None

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

def fuzzy_choice(text: str, choices: List[str], cutoff: float = 0.86):
    if not choices: return None
    best = None; best_score = 0.0
    for c in choices:
        r = difflib.SequenceMatcher(None, text, c).ratio()
        if r > best_score:
            best, best_score = c, r
    return best if best_score >= cutoff else None

# === Diagnostics: micro logger (safe to live in Utilities cell) ===
import time, os
from contextlib import contextmanager

DEBUG_TIMING = True  # flip to False to silence all timing output

# psutil is optional; we'll degrade gracefully if it's not available
try:
    import psutil
    _PROC = psutil.Process(os.getpid())
    def _mem_mb():
        try:
            return _PROC.memory_info().rss / (1024*1024)
        except Exception:
            return float('nan')
except Exception:
    def _mem_mb():
        return float('nan')

@contextmanager
def log_time(label: str):
    """Usage:
    with log_time("Stage 3: edible-yield"):
        # code here
    """
    if not DEBUG_TIMING:
        yield
        return
    t0 = time.perf_counter()
    m0 = _mem_mb()
    print(f"[timing] ▶ {label} ... (RSS {m0:,.1f} MB)")
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        m1 = _mem_mb()
        print(f"[timing] ✔ {label} in {dt:,.3f}s (ΔRSS {m1 - m0:+.1f} MB)")

# === GPT bridge used by Stage 1 & Stage 2 (improved) ===

_GPT_CLIENT = None

def _get_openai_client(api_key: str):
    """Cache the OpenAI client so we don't recreate it for every call."""
    global _GPT_CLIENT
    if _GPT_CLIENT is None:
        from openai import OpenAI
        _GPT_CLIENT = OpenAI(api_key=api_key)
    return _GPT_CLIENT


def _call_gpt(messages, model=None, temperature=0, timeout=60):
    """
    messages: [{"role":"system"/"user"/"assistant","content": "..."}]
    Returns plain string content.
    Reuses an existing helper if present; otherwise calls OpenAI directly.
    """

    # 1) Compat aliases to reuse your existing helper if present
    for alias in ("call_gpt", "gpt_chat", "openai_chat", "chat_gpt", "gpt_call"):
        fn = globals().get(alias)
        if callable(fn):
            return fn(messages, model=model, temperature=temperature)

    # 2) OpenAI Chat Completions fallback
    import os

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not set and no existing GPT helper found.")

    use_model = model or OPENAI_MODEL

    client = _get_openai_client(api_key)

    request_kwargs = {
        "model": use_model,
        "messages": messages,
        "timeout": timeout,
    }

    if use_model == "gpt-6-luna":
        request_kwargs["reasoning_effort"] = OPENAI_REASONING_EFFORT

    resp = client.chat.completions.create(**request_kwargs)

    return resp.choices[0].message.content

# ------------------------------------------------------------------
# Readable fish_cut labels for shellfish and cephalopods.
# Stage 3's FISH_CUT_YIELDS curves are keyed on finfish cut names ("Whole / Round"
# ... "Skinless Fillet"). For shrimp, crab, lobster, octopus and squid those names
# are only positions on that species' curve, so Stage 1 writes these labels instead
# (e.g. peeled tail-on shrimp shows "Shrimp: peeled, tail-on", not "Skin-On Fillet")
# and Stage 3 maps each label back to its curve point. The finfish names are still
# accepted, so older or hand-edited files work unchanged.
# ------------------------------------------------------------------
SHELLFISH_CUT_LABELS = {
    "shrimp":  {"Whole / Round": "Shrimp: whole (head-on, shell-on)", "Dressed": "Shrimp: headless, shell-on",
                "Skin-On Fillet": "Shrimp: peeled, tail-on", "Skinless Fillet": "Shrimp: peeled, tail-off"},
    "crab":    {"Whole / Round": "Crab: whole", "H&G": "Crab: legs, claws or clusters in shell",
                "Skinless Fillet": "Crab: meat"},
    "lobster": {"Whole / Round": "Lobster: whole", "Steaks / Cutlets": "Lobster: tails or claws in shell",
                "Skinless Fillet": "Lobster: meat"},
    "octopus": {"Whole / Round": "Octopus: whole", "Drawn": "Octopus: whole, cleaned",
                "Skinless Fillet": "Octopus: tentacles or pieces"},
    "squid":   {"Whole / Round": "Squid: whole", "Skinless Fillet": "Squid: cleaned (tubes, rings, tentacles)"},
}

# ------------------------------------------------------------------
# Canned and cooked beans, rice and pasta (Stage 2; Stage 3 and Cell 8b for categories files not split yet).
# factors.csv gives "beans and pulses (dried)" and "rice" per kg as bought dry, and "wheat/rye" per kg of
# bread, dry pasta and baked goods. Canned or cooked, most of their weight is water taken up in cooking, so
# cooked_split() splits such an ingredient into its dry equivalent (same category) and water. Shares are the
# cooked / dry ratio of calories per kg (USDA FoodData Central): beans and pulses 0.38-0.45, rice about 0.35,
# pasta about 0.42 of dry pasta = 0.55 of the bread-based wheat/rye factor. Change these numbers to adjust.
# ------------------------------------------------------------------
COOKED_DRY_SHARE = {
    "beans and pulses (dried)": 0.40,
    "rice": 0.35,
    "wheat/rye (bread/pasta/baked goods)": 0.55,
}
# Plain canned beans: drained beans are usually 55-65% of a can's weight. A higher bean percent from Stage 1
# is cut to this, and the rest counted as water too.
CANNED_BEANS_MAX_PERCENT = 65
# Product names that mean the food was bought cooked, and names that mean it was bought dry
_COOKED_PRODUCT_RX = re.compile(r"\b(?:canned|cnd|cn|can|cans|in brine|in water|cooked|ckd|refried|baked beans?|"
                                r"hummus|houmous|soups?|chili|ready to eat|rte|ready to serve|stuffed|steamed)\b")
# ...plus, by category: frozen beans / pulses are fresh green ones (lima, fava) and frozen rice is cooked;
# beans in burgers and burritos, and pasta in salads or macaroni & cheese, are cooked
_COOKED_PRODUCT_EXTRA_RX = {
    "beans and pulses (dried)": re.compile(r"\b(?:frozen|iqf|fresh|salads?|burgers?|patty|patties|burritos?|"
                                           r"enchiladas?|tamales?|quesadillas?)\b"),
    "rice": re.compile(r"\b(?:frozen|iqf|salads?|sushi|burritos?|macaroni (?:and )?cheese|mac (?:and )?cheese)\b"),
    "wheat/rye (bread/pasta/baked goods)": re.compile(r"\b(?:salads?|macaroni (?:and )?cheese|mac (?:and )?cheese)\b"),
}
_DRY_PRODUCT_RX = re.compile(r"\b(?:dried|dry|raw|uncooked|dehydrated|instant|flakes?|flour|mix)\b")
# Ingredients split, by category: whole beans / pulses, rice, pasta and noodles (not breading, flour etc.)
_COOKED_INGREDIENT_RX = {
    "beans and pulses (dried)": re.compile(r"\b(?:beans?|peas?|chickpeas?|garbanzos?|lentils?|pulses?|legumes?|"
                                           r"dal|dhal)\b"),
    "rice": re.compile(r"\brice\b"),
    "wheat/rye (bread/pasta/baked goods)": re.compile(
        r"\b(?:pasta|noodles?|macaroni|spaghetti|penne|lasagn[ae]|tortellini|ravioli|fettuccine|linguine|rotini|"
        r"ziti|orzo|couscous|gnocchi|shells|elbows?)\b"),
}
_NOT_WHOLE_RX = re.compile(r"\b(?:flour|starch|protein|powder|isolate|concentrate|fib(?:er|re)|bran|syrup|vinegar|"
                           r"oil|milk|crisps?|crackers?|cakes?|paper|cereal|sauce|paste|extract|puffed|bread|"
                           r"breading|crumbs?)\b")
_PLAIN_BEANS_HEAD_RX = re.compile(r"^(?:beans?|peas?|chickpeas?|garbanzos?|lentils?)$")
_CANNED_RX = re.compile(r"\b(?:canned|cnd|cn|can|cans|in brine|in water)\b")
_BEANS_IN_SAUCE_RX = re.compile(r"\b(?:baked|refried|chili|sauce|seasoned|soups?|salad|pork|tomato|bbq|barbecue|"
                                r"molasses|maple|vegetarian|ranch|hummus)\b")
DRY_EQUIVALENT_SUFFIX = " (dry equivalent)"

# ------------------------------------------------------------------
# Broth and stock (same places as the canned / cooked split). foodcategories.json files "chicken broth" under
# chicken, "beef stock" under beef and so on, but ready-to-use broth is nearly all water. cooked_split() counts a
# broth ingredient as this share of its category (the rest as water): kcal per kg of the broth (USDA FoodData
# Central, SR Legacy, ready-to-serve: chicken broth 60, beef broth 70, clam liquid 20, vegetable broth 50) divided
# by the category's kcal per kg in factors.csv. Categories not listed are not split. Concentrates (bases, pastes,
# powders, bouillon, cubes, dehydrated stock) are not split; condensed soups, diluted 1:1, count twice the share.
# ------------------------------------------------------------------
BROTH_EQUIVALENT_SHARE = {
    "chicken": 0.03,
    "turkey": 0.03,
    "beef": 0.05,
    "bivalves": 0.08,
    "other vegetables": 0.22,
}
_BROTH_INGREDIENT_RX = re.compile(r"\b(?:broth|stock|consomm[e\u00e9])\b")
_BROTH_CONCENTRATE_RX = re.compile(r"\b(?:base|paste|powder(?:ed)?|concentrated?|concentrates|bouillon|granules?|"
                                   r"granulated|cubes?|dehydrated|dried|dry|demi|glace|reduction|extract|seasoning|"
                                   r"mix)\b")
_CONDENSED_RX = re.compile(r"\b(?:condensed|cond|cnds)\b")

# Stock is made mostly from bones, with some clinging meat, trimmings and skin, and its fat is skimmed off, so
# these numbers are kept modest so as not to overstate impacts:
# - Demi-glace (a ready sauce, not a paste) is stock reduced about 4:1: its stock counts this many times the
#   broth share above (beef / veal 20%, chicken 12%), the rest as water.
DEMI_GLACE_STOCK_FACTOR = 4
# - Soup bases, pastes, bouillon and powders are mostly salt, sugar, starch and flavorings: their meat, stock and
#   seafood ingredients together count as at most this percent of the product's weight; the rest of their weight
#   counts as "salt and other base ingredients" (category additive: no impacts). Animal fat listed separately
#   keeps counting as animal fat.
BASE_ANIMAL_MAX_PERCENT = 25
BASE_OTHER_INGREDIENT = "salt and other base ingredients"
_DEMI_RX = re.compile(r"\b(?:demi|glace)\b")
_BASE_PRODUCT_RX = re.compile(r"\b(?:base|bases|bouillon|granules?)\b")
_BASE_FORM_RX = re.compile(r"\b(?:paste|powder(?:ed)?|concentrated?|concentrates|dehydrated)\b")
_BASE_KIND_RX = re.compile(r"\b(?:soups?|broth|stock|gravy|sauce|demi|glace|jus)\b")
# meat and seafood categories of factors.csv (as SHARES_ANIMAL_CATEGORIES in Cell 8b, without dairy, eggs, fats)
_BASE_MEAT_CATEGORIES = {"beef", "chicken", "goat", "lamb/mutton", "pork", "turkey", "unidentified meat"}
_BASE_SEAFOOD_CAT_RX = re.compile(
    r"\bfish|bivalve|anchov|salmon|tuna|cod\b|haddock|pollock|sardine|halibut|mahi|catfish|tilapia|trout|bass|"
    r"hake|sole|flounder|whiting|carp|conger|milkfish|mullet|sauries|smelt|amberjack|unagi|barramundi|jacks|"
    r"pangasius|shrimp|crab|lobster|octopus|squid|mackerel|herring")

def is_base_product(product_text) -> bool:
    """A soup / sauce base, bouillon or similar concentrate, by its product name."""
    pr = norm_text(product_text)
    return bool(_BASE_PRODUCT_RX.search(pr) or (_BASE_FORM_RX.search(pr) and _BASE_KIND_RX.search(pr)))

def _is_meat_or_seafood_category(cat) -> bool:
    c = str(cat or "").strip().lower()
    return c in _BASE_MEAT_CATEGORIES or bool(_BASE_SEAFOOD_CAT_RX.search(c))

def base_cap(df: pd.DataFrame):
    """In soup bases, pastes, bouillon and powders, cut the meat / seafood ingredient rows to BASE_ANIMAL_MAX_PERCENT
    of the product together and add a row 'salt and other base ingredients' (category additive) for the rest.
    Products already capped are left alone. Returns (DataFrame, number of products capped)."""
    if df is None or df.empty or not {"ingredient", "category", "product", "percent"} <= set(df.columns):
        return df, 0
    key = "row_sheet" if "row_sheet" in df.columns else "product"
    pct = pd.to_numeric(df["percent"], errors="coerce")
    meat = df["category"].map(_is_meat_or_seafood_category) & pct.gt(0)
    base = df["product"].map(is_base_product)
    done = df["ingredient"].astype(str).eq(BASE_OTHER_INGREDIENT)
    scale_cols = [c for c in df.columns if c == "percent" or c.startswith(("ingredient_weight_", "weight_"))]
    add, n = [], 0
    df = df.copy()
    for _, idx in df[base & meat].groupby(key).groups.items():
        rows = df.index.isin(df[df[key] == df.loc[idx[0], key]].index)
        if done[rows].any():
            continue
        total = float(pct[idx].sum())
        if total <= BASE_ANIMAL_MAX_PERCENT:
            continue
        frac = BASE_ANIMAL_MAX_PERCENT / total
        extra = df.loc[idx[0]].copy()
        for col in scale_cols:
            v = pd.to_numeric(df.loc[idx, col], errors="coerce")
            if v.notna().any():
                extra[col] = round(float(v.sum()) * (1 - frac), 4)
                df.loc[idx, col] = (v * frac).round(4)
        extra["ingredient"] = BASE_OTHER_INGREDIENT
        if "ingredient_key" in extra.index:
            extra["ingredient_key"] = norm_text(BASE_OTHER_INGREDIENT)
        extra["category"] = "additive"
        if "match_method" in extra.index:
            extra["match_method"] = "rule_base"
            df.loc[idx, "match_method"] = "rule_base"
        add.append(extra)
        n += 1
    if not n:
        return df, 0
    out = pd.concat([df, pd.DataFrame(add, columns=df.columns)], ignore_index=True)
    # keep each product's rows together (the new row follows its product's rows)
    out["__order"] = range(len(out))
    out["__first"] = out.groupby(key, sort=False)["__order"].transform("min")
    out = out.sort_values(["__first", "__order"], kind="stable")
    return out.drop(columns=["__order", "__first"]).reset_index(drop=True).infer_objects(), n

def cooked_split(df: pd.DataFrame):
    """Split canned / cooked bean, rice and pasta ingredient rows into '<ingredient> (dry equivalent)' (same
    category) and 'water (in <ingredient>)' (category water), match_method rule_cooked, and broth / stock rows
    into '<ingredient> (<category> equivalent)' and water, match_method rule_broth (demi-glace: x
    DEMI_GLACE_STOCK_FACTOR), then caps the meat / seafood in soup bases (base_cap, rule_base); percent and ingredient
    weights are shared out, so a product's weight stays the same. Rows already split are left alone.
    Returns (DataFrame, number of rows split)."""
    if df is None or df.empty or not {"ingredient", "category", "product"} <= set(df.columns):
        return df, 0
    shares = {norm_text(k): v for k, v in COOKED_DRY_SHARE.items()}
    broth_shares = {norm_text(k): v for k, v in BROTH_EQUIVALENT_SHARE.items()}
    ing_rx = {norm_text(k): v for k, v in _COOKED_INGREDIENT_RX.items()}
    extra_rx = {norm_text(k): v for k, v in _COOKED_PRODUCT_EXTRA_RX.items()}
    beans = norm_text("beans and pulses (dried)")
    pct = pd.to_numeric(df["percent"], errors="coerce") if "percent" in df.columns else pd.Series(float("nan"), index=df.index)
    fracs = []
    for cat, ing, prod, p in zip(df["category"], df["ingredient"], df["product"], pct):
        c, i, pr = norm_text(cat) if isinstance(cat, str) else "", norm_text(ing), norm_text(prod)
        f = None
        done = str(ing).rstrip().endswith(" equivalent)")   # already split
        if (c in shares and not done
                and ing_rx.get(c) is not None and ing_rx[c].search(i) and not _NOT_WHOLE_RX.search(i)
                and (_COOKED_PRODUCT_RX.search(pr) or extra_rx[c].search(pr)) and not _DRY_PRODUCT_RX.search(pr)):
            f = shares[c]
            head = norm_text(str(prod).split(",")[0])
            if (c == beans and _PLAIN_BEANS_HEAD_RX.match(head) and _CANNED_RX.search(pr)
                    and not _BEANS_IN_SAUCE_RX.search(pr) and pd.notna(p) and p > CANNED_BEANS_MAX_PERCENT):
                f *= CANNED_BEANS_MAX_PERCENT / p
            f = (f, "cooked")
        elif (c in broth_shares and not done and _BROTH_INGREDIENT_RX.search(i)
                and not _BROTH_CONCENTRATE_RX.search(i)
                and not _BROTH_CONCENTRATE_RX.search(_DEMI_RX.sub(" ", pr))):
            mult = DEMI_GLACE_STOCK_FACTOR if _DEMI_RX.search(pr) else (2 if _CONDENSED_RX.search(pr) else 1)
            f = (min(1.0, broth_shares[c] * mult), "broth")
        fracs.append(f)
    n = sum(f is not None for f in fracs)
    if not n:
        return base_cap(df)
    scale = [c for c in df.columns if c == "percent" or c.startswith(("ingredient_weight_", "weight_"))]
    rows = []
    for (_, r), fk in zip(df.iterrows(), fracs):
        if fk is None:
            rows.append(r)
            continue
        f, kind = fk
        suffix = DRY_EQUIVALENT_SUFFIX if kind == "cooked" else f" ({r['category']} equivalent)"
        method = "rule_cooked" if kind == "cooked" else "rule_broth"
        for name, frac, cat in ((f"{r['ingredient']}{suffix}", f, r["category"]),
                                (f"water (in {r['ingredient']})", 1 - f, "water")):
            s = r.copy()
            s["ingredient"] = name
            if "ingredient_key" in s.index:
                s["ingredient_key"] = norm_text(name)
            for col in scale:
                v = pd.to_numeric(s[col], errors="coerce")
                if pd.notna(v):
                    s[col] = round(float(v) * frac, 4)
            s["category"] = cat
            if "match_method" in s.index:
                s["match_method"] = method
            rows.append(s)
    out, n_base = base_cap(pd.DataFrame(rows, columns=df.columns).reset_index(drop=True).infer_objects())
    return out, n + n_base


In [ ]:
# === Cell 3: Load & Standardize (Enhanced Version) ===
# Purpose: Detect vendor format (type01 / type02 / type03) and normalize data to a common schema.
# Inputs:
#   - input_filename: raw procurement CSV
#   - file_type: explicit ('type01'/'type02'/'type03') or 'auto' for autodetection
# Dependencies:
#   - Utilities from Cell 2 (safe_read_csv, parse_qty_unit, to_weight, resolve_weight_unit)
# Outputs:
#   - standardized DataFrame with columns:
#       ['product', 'description', 'weight_kg' or 'weight_lbs', 'qty' (if available)] !!!!!
#     The weight unit is the worksheet's own (INPUT_WEIGHT_UNIT in Cell 1; "auto" = majority of lines).
# Behavior:
#   - Reads raw file using safe_read_csv()
#   - If file_type='auto', uses comprehensive detection strategies
#   - Dispatches to handle_type01/02/03 to standardize column names and compute weights
#   - Returns a DataFrame ready for ingredient extraction (Stage 1).

import pandas as pd
import numpy as np
import logging
import re
from typing import Optional

def detect_vendor_fingerprint(df: pd.DataFrame) -> Optional[str]:
    """
    Detect specific vendor formats based on unique characteristics
    """
    cols_str = ' '.join(str(c).lower() for c in df.columns)
    
    # Get sample of first column data for content analysis
    first_col_sample = df.iloc[:min(50, len(df)), 0].astype(str).str.lower() if len(df) > 0 else pd.Series()
    first_col_text = ' '.join(first_col_sample.tolist())
    
    vendor_signatures = {
        "type01": [
            lambda df: 'quantity (base unit)' in cols_str,
            lambda df: 'qty (base unit)' in cols_str,
            lambda df: bool(re.search(r'quantity.*base.*unit', cols_str))
        ],
        
        "type02": [
            lambda df: 'net wght shipped' in cols_str,
            lambda df: 'total weight' in cols_str and 'item #' in cols_str,
            lambda df: 'qty shipped' in cols_str and 'total weight' in cols_str,
            lambda df: 'item description' in cols_str and not 'qty sold' in cols_str
        ],
        
        "type03": [
            lambda df: 'qty sold cw' in cols_str,
            lambda df: 'stock unit' in cols_str,
            lambda df: 'item description' in cols_str and 'stock unit' in cols_str,
        ]
    }
    
    for file_type, checks in vendor_signatures.items():
        matches = sum(1 for check in checks if check(df))
        if matches >= 2:  # Need at least 2 matching signatures
            logging.info(f"Vendor fingerprint detected: {file_type} ({matches} signatures matched)")
            return file_type
    
    return None

def statistical_type_detection(df: pd.DataFrame) -> str:
    """
    Use statistical analysis to detect type when column names are ambiguous
    """
    if len(df) == 0:
        logging.warning("Empty dataframe, defaulting to type01")
        return "type01"
    
    # Analyze first non-header column (usually product/description)
    first_col = df.iloc[:, 0].dropna().astype(str)
    if len(first_col) == 0:
        logging.warning("No data in first column, defaulting to type01")
        return "type01"
    
    # Count different pattern occurrences
    scores = {"type01": 0, "type02": 0, "type03": 0}
    
    # Check for type01 patterns (separate qty/unit values like "80.000 OZ")
    qty_unit_separate = first_col.str.match(r'^\d+(?:[.,]\d+)?\s+[A-Z]{1,}$', case=False)
    scores["type01"] += qty_unit_separate.sum() * 3
    
    # Also check in second column if exists
    if len(df.columns) > 1:
        second_col = df.iloc[:, 1].dropna().astype(str)
        qty_unit_in_second = second_col.str.match(r'^\d+(?:[.,]\d+)?\s+[A-Z]{1,}$', case=False)
        scores["type01"] += qty_unit_in_second.sum() * 3
    
    # Check for type02 patterns (clean product names with separate weight column)
    clean_products = first_col.str.match(r'^[A-Za-z][A-Za-z\s\-\'\"&,\.]+$')
    has_weight_col = any('weight' in str(c).lower() or 'wght' in str(c).lower() for c in df.columns)
    has_numeric_col = False
    
    if len(df.columns) > 1:
        for col in df.columns[1:]:
            sample = df[col].dropna().head(10)
            numeric_ratio = pd.to_numeric(sample, errors='coerce').notna().sum() / max(len(sample), 1)
            if numeric_ratio > 0.7:
                has_numeric_col = True
                break
    
    if has_weight_col and has_numeric_col:
        scores["type02"] += clean_products.sum() * 2
        scores["type02"] += 5  # Bonus for having weight column
    
    # Check for type03 patterns (embedded weights in product description)
    # Pattern examples: "CHEDDAR CLOTHBOUND AGED CABOT 30 LB MC", "BLUE BAYLEY HAZEN BLUE 7 LB JASPER HILL"
    embedded_patterns = [
        r'\d+\s*LB\s+',  # "30 LB MC"
        r'\d+\s*[X×]\s*\d+(?:[.,]\d+)?\s*(?:OZ|LB|KG|G|L|ML|CL)\b',  # "12X12 OZ", "6 x 1 kg"
        r'\d+/\d+\s*(?:LB|OZ|GAL|KG|G|L)\b',  # "6/2.5 LB"
        r'\d+(?:[.,]\d+)?\s*(?:LB|OZ|GAL|QT|PT|KG|G|L|ML|CL)\s+[A-Z]',  # "5 LB CABOT", "2,5 KG EMMENTAL"
        r'[A-Z]+\s+\d+(?:[.,]\d+)?\s*(?:LB|OZ|KG|G)\s+[A-Z]'  # "TUB 5 LB CABOT"
    ]
    
    for pattern in embedded_patterns:
        matches = first_col.str.contains(pattern, case=False, regex=True)
        scores["type03"] += matches.sum() * 2
    
    # Look for case/carton patterns common in type03
    case_patterns = first_col.str.contains(r'\d+/\d+|\d+X\d+|\d+\s*(?:CS|CTN|CASE|EACH|EA)', case=False)
    scores["type03"] += case_patterns.sum()
    
    # Additional type03 indicators: mixed units in product names
    unit_variety = first_col.str.contains(r'\b(?:LB|OZ|GAL|KG|G|L|ML|CL|EACH|EA|CTN|CS|PCS|STK)\b', case=False)
    if unit_variety.sum() > len(first_col) * 0.3:
        scores["type03"] += 3
    
    best_type = max(scores, key=scores.get)
    max_score = scores[best_type]
    confidence = max_score / max(len(first_col), 1)
    
    # Log all scores for debugging
    logging.debug(f"Type detection scores: {scores}")
    
    if confidence < 0.2 and max_score < 3:
        logging.warning(f"Low confidence detection: {best_type} with score {max_score} (confidence: {confidence:.2f})")
    else:
        logging.info(f"Statistical detection: {best_type} with score {max_score} (confidence: {confidence:.2f})")
    
    return best_type

def validate_type_detection(df: pd.DataFrame, file_type: str) -> bool:
    """
    Try parsing a sample with the detected type to validate
    """
    try:
        # Use a small sample to test
        sample_df = df.head(min(10, len(df))).copy()
        
        if file_type == "type01":
            result = handle_type01(sample_df, verbose=False)
        elif file_type == "type02":
            result = handle_type02(sample_df, verbose=False)
        elif file_type == "type03":
            result = handle_type03(sample_df, verbose=False)
        else:
            return False

        # Check if we got reasonable results
        if '__w_amt' not in result.columns:
            return False

        valid_weights = (result['__w_amt'].notna() & result['__w_uom'].map(lambda u: uom_system(u) is not None)).sum()
        valid_ratio = valid_weights / max(len(result), 1)
        
        # Type03 might have lower valid ratio due to complex parsing
        threshold = 0.2 if file_type == "type03" else 0.3
        
        return valid_ratio >= threshold
        
    except Exception as e:
        logging.debug(f"Validation failed for {file_type}: {str(e)}")
        return False

def autodetect_type_improved(df: pd.DataFrame) -> str:
    """
    Enhanced autodetection with multiple pattern matching strategies
    """
    if len(df.columns) == 0:
        logging.error("Auto-detect: Input file has no columns")
        raise ValueError("Auto-detect failed: CSV has no columns")
    
    # Normalize column names for comparison
    cols_lower = [str(c).lower().strip() for c in df.columns]
    cols_map = {str(c).lower().strip(): c for c in df.columns}
    
    # === Type01 Detection ===
    type01_col_patterns = [
        r'quantity.*base.*unit',
        r'qty.*base',
        r'product.*weight'
    ]
    
    for pattern in type01_col_patterns:
        if any(re.search(pattern, col) for col in cols_lower):
            # Additional validation: check if values contain quantity+unit
            target_col = None
            for col in cols_lower:
                if re.search(pattern, col):
                    target_col = cols_map[col]
                    break
            
            if target_col and target_col in df.columns:
                sample = df[target_col].dropna().head(20).astype(str)
                if len(sample) > 0:
                    # Check for patterns like "80.000 OZ", "4.400 LB", "84.000 lb"
                    qty_unit_pattern = r'^\d+(?:[.,]\d+)?\s+[A-Za-z]{1,10}$'
                    matches = sample.str.match(qty_unit_pattern, case=False)
                    if matches.sum() > len(sample) * 0.3:
                        logging.info("Auto-detect: Type01 - found quantity+unit combined format in column data")
                        return "type01"
    
    # === Type02 Detection ===
    has_weight_col = any('weight' in col or 'wght' in col for col in cols_lower)
    has_qty_col = any('qty' in col or 'quantity' in col for col in cols_lower)
    has_unit_col = any('unit' in col or 'uom' in col for col in cols_lower)
    
    # Type02 typically has weight column with numeric values and separate qty
    if has_weight_col:
        weight_cols = [cols_map[c] for c in cols_lower if 'weight' in c or 'wght' in c]
        if weight_cols and weight_cols[0] in df.columns:
            sample_weights = df[weight_cols[0]].dropna().head(20)
            if len(sample_weights) > 0:
                numeric_ratio = pd.to_numeric(sample_weights, errors='coerce').notna().sum() / len(sample_weights)
                if numeric_ratio > 0.7:
                    # Additional check: if we have qty column or no unit column, likely type02
                    if has_qty_col or not has_unit_col:
                        logging.info("Auto-detect: Type02 - separate numeric weight column detected")
                        return "type02"
    
    # === Type03 Detection ===
    # Look for product/description columns
    product_cols = []
    for priority_term in ['product', 'description', 'item']:
        for col in cols_lower:
            if priority_term in col and cols_map[col] not in product_cols:
                product_cols.append(cols_map[col])
                break
    
    # If no product column found, use first column
    if not product_cols and len(df.columns) > 0:
        product_cols = [df.columns[0]]
    
    if product_cols and product_cols[0] in df.columns:
        sample_products = df[product_cols[0]].dropna().head(30).astype(str)
        
        if len(sample_products) > 0:
            # Look for embedded weight patterns characteristic of type03
            embedded_patterns = [
                r'\d+\s*(?:LB|KG)\b',  # "30 LB", "10 KG"
                r'\d+\s*[X×]\s*\d+(?:[.,]\d+)?\s*(?:OZ|LB|KG|G|L|ML|CL)\b',  # "12X12 OZ", "6 x 1 kg"
                r'\d+/\d+\s*(?:LB|OZ|GAL|KG|G|L)\b',  # "6/2.5 LB"
                r'\b\d+(?:[.,]\d+)?\s*(?:LB|OZ|GAL|QT|PT|KG|G|L|ML|CL|EA|EACH|CTN|CS|PCS|STK)\b'  # Various units embedded
            ]
            
            pattern_matches = 0
            for pattern in embedded_patterns:
                pattern_matches += sample_products.str.contains(pattern, case=False, regex=True).sum()
            
            match_ratio = pattern_matches / len(sample_products)
            if match_ratio > 0.4:
                logging.info(f"Auto-detect: Type03 - weight embedded in product description (match ratio: {match_ratio:.2f})")
                return "type03"
    
    # If no clear match from column analysis, use statistical detection
    return statistical_type_detection(df)

def autodetect_type(df: pd.DataFrame) -> str:
    """
    Comprehensive type detection with multiple strategies and validation
    """
    import logging
    
    # Strategy 1: Check for explicit type markers in first column
    if len(df) > 0 and len(df.columns) > 0:
        first_col = df.iloc[:, 0].astype(str).str.lower()
        for type_name in ["type01", "type02", "type03"]:
            if any(type_name in str(val) for val in first_col if pd.notna(val)):
                logging.info(f"Explicit type marker found: {type_name}")
                return type_name
    
    # Strategy 2: Vendor fingerprinting
    vendor_type = detect_vendor_fingerprint(df)
    if vendor_type:
        # Validate the vendor detection
        if validate_type_detection(df, vendor_type):
            return vendor_type
        else:
            logging.warning(f"Vendor fingerprint {vendor_type} failed validation, continuing detection")
    
    # Strategy 3: Enhanced pattern matching
    detected_type = autodetect_type_improved(df)
    
    # Strategy 4: Validation through sample parsing
    if validate_type_detection(df, detected_type):
        logging.info(f"Validated type detection: {detected_type}")
        return detected_type
    
    # If validation fails, try alternative types
    logging.warning(f"Primary detection {detected_type} failed validation, trying alternatives")
    
    for alt_type in ["type01", "type02", "type03"]:
        if alt_type != detected_type and validate_type_detection(df, alt_type):
            logging.info(f"Alternative type {alt_type} passed validation")
            return alt_type
    
    # Default to statistical detection if all else fails
    final_type = statistical_type_detection(df)
    logging.info(f"Using statistical detection fallback: {final_type}")
    return final_type

# Column-name candidates (names are normalized by safe_read_csv: lower case, "_" for spaces)
_PRODUCT_COLS = ["product", "item_description", "item description", "description", "item", "product_name",
                 "article", "artikel", "bezeichnung", "produit", "producto", "designation"]
_WEIGHT_COLS  = ["product_weight", "total_weight", "total weight", "weight", "net_wght_shipped",
                 "net wght shipped", "net_weight", "total_net_weight", "shipped_weight"]
_WEIGHT_COL_RX = re.compile(r"weight|wght|gewicht|poids|peso|\bwt\b|_wt\b|^wt_")
_UNIT_COLS    = ["unit", "uom", "unit_of_measure", "unit_of_measurement", "units", "einheit", "unite", "unidad"]
_QTY_COLS     = ["qty", "quantity", "qty_shipped", "qty shipped", "quantity_shipped", "menge", "anzahl",
                 "cantidad", "quantite"]

def _first_col(cols: dict, candidates) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return cols[c]
    return None

def _heading_uom(colname: str) -> Optional[str]:
    """Unit named in a column heading, e.g. 'product_weight_kg' -> 'KG'."""
    toks = set(re.split(r"[^a-z]+", str(colname).lower()))
    if toks & {"kg", "kgs", "kilo", "kilos", "kilogram", "kilograms"}:
        return "KG"
    if toks & {"g", "gr", "gram", "grams"}:
        return "G"
    if toks & {"lb", "lbs", "pound", "pounds"}:
        return "LB"
    if toks & {"oz", "ounce", "ounces"}:
        return "OZ"
    return None

def handle_type01(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Handle type01 format: quantity and units in combined column"""
    cols = {c.lower(): c for c in df.columns}

    # Ensure we have a product column
    if "product" not in df.columns:
        df["product"] = df.iloc[:, 0].astype(str)

    # Look for the quantity/weight column
    if "product_weight" in cols:
        pw = cols["product_weight"]
    elif any("quantity" in c.lower() and "base" in c.lower() for c in df.columns):
        # Find the quantity base unit column
        pw = next(c for c in df.columns if "quantity" in c.lower() and "base" in c.lower())
    else:
        # Fallback: look for a column with quantity+unit patterns
        pw = None
        for col in df.columns[1:]:  # Skip first column (likely product name)
            sample = df[col].dropna().head(10).astype(str)
            if sample.str.match(r'^\d+(?:[.,]\d+)?\s+[A-Za-z]+$', case=False).sum() > len(sample) * 0.3:
                pw = col
                break

    if pw and pw in df.columns:
        # Split the quantity and unit
        parts = df[pw].astype(str).str.strip().str.split(r'\s+', n=1, expand=True)
        df["qty"] = pd.to_numeric(parts[0].map(parse_number), errors="coerce")
        df["uom"] = parts[1] if parts.shape[1] == 2 else np.nan
    else:
        # Extract from product column as fallback
        q, u = zip(*df["product"].map(parse_qty_unit))
        df["qty"], df["uom"] = q, u
    df["__w_amt"] = df["qty"]
    df["__w_uom"] = df["uom"]

    df["description"] = df["product"]
    base = ["product", "description", "__w_amt", "__w_uom"]
    if "qty" in df.columns:
        base.append("qty")
    return df[base]

def handle_type02(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Handle type02 format: separate weight column (lb or kg)"""
    cols = {c.lower(): c for c in df.columns}

    # Find product column
    prod_col = _first_col(cols, _PRODUCT_COLS) or df.columns[0]
    df["product"] = df[prod_col].astype(str)

    # Find weight column: known names first, then any heading that looks like a weight
    prod_wt_col = _first_col(cols, _WEIGHT_COLS)
    if prod_wt_col is None:
        for c in df.columns:
            cl = str(c).lower()
            if _WEIGHT_COL_RX.search(cl) and "unit" not in cl:
                prod_wt_col = c
                break

    # Find unit and quantity columns
    unit_col = _first_col(cols, _UNIT_COLS)
    qty_col = _first_col(cols, _QTY_COLS)

    if qty_col and qty_col in df.columns:
        df["qty"] = num_series(df[qty_col])

    if prod_wt_col and prod_wt_col in df.columns:
        amt = num_series(df[prod_wt_col])
        heading = _heading_uom(prod_wt_col)
        setting = str(globals().get("INPUT_WEIGHT_UNIT", "auto")).lower()
        assumed = heading or ("KG" if setting == "kg" else "LB")
        units = None
        if unit_col and unit_col in df.columns:
            u = df[unit_col].astype(str).str.strip().str.upper()
            u = u.where(~u.isin(["", "NAN", "NONE"]), np.nan)
            if u.notna().any():
                units = u.fillna(assumed)   # blank unit cells use the heading/assumed unit
        if units is None:
            units = pd.Series(assumed, index=df.index)
            if verbose and heading is None:
                print(f"[units] type02: weight column '{prod_wt_col}' has no units (unit column missing or empty, "
                      f"and none in its heading); reading the weights as {'kg' if assumed == 'KG' else 'lb'}. "
                      f"Put 'kg' in the heading (e.g. 'product_weight (kg)') or add a 'unit' column "
                      f"if that is wrong.")
        df["__w_amt"] = amt
        df["__w_uom"] = units
    elif qty_col and unit_col and qty_col in df.columns and unit_col in df.columns:
        # Calculate from qty and unit
        df["__w_amt"] = num_series(df[qty_col])
        df["__w_uom"] = df[unit_col].astype(str)
    else:
        # Try to extract from product description as last resort
        q, u = zip(*df["product"].map(parse_qty_unit))
        df["__w_amt"] = pd.Series(q, index=df.index)
        df["__w_uom"] = pd.Series(u, index=df.index)

    df["description"] = df["product"]

    base = ["product", "description", "__w_amt", "__w_uom"]
    if "qty" in df.columns:
        base.append("qty")
    return df[base]

def handle_type03(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Handle type03 format: weight info embedded in product description"""
    cols = {c.lower(): c for c in df.columns}

    # Find product column
    prod_col = _first_col(cols, _PRODUCT_COLS) or df.columns[0]
    df["product"] = df[prod_col].astype(str)

    # Look for a quantity sold column (common in Baldor format)
    qty_col = cols.get("qty sold cw") or cols.get("qty_sold_cw") or _first_col(cols, _QTY_COLS)
    if qty_col and qty_col in df.columns:
        df["qty_sold"] = num_series(df[qty_col])

    # Extract weight from product description
    q, u = zip(*df["product"].map(parse_qty_unit))
    df["qty"] = pd.to_numeric(pd.Series(q, index=df.index), errors="coerce")
    df["__w_amt"] = df["qty"]
    df["__w_uom"] = pd.Series(u, index=df.index)

    # If we have qty_sold and a unit column, we might be able to calculate total weight
    if "qty_sold" in df.columns:
        unit_col = cols.get("stock unit") or cols.get("stock_unit") or _first_col(cols, _UNIT_COLS)
        if unit_col and unit_col in df.columns:
            # For items sold by weight units (LB, KG, OZ, G, ...), the weight is qty_sold x unit
            unit_series = df[unit_col].astype(str).str.upper().str.strip()
            is_weight_unit = unit_series.map(lambda x: uom_system(x) is not None and _uom_key(x) not in
                                             ("L", "LT", "LTR", "ML", "CL", "DL", "GAL", "QT", "PT"))
            sold = is_weight_unit & df["qty_sold"].notna()
            df.loc[sold, "__w_amt"] = df.loc[sold, "qty_sold"]
            df.loc[sold, "__w_uom"] = unit_series[sold]

        # Use qty_sold as qty if we don't have qty from parsing
        if df["qty"].isna().all() and df["qty_sold"].notna().any():
            df["qty"] = df["qty_sold"]

    df["description"] = df["product"]
    base = ["product", "description", "__w_amt", "__w_uom", "qty"]
    return df[base]

def load_and_standardize(path: str, file_type: str = "auto") -> pd.DataFrame:
    """
    Main function to load and standardize procurement data
    
    Args:
        path: Path to the CSV file
        file_type: One of 'auto', 'type01', 'type02', 'type03'
    
    Returns:
        Standardized DataFrame with columns: product, description, weight_kg or weight_lbs, qty (optional)
    """
    # Load the raw data
    reset_sheet_rows()   # row numbers come from this worksheet's Worksheet row column, if any (Cell 2)
    raw = safe_read_csv(path)
    
    # Determine the file type
    ft = file_type
    if ft == "auto":
        ft = autodetect_type(raw)
        logging.info(f"Auto-detected file_type: {ft}")
    elif ft not in ["type01", "type02", "type03"]:
        raise ValueError("file_type must be one of {'auto','type01','type02','type03'}")
    
    # Process according to detected/specified type
    if ft == "type01":
        std = handle_type01(raw)
    elif ft == "type02":
        std = handle_type02(raw)
    elif ft == "type03":
        std = handle_type03(raw)
    else:
        raise ValueError(f"Unexpected file_type: {ft}")

    # Decide the run's weight unit from the units on the lines, then convert each line once
    # (lines already in that unit are not converted).
    std = std.copy()
    std["__src_row"] = std.index   # worksheet row, to find the row's INPUT_ATTRIBUTES
    has_amt = std["__w_amt"].notna()
    systems = std["__w_uom"].map(uom_system)
    n_metric = int((has_amt & systems.eq("kg")).sum())
    n_us = int((has_amt & systems.eq("lb")).sum())
    unit = resolve_weight_unit(n_metric, n_us, context=f"{Path(path).name} ({ft})")
    wcol = f"weight_{weight_suffix(unit)}"
    std[wcol] = [to_weight(a, u, unit) if pd.notna(a) and pd.notna(u) else np.nan
                 for a, u in zip(std["__w_amt"], std["__w_uom"])]
    std[wcol] = pd.to_numeric(std[wcol], errors="coerce")
    keep = ["product", "description", wcol] + (["qty"] if "qty" in std.columns else []) + ["__src_row"]
    std = std[keep]

    # Log summary statistics
    total_rows = len(std)
    valid_weights = std[wcol].notna().sum()
    logging.info(f"Standardization complete: {valid_weights}/{total_rows} rows with valid weights ({valid_weights/total_rows*100:.1f}%)")
    
    if valid_weights / total_rows < 0.3:
        logging.warning("Low weight extraction rate. Consider checking the file format or type detection.")
    
    return std

In [ ]:
# === Cell 4 — AI Ingredient Extraction ===
# Purpose: Define GPT-based function(s) that infer ingredient lists and estimated proportions for each product.
# Includes: Guard lists (protected items, vegan/meatless handling, fish terms) and exact category helpers.
# Inputs:
#   - product text string(s)
#   - optional explicit_ingredients (for compatibility with non-AI modes)
# Dependencies:
#   - Utilities from Cell 2 (_norm, safe_json, etc.)
#   - Config flags from Cell 1 (PROTECTED_ITEMS, VEGAN_GUARD)
# Outputs:
#   - List[Dict[str, float]] per product, each containing {"ingredient": <name>, "percent": <float>}
# Non-decomposition rules & exact category resolvers used by Stages 1 & 2.
# Behavior:
#   - Calls GPT via ai_extract_ingredients() with structured prompts.
#   - Applies post-processing:
#       • normalize capitalization and spacing
#       • protect compound food names (e.g., “ice cream”, “almond milk”)
#       • enforce vegan/meatless exclusion filters.

import re
from typing import List, Dict, Optional
from pathlib import Path
import pandas as pd

def _norm(x: str) -> str:
    """Use the notebook's normalizer so everything stays consistent."""
    return norm_text(x)

def _canon_key(s: str) -> str:
    """
    Canonical key for STRICT equality:
      - lowercase
      - replace any non-alphanumeric with a single space ((),/,&,commas, hyphens, etc.)
      - collapse whitespace, strip
    NOTE: This is still equality-based (no substring matching).
    """
    s = str(s or "").lower()
    s = re.sub(r"[^0-9a-z]+", " ", s)     # punctuation/paren → space
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ------------------ Load factors categories (source of truth) ------------------
try:
    _fac_df = safe_read_csv(factors_filename, remember_format=False)  # prefer your safer loader
except Exception:
    _fac_df = pd.read_csv(factors_filename)

if "category" in _fac_df.columns:
    _fac_series = _fac_df["category"].astype(str)
else:
    _fac_series = _fac_df.iloc[:, 0].astype(str)

ALLOWED_CATEGORIES: List[str] = [c.strip() for c in _fac_series.tolist() if str(c).strip()]

# Canonical maps keyed by strict canonical equality (no substrings)
CANON_CAT_BY_CANON: Dict[str, str] = {}
EXACT_ALLOWED_CATEGORIES: set[str] = set()
for cat in ALLOWED_CATEGORIES:
    k = _canon_key(cat)
    if not k or k == "unknown":
        continue
    EXACT_ALLOWED_CATEGORIES.add(k)
    # Preserve original label as the canonical output
    CANON_CAT_BY_CANON.setdefault(k, cat)

# ------------------ Category map (from your foodcategories.json) ------------------
def load_category_map(json_path: Path) -> Dict[str, str]:
    """Return normalized-name -> category map loaded earlier (FOODCATS)."""
    return FOODCATS

CATEGORY_MAP = load_category_map(categories_filename)

# ------------------ Protections used by later cells ------------------
DO_NOT_DECOMPOSE = set(map(_norm, [
    # Core dairy intact items:
    "milk","dairy milk","skim milk","concentrated milk","buttermilk",
    "cheese","yogurt","low fat yogurt","cream","sour cream","butter","ghee",
    "ice cream","whey powder","milk powder",
    # Plant milks intact as single items:
    "almond milk","oat milk","soy milk","rice milk","coconut milk","pea milk",
]))

# Minimal fish list so downstream checks won’t fail; include milkfish explicitly.
FISH_SPECIES = set(map(_norm, [
    "salmon","trout","tilapia","catfish","bass","cod","pollock","haddock","mahi","octopus","squid",
    "tuna","sardine","anchovy","halibut","carp","milkfish"
]))

ANIMAL_INGREDIENTS = set(map(_norm, [
    "milk","dairy milk","cheese","butter","cream","yogurt","whey","casein","ghee","buttermilk","lactose",
    "egg","egg whites","egg yolk","honey","gelatin", "dairy cheese",
    "beef","pork","chicken","turkey","lamb","bacon","sausage","octopus","squid",
    "fish","salmon","tuna","shrimp","tilapia","trout","catfish","bass","milkfish"
]))

# ------------------ Prioritized list = EXACT categories only (for prompts, if used) ------------------
PRIORITIZED_INGREDIENTS: List[str] = sorted({CANON_CAT_BY_CANON[k] for k in EXACT_ALLOWED_CATEGORIES})

# ------------------ Exact-only resolver (canonical equality, no substrings) ------------------
def resolve_exact_category(product_text: str) -> Optional[str]:
    """
    STRICT equality after canonicalization:
      1) If canon(product) equals canon(category) from factors.csv → return that category (canonical label).
      2) Else, if product matches a key in CATEGORY_MAP and that mapped value is a valid factors category,
         accept the mapped category (also via canonical equality).
      3) Else, return None.

    Notes:
      - No token splitting, no partial/substring/phrase matching.
      - 'milkfish' will NOT equal 'milk' in canonical form.
      - 'barley (beer)' equals 'barley beer' canonically on both sides, so it matches cleanly.
      - Multi-word categories like 'low fat yogurt' remain distinct from 'yogurt'.
    """
    name_can = _canon_key(product_text)
    if not name_can:
        return None

    # (1) Direct exact to factors category
    if name_can in CANON_CAT_BY_CANON:
        return CANON_CAT_BY_CANON[name_can]

    # (2) Exact to a key in CATEGORY_MAP, but only accept if mapped value is in factors
    #     (We canonicalize the mapped value and check against CANON_CAT_BY_CANON.)
    k_norm = _norm(product_text)
    if k_norm in CATEGORY_MAP:
        mapped = CATEGORY_MAP[k_norm]
        mapped_can = _canon_key(mapped)
        if mapped_can in CANON_CAT_BY_CANON:
            return CANON_CAT_BY_CANON[mapped_can]

    return None

logging.info(
    "[Cell 4] Exact-only canonical mode: %d categories; PRIORITIZED_INGREDIENTS=%d; DO_NOT_DECOMPOSE=%d",
    len(EXACT_ALLOWED_CATEGORIES), len(PRIORITIZED_INGREDIENTS), len(DO_NOT_DECOMPOSE)
)

# --- Adapter for legacy calls expecting `categorize_name_from_map` ---
def categorize_name_from_map(name: str) -> Optional[str]:
# Use the exact-only canonical resolver
    return resolve_exact_category(name)

# ------------------ Plant-based judgment and worksheet attributes (Stage 1; from 2026-11-01a) ------------------
# Plant-based is judged for each PURCHASED PRODUCT as a whole: a product that qualifies counts with all
# its ingredients, one that does not (e.g. pepperoni pizza) contributes nothing, crust and sauce included.
# judge_plant_based() decides, in this order:
#   1. a y in the worksheet's "Plant Based (enter y if yes)" column wins;
#   2. products made mostly of things that are neither plant nor animal (bottled water, soft drinks,
#      salt, baking soda ...) are not plant-based;
#   3. a vegan / plant-based label means yes; an animal word in the name (pepperoni, cheese, honey ...)
#      means no (plant versions such as "almond milk", "peanut butter", "vegan cheese" are not animal words);
#   4. otherwise the ingredient list is checked for animal words and, for products sent to GPT, compared
#      with GPT's answer for the whole product (which also covers minor ingredients such as honey or
#      gelatin that are dropped from the list). If the two disagree the product is marked "review",
#      which does not count as plant-based until it is edited.
# Minor additives that vegan products commonly contain from non-animal sources (e.g. vitamin D3) do not
# disqualify a product. A plant-based product also gets a processing category (PLANT_CATEGORIES, from
# 2026-11-01d; the NOVA groups that STARS OP-7 uses), so an institution can count only the more conservative
# categories (PLANT_BASED_MAX_CATEGORY in Cell 1). The result is written to the ingredients file
# (plant_based = 1-4, 0 = not plant-based, review, or y = plant-based in every category; pb_basis = the
# reason) on every row of the product, so it can be edited before re-running with
# START_MODE = "after_ingredients". drink_water = y marks bottled water and diluted drinks such as soda,
# whose water is left out of the total weight in the shares report (Cell 8b).
PLANT_CATEGORIES = {
    1: "whole or minimally processed plant foods",
    2: "plant cooking ingredients (oils, sugars, syrups, starches, vinegar)",
    3: "simple processed plant foods and meat / dairy alternatives",
    4: "ultra-processed plant foods",
}

# Extra questions added to Stage 1's ingredient request to GPT (no extra calls); answers kept per product
PRODUCT_FLAG_RULES = (
    "Also answer four questions about the WHOLE product as purchased, as extra keys of the same JSON object:\n"
    "- animal_derived: true if it contains ANY ingredient definitely of animal origin (meat, poultry, fish, "
    "shellfish, dairy such as milk, cream, butter, cheese, whey, casein or sodium caseinate (found in most "
    "'non-dairy' creamers and whipped toppings), eggs, honey, gelatin, lard or other "
    "animal fat, meat or fish broth, anchovy as in Worcestershire sauce or Caesar dressing, carmine), including "
    "minor ingredients left out of items; false otherwise. Vitamins such as D3, natural flavors and other minor "
    "additives that vegan products commonly contain from non-animal sources do NOT count.\n"
    "- neither_plant_nor_animal: true if it is made mostly of water, salt, minerals or synthetic ingredients and "
    "is not a meat or dairy alternative (e.g. soft drinks, flavored or mineral water, sports drinks, salt, "
    "baking soda); false for foods and for 100% fruit or vegetable juice.\n"
    "- diluted_beverage: true if it is bottled water or a drink made mainly of added water (soda, sports or "
    "energy drink, lemonade, iced tea, juice drink or cocktail, flavored water); false for 100% juice, milk, "
    "plant milk, soup, broth, sauces and concentrates.\n"
    "- plant_category: how processed the product is, as a number (the NOVA groups): 1 = whole or minimally "
    "processed plant food (fresh, frozen, dried, plain canned, cut, ground or roasted: produce, plain beans, rice, "
    "oats including plain instant oats, flour, nuts, seeds, spices and herbs, blends made only of spices and herbs "
    "such as chili powder, curry powder, garam masala or za'atar, chips made only of dried or freeze-dried fruit or "
    "vegetables with nothing added, tea, coffee); 2 = cooking ingredient taken "
    "directly from plants (oil, sugar, maple syrup, starch, vinegar); 3 = a category 1 food with only salt, sugar, "
    "oil, vinegar or other category 1-2 ingredients added, or a meat or dairy alternative (100% juice, pasta, "
    "plain bread or tortillas, tofu, tempeh, hummus, nut butter, plant milk, vegan meat or cheese, salsa, pickles, "
    "jam, canned beans in sauce including baked beans, dried fruit with added sugar, sweetened coconut, candied or "
    "salted nuts, simple canned soups, plain frozen fries, potatoes or plantains even if par-fried in oil, "
    "seasoning mixes of spices with salt); 4 = ultra-processed: the product likely contains additives not used in "
    "home cooking, such as emulsifiers, flavorings, colors, modified starch, dextrose, maltodextrin, hydrolyzed "
    "protein or MSG (most breakfast cereals, chips, snack bars, cookies, candy, ketchup, dressings, most sauces, "
    "flavored syrups, margarine-type spreads, shortening, bouillon and soup bases, coated or battered fries, "
    "flavored instant oatmeal, seasoning mixes with flavorings or MSG). Added sugar or salt alone does not make a "
    "food category 4, with one exception: commercially made snack chips (potato, corn or tortilla, pita, "
    "vegetable, banana and other fried or extruded chips) are 4 even if the label lists only the plant, oil and "
    "salt, because they are made by industrial processes not used in home kitchens. Judge what the product is, not the catalog words in its name: SEASONING, FROZEN, INSTANT, "
    "SHELF STABLE, REF or BAG describe how it is sold. Use null if it contains animal-derived ingredients or is "
    "mostly water or salt."
)
PRODUCT_FLAG_KEYS = ("animal_derived", "neither_plant_nor_animal", "diluted_beverage")
S1_PRODUCT_FLAGS: Dict[str, Dict] = {}   # product text -> GPT's answers (the keys above and plant_category)

def _as_bool(v) -> Optional[bool]:
    if isinstance(v, bool):
        return v
    t = str(v).strip().lower()
    return True if t in ("true", "yes") else False if t in ("false", "no") else None

def _as_category(v) -> Optional[int]:
    """A plant category 1-4 from a GPT reply or an edited file ('3', 3, 3.0); None otherwise."""
    try:
        c = int(float(str(v).strip()))
    except (TypeError, ValueError):
        return None
    return c if c in PLANT_CATEGORIES else None

def store_product_flags(product_text: str, obj) -> None:
    """Keep GPT's whole-product answers (PRODUCT_FLAG_KEYS) from a Stage 1 reply."""
    if not isinstance(obj, dict):
        return
    flags = {k: _as_bool(obj.get(k)) for k in PRODUCT_FLAG_KEYS}
    flags["plant_category"] = _as_category(obj.get("plant_category"))
    if any(v is not None for v in flags.values()):
        S1_PRODUCT_FLAGS[str(product_text).strip()] = flags

# Animal words (matched as whole words on normalized text)
_DAIRY_W = (r"milks?|cream|creams|creamer|butter|buttermilk|cheese\w*|cheddar|mozzarella|parmesan|parmigiano|"
            r"provolone|ricotta|feta|brie|gouda|asiago|romano|queso|paneer|yogh?urts?|whey|casein\w*|ghee|lactose|"
            r"kefir|custard|alfredo")
_EGG_W = r"eggs?|egg whites?|egg yolks?|mayo|mayonnaise|aioli|meringue"
_OTHER_W = r"honey|gelatine?|lard|tallow|suet|collagen|isinglass|carmine|cochineal"
_MEAT_W = (r"beef|pork|chicken|turkey|lamb|mutton|bacon|sausages?|pepperoni|salami|ham|prosciutto|pancetta|veal|"
           r"venison|duck|goat|bison|meats?|meatballs?|meatloaf|jerky|hot dogs?|frankfurters?|bologna|chorizo|"
           r"carnitas|brisket|pastrami|gyros?")
_FISH_W = (r"fish|\w+fish|seafood|salmon|tuna|cod|haddock|pollock|anchov\w*|sardines?|halibut|mahi\w*|tilapia|"
           r"trout|bass|swai|basa|shrimps?|prawns?|crabs?|lobsters?|clams?|oysters?|mussels?|scallops?|squid|"
           r"calamari|octopus|surimi|roe|caviar")
_ANY_ANIMAL_W = f"{_DAIRY_W}|{_EGG_W}|{_OTHER_W}|{_MEAT_W}|{_FISH_W}"
_ANIMAL_RX = re.compile(rf"\b(?:{_ANY_ANIMAL_W})\b")
# Phrases removed before looking for animal words: labeled plant versions and plant foods named after animal ones
_NOT_ANIMAL_RXS = [
    re.compile(rf"\b(?:vegan|plant based|plant derived)\s+(?:\w+\s+){{0,2}}(?:{_ANY_ANIMAL_W})\b"),
    re.compile(rf"\b(?:vegetarian|meatless|meat free|veggie|plant)\s+(?:\w+\s+){{0,2}}(?:{_MEAT_W}|{_FISH_W})\b"),
    re.compile(rf"\b(?:dairy free|non dairy|nondairy|milk free)\s+(?:\w+\s+){{0,2}}(?:{_DAIRY_W})\b"),
    re.compile(r"\b(?:coconut|almond|oat|oats|soy|soya|rice|cashew|hemp|pea|macadamia|hazelnut|flax|pistachio|"
               r"plant|nut)\s+(?:\w+\s+)?(?:milks?|cream|creamer|yogh?urts?|cheese|butter|beverage)\b"),
    # the same with the words reversed, as invoices write them: "MILK COCONUT", "CREAM COCONUT"
    re.compile(r"\b(?:milks?|cream|creamer|yogh?urts?|cheese|butter)\s+(?:coconut|almond|oat|oats|soy|soya|cashew|"
               r"hemp|pea|macadamia|hazelnut|flax|pistachio|peanut|sunflower)\b"),
    # the country, not the bird: "IMPORTED TURKEY", "PRODUCT OF TURKEY", "Türkiye"
    re.compile(r"\b(?:imported|import|product of|from|origin|grown in|made in)\s+(?:\w+\s+)?turkey\b|\bt\s?u?rkiye\b"),
    re.compile(r"\b(?:peanut|nut|almond|cashew|sunflower|seed|apple|pumpkin|cocoa|cacao|shea|soy|pecan|hazelnut|"
               r"walnut|pistachio|sesame|cookie)\s+butter\b"),
    re.compile(r"\bbutter\s+(?:beans?|lettuce|squash)\b|\bcream\s+(?:of\s+tartar|of\s+coconut|style|sodas?)\b"),
    re.compile(r"\begg\s+(?:substitutes?|replacers?)\b|\begg\s+free\b|\bmilk\s+thistle\b"),
    # "MILK SUB", "BEEF SUB, PATTY", "CHEESE ALTERNATIVE": the product's ingredients and GPT then decide
    re.compile(rf"\b(?:{_ANY_ANIMAL_W})\s+(?:\w+\s+)?(?:subs?|substitutes?|alternatives?|analogu?es?)\b"),
    re.compile(r"\b(?:hot dog|hamburger|burger|sandwich|sub|slider)\s+(?:buns?|rolls?)\b"),
    re.compile(rf"\bfor\s+(?:\w+\s+)?(?:{_MEAT_W}|{_FISH_W})\b"),            # "seasoning for chicken"
]
_SEASONING_NAME_RX = re.compile(r"^\s*(?:seasonings?|spices?|rubs?|spice blends?)\b", re.I)
_VEGAN_LABEL_RX = re.compile(r"\b(?:vegan|plant based|plant derived)\b")

# Products made mostly of things that are neither plant nor animal, and drinks whose water is left out
_NEITHER_NAME_RX = re.compile(r"\b(?:sodas?|soft drinks?|colas?|seltzers?|club soda|tonic water|sports drinks?|"
                              r"energy drinks?|electrolyte \w+|flavou?red water|sparkling water|mineral water|"
                              r"spring water|bottled water|drinking water|purified water|distilled water|"
                              r"baking soda|baking powder|ice cubes?|bagged ice|"
                              # soft-drink concentrates and juice drinks: "SYRUP, FOUNTAIN ...", "DRINK BASE, ..."
                              r"fountain|drink (?:base|mix|concentrate)|(?:juice|fruit|cranberry) drinks?)\b")
_NOT_A_DRINK_RX = re.compile(r"\b(?:crackers?|saltines?|breads?|biscuits?)\b")
# Plant waters: juices, not water ("WATER, COCONUT"), even when Stage 1 lists them as the single ingredient "water"
_PLANT_WATER_RX = re.compile(r"\b(?:coconut|maple|birch|cactus|aloe)\b")
_NEITHER_SINGLE = {"water", "salt", "ice", "sea salt", "kosher salt", "baking soda", "baking powder",
                   "carbonated water", "mineral water", "sparkling water"}
_DRINK_NAME_RX = re.compile(r"\b(?:sodas?|soft drinks?|colas?|seltzers?|club soda|tonic water|ginger ale|"
                            r"root beer|lemonade|limeade|iced tea|sweet tea|juice drinks?|juice cocktails?|"
                            r"fruit punch|sports drinks?|energy drinks?|electrolyte \w+|flavou?red water|"
                            r"sparkling water|mineral water|spring water|bottled water|drinking water|"
                            r"purified water|kombucha)\b")

def animal_words(text: str) -> List[str]:
    """Animal-derived words in a product or ingredient name, ignoring plant versions (e.g. 'almond milk')."""
    t = norm_text(text)
    for rx in _NOT_ANIMAL_RXS:
        t = rx.sub(" ", t)
    return sorted(set(m.group(0) for m in _ANIMAL_RX.finditer(t)))

# Invoice names often read "HEAD, MODIFIERS" ("MILK, OAT", "BEANS, BUTTER", "SAUSAGE, VEGETARIAN"), so a name
# is also read with the head moved to the end ("oat milk"); an animal word counts only if both readings have it.
_NAME_FILLER_WORDS = {"canned", "frozen", "fresh", "organic", "dried", "dry", "sliced", "whole", "chopped", "diced",
                      "cut", "iqf", "raw", "cooked", "natural", "plain", "original", "large", "small", "ref"}

def name_animal_words(product_text: str) -> List[str]:
    """animal_words() for a product name, read both as written and with its first comma part moved to the end."""
    parts = [p for p in str(product_text).split(",") if p.strip()]
    found = set(animal_words(product_text))
    if len(parts) > 1 and found:
        rest = [w for w in norm_text(" ".join(parts[1:])).split() if w not in _NAME_FILLER_WORDS]
        found &= set(animal_words(" ".join(rest + [norm_text(parts[0])])))
    return sorted(found)

def _is_yes(v) -> bool:
    return str(v).strip().lower() in ("y", "yes")

# Processing category of the single-ingredient staples that Stage 1 settles without GPT
_STAPLE_CATEGORY = {"flour": 1, "rice": 1, "oats": 1, "oil": 2, "sugar": 2, "pasta": 3}
# Plant products whose name settles their category before any other rule
_NAMED_PLANT_CATEGORY = [(re.compile(r"\bcream of tartar\b"), 2, "cream of tartar comes from grapes"),
                         (re.compile(r"\b(?:coconut|maple|birch|cactus|aloe)\b.*\bwater\b|\bwater\b.*\b(?:coconut|maple|"
                                     r"birch|cactus|aloe)\b"), 3, "a plant water, counted like juice")]
# Meat, dairy and egg alternatives ("MILK SUB", "BEEF SUB, PATTY", "VEGAN CHEESE") are category 3, as STARS lists them
_ALTERNATIVE_RX = re.compile(rf"\b(?:{_ANY_ANIMAL_W})\s+(?:\w+\s+)?(?:subs?|substitutes?|alternatives?|analogu?es?)\b")
# "Non-dairy" creamers and whipped toppings usually contain sodium caseinate, a milk protein
_CASEINATE_NAME_RX = re.compile(r"\b(?:creamers?|whipped toppings?|whip toppings?|non dairy toppings?|dessert toppings?|"
                                r"toppings?\s+(?:whipped|whip|non dairy|dessert))\b")
_PLANT_CREAMER_RX = re.compile(r"\b(?:oat|oats|almond|soy|soya|coconut|cashew|pea|plant|hemp|rice)\b")

def _category(flags, ings, fallback: int) -> Tuple[str, str]:
    """(category as text, note) for a plant-based product: GPT's answer, else the staple table, else fallback."""
    c = _as_category(flags.get("plant_category"))
    if c is None and len(ings) == 1:
        c = _STAPLE_CATEGORY.get(ings[0])
    if c is None:
        return str(fallback), f"; category {fallback} assumed (no processing answer)"
    return str(c), f"; category {c}"

def judge_plant_based(product_text: str, items, flags=None, user_value="") -> Tuple[str, str, bool]:
    """(plant_based, pb_basis, drink) for one purchased product. plant_based is a category '1'-'4', '0' (not
    plant-based), 'review', or 'y' (the worksheet's y: plant-based in every category)."""
    name = norm_text(product_text)
    ings = [norm_text(it.get("ingredient", "")) for it in (items or [])]
    flags = flags or {}
    plant_water = bool(_PLANT_WATER_RX.search(name)) and bool(re.search(r"\bwater\b", name))
    single_water = ings in (["water"], ["carbonated water"], ["mineral water"], ["sparkling water"]) and not plant_water
    drink = bool(flags.get("diluted_beverage") is True or single_water
                 or (_DRINK_NAME_RX.search(name) and not _NOT_A_DRINK_RX.search(name)))

    user_cat = _as_category(user_value)
    if _is_yes(user_value) or user_cat:
        hits = sorted(set(name_animal_words(product_text)) | {w for i in ings for w in animal_words(i)})
        note = f" (note: animal word(s) found: {', '.join(hits)})" if hits else ""
        return (str(user_cat) if user_cat else "y"), f"marked {user_value} in the worksheet" + note, drink
    for rx, cat, why in _NAMED_PLANT_CATEGORY:
        if rx.search(name):
            return str(cat), f"plant-based: {why}; category {cat}", drink
    neither_name = bool(_NEITHER_NAME_RX.search(name)) and not _NOT_A_DRINK_RX.search(name)   # soda crackers
    if (len(ings) == 1 and ings[0] in _NEITHER_SINGLE and not plant_water) or neither_name \
            or flags.get("neither_plant_nor_animal") is True:
        return "0", "mostly neither plant nor animal (e.g. water, soft drink, salt)", drink
    if not ings or all(i in ("unknown", "") for i in ings):
        return "0", "ingredients unknown; not judged", drink
    vegan = bool(_VEGAN_LABEL_RX.search(name))
    alternative = bool(_ALTERNATIVE_RX.search(name)) or (vegan and bool(_ANIMAL_RX.search(name)))
    if vegan:
        if alternative:
            return "3", "labeled vegan / plant-based; meat / dairy alternative, category 3", drink
        cat, note = _category(flags, ings, 3)
        return cat, "labeled vegan / plant-based" + note, drink
    name_hits = [] if _SEASONING_NAME_RX.match(str(product_text)) else name_animal_words(product_text)
    if name_hits:
        return "0", f"animal-derived word in the name: {', '.join(name_hits)}", drink
    if _CASEINATE_NAME_RX.search(name) and not _PLANT_CREAMER_RX.search(name):
        return "0", "non-dairy creamers and whipped toppings usually contain sodium caseinate (a milk protein)", drink
    ing_hits = sorted({w for i in ings for w in animal_words(i)})
    ai = flags.get("animal_derived")
    if ing_hits and ai is False:
        return "review", (f"ingredient list has {', '.join(ing_hits)} but GPT judged the product free of "
                          f"animal-derived ingredients"), drink
    if ing_hits:
        return "0", f"animal-derived ingredient: {', '.join(ing_hits)}", drink
    if ai is True:
        return "review", "GPT judged that it contains an animal-derived ingredient not in the ingredient list", drink
    cat, note = ("3", "; meat / dairy alternative, category 3") if alternative else _category(flags, ings, 4)
    if ai is False:
        return cat, "no animal-derived ingredient (ingredient list and GPT agree)" + note, drink
    return cat, "no animal-derived ingredient in the name or ingredient list" + note, drink

def product_attributes(src_row, product_text: str, items) -> Dict[str, str]:
    """Columns Stage 1 adds to every ingredient row of a product: plant_based, pb_basis, drink_water, and the
    worksheet's certification columns (aw_food_group, aw, env, antibiotics, labor, the six aashe_* columns, local,
    social_impact, explanation) when the worksheet has them (INPUT_ATTRIBUTES, Cell 2)."""
    attrs = globals().get("INPUT_ATTRIBUTES")
    user = {}
    try:
        r = int(src_row)
        if attrs is not None and r in attrs.index:
            user = attrs.loc[r].to_dict()
    except (TypeError, ValueError):
        pass
    flags = S1_PRODUCT_FLAGS.get(str(product_text).strip(), {})
    pb, basis, drink = judge_plant_based(product_text, items, flags, user.get("plant_based_user", ""))
    out = {"plant_based": pb, "pb_basis": basis, "drink_water": "y" if drink else ""}
    if attrs is not None:
        for k in ("aw_food_group", "aw", "env", "antibiotics", "labor", "aashe_organic", "aashe_animal_welfare", "aashe_environment", "aashe_labor",
                  "aashe_small_producers", "aashe_seafood", "local", "social_impact", "explanation"):
            if k in attrs.columns:
                out[k] = user.get(k, "")
    return out


In [ ]:
# === Cell 5 — Stage 1 (type03) Runner ===
# Purpose: Execute AI ingredient extraction for type03 vendor files.
# Inputs:
#   - standardized DataFrame from Cell 3 (type03 schema)
#   - external item_weights.json (optional overrides; pounds per item unless it contains "_unit": "kg")
#   - ai_extract_ingredients() from Cell 4
# Outputs:
#   - ingredients_{BASE}.csv (one row per product–ingredient pair, with estimated weights in the
#     worksheet's unit: product_weight_kg/ingredient_weight_kg or ..._lbs)
# Behavior:
#   - Merges item_weights overrides if present.
#   - Applies GPT extraction per product description.
#   - Normalizes percentages to sum ≈ 100 %.
#   - Logs progress and total ingredient count.
# Overwrites existing ingredients_{BASE}.csv if START_MODE='full'; skips otherwise.

import os, json, re, concurrent.futures as cf
from pathlib import Path
from typing import Any, Dict, List, Optional
import numpy as np
import pandas as pd

# === Guard flag ===
_run_cell5 = False
try:
    _run_cell5 = str(file_type).lower() == "type03"
except NameError:
    _run_cell5 = False

# --- Logger ---
try:
    log_time
except NameError:
    from contextlib import contextmanager
    import time
    @contextmanager
    def log_time(label: str):
        t0 = time.perf_counter()
        print(f"[timing] ▶ {label} ...")
        try:
            yield
        finally:
            dt = time.perf_counter() - t0
            print(f"[timing] ✓ {label} in {dt:,.3f}s")

DEBUG_WEIGHT_CALC = True

def debug_print(msg: str, row_sheet: Optional[int] = None):
    if DEBUG_WEIGHT_CALC:
        if row_sheet is not None:
            print(f"[Row {row_sheet}] {msg}")
        else:
            print(msg)

# ============================================================================
# WEIGHT EXTRACTION WITH ALL FIXES (v110b)
# ============================================================================

ITEM_WEIGHTS = {}
ITEM_WEIGHTS_UNIT = "lb"   # unit of the numbers in item_weights.json ("_unit" key; default pounds)

def load_item_weights():
    global ITEM_WEIGHTS, ITEM_WEIGHTS_UNIT
    weights_file = Path(globals().get("item_weights_filename")
                        or Path(globals().get("base_dir", ".")) / "item_weights.json")
    if not weights_file.exists():
        print(f"[Warning] {weights_file} not found. Using minimal defaults.")
        ITEM_WEIGHTS_UNIT = "lb"
        ITEM_WEIGHTS = {
            r'\bwatermelons?\b': 15.0,
            r'\bcantaloupes?\b': 3.0,
            r'\bhoneydewe?s?\b': 5.0,
            r'\bpineapples?\b': 4.0,
            r'\bapples?\b': 0.33,
        }
        return
    try:
        with open(weights_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        unit = str(data.get("_unit", "lb")).strip().lower()
        ITEM_WEIGHTS_UNIT = "kg" if unit in ("kg", "kgs", "kilogram", "kilograms") else "lb"
        for item_name, weight in data.items():
            if str(item_name).startswith("_"):
                continue   # settings such as "_unit"
            pattern = rf'\b{item_name}e?s?\b'
            ITEM_WEIGHTS[pattern] = weight
        print(f"[Config] Loaded {len(ITEM_WEIGHTS)} item weights ({ITEM_WEIGHTS_UNIT} per item) from {weights_file}")
    except Exception as e:
        print(f"[Error] Failed to load {weights_file}: {e}")
        ITEM_WEIGHTS = {r'\bapples?\b': 0.33}
        ITEM_WEIGHTS_UNIT = "lb"

load_item_weights()

# Reference weights below are US trade measures defined in pounds; on a kg run the estimates
# made from them are converted to kg. Weights written in metric units are read directly in kg.
CAN_WEIGHTS = {10: 6.5, 5: 3.5, 2: 1.5}
BUSHEL_WEIGHTS = {
    'lima bean': 30, 'bean': 30, 'beet': 50, 'cabbage': 50, 'carrot': 50,
    'cucumber': 59, 'eggplant': 34, 'okra': 30, 'onion': 52, 'parsnip': 50,
    'potato': 60, 'rutabaga': 53, 'spinach': 23, 'greens': 23, 'squash': 40,
    'sweet potato': 52, 'tomato': 55, 'turnip': 55, 'apple': 48,
    'blueberry': 42, 'cantaloupe': 50, 'peach': 50, 'pear': 50, 'plum': 50,
    'pepper': 28, 'corn': 56,
}

def get_bushel_weight(product_text: str) -> float:
    text_lower = product_text.lower()
    for product_name, weight in BUSHEL_WEIGHTS.items():
        if product_name in text_lower:
            return weight
    return 35

def liquid_kg_per_liter(product_text: str) -> float:
    """Approximate density (kg per liter) for liquids sold by metric volume."""
    text_lower = product_text.lower()
    if re.search(r'\b(water|tonic)\b', text_lower):
        return 1.00
    elif re.search(r'\b(juice|cider)\b', text_lower):
        return 1.02
    elif re.search(r'\b(oil)\b', text_lower):
        return 0.92
    elif re.search(r'\b(vinegar|wine)\b', text_lower):
        return 1.00
    elif re.search(r'\b(honey|syrup)\b', text_lower):
        return 1.50
    else:
        return 1.00

def get_liquid_density(product_text: str) -> float:
    """Approximate density (lb per US gallon) for liquids sold by US volume."""
    text_lower = product_text.lower()
    if re.search(r'\b(cider|juice|lemonade)\b', text_lower):
        return 8.6
    elif re.search(r'\b(olive oil|oil)\b', text_lower):
        return 7.5
    elif re.search(r'\b(vinegar|wine)\b', text_lower):
        return 8.4
    elif re.search(r'\b(honey|syrup)\b', text_lower):
        return 12.0
    else:
        return 8.34

def _unit_label() -> str:
    return "kg" if WEIGHT_UNIT == "kg" else "lbs"

def _in_run_unit(value, native: str):
    """Convert a weight from its native unit ('lb' or 'kg') to the run's weight unit."""
    return convert_weight(value, native, WEIGHT_UNIT or "lb")

def _normalize_pack_text(product_text: str) -> str:
    """Upper-case and tidy pack notation: '6 x 1 kg' / '6×1kg' -> '6X1 KG'; '2,5 kg' -> '2.5 KG'."""
    t = product_text.upper().replace("×", "X")
    t = re.sub(r"(\d)\s*X\s*(?=\d)", r"\1X", t)
    if CSV_DECIMAL == ",":
        t = re.sub(r"(\d),(\d)", r"\1.\2", t)
    else:
        t = re.sub(r"(\d),(\d{3})(?!\d)", r"\1\2", t)   # 1,000 -> 1000
        t = re.sub(r"(\d),(\d)", r"\1.\2", t)            # 2,5 -> 2.5
    return t

def _parse_fraction(num_str):
    if '/' in num_str:
        parts = num_str.split('/')
        if len(parts) == 2:
            try:
                return float(parts[0]) / float(parts[1])
            except:
                return float(num_str.replace('/', ''))
    return float(num_str)

def extract_weight_from_text(product_text: str, row_sheet: Optional[int] = None):
    """Per-unit weight in the run's weight unit (WEIGHT_UNIT), and how it was found."""
    if not product_text:
        return None, "unknown"

    product_upper = _normalize_pack_text(product_text)
    bushel_wt = get_bushel_weight(product_text)
    U = _unit_label()

    def _done(value, native, label, kind="weight_parsed", do_round=True):
        result = _in_run_unit(value, native)
        debug_print(f"'{product_text}' {label} → {result:.4f} {U}", row_sheet)
        return (round(result, 4) if do_round else result), kind

    # BUSHEL PATTERNS (support both "1-1/9 BU" and "BU 1-1/9" formats) -- bushel weights are in lb

    # Format: BU 1-1/9 (unit before number)
    bushel_match = re.search(r'(?:BU|BSH|BUSH)\b\s*(\d+)-(\d+)/(\d+)', product_upper)
    if bushel_match:
        whole, num, denom = int(bushel_match.group(1)), int(bushel_match.group(2)), int(bushel_match.group(3))
        return _done((whole + num/denom) * bushel_wt, "lb", "matched bushel (unit-first mixed)")

    # Format: 1-1/9 BU (number before unit)
    bushel_match = re.search(r'(\d+)-(\d+)/(\d+)\s*(?:BU|BSH|BUSH)\b', product_upper)
    if bushel_match:
        whole, num, denom = int(bushel_match.group(1)), int(bushel_match.group(2)), int(bushel_match.group(3))
        return _done((whole + num/denom) * bushel_wt, "lb", "matched bushel (mixed)")

    bushel_match = re.search(r'(\d+)\s+(\d+)/(\d+)\s*(?:BU|BSH|BUSH)\b', product_upper)
    if bushel_match:
        whole, num, denom = int(bushel_match.group(1)), int(bushel_match.group(2)), int(bushel_match.group(3))
        return _done((whole + num/denom) * bushel_wt, "lb", "matched bushel (space)")

    # Format: BU 1/2 (unit before fraction)
    bushel_match = re.search(r'(?:BU|BSH|BUSH)\b\s*(?:1/2|½)', product_upper)
    if bushel_match:
        return _done(0.5 * bushel_wt, "lb", "matched bushel (unit-first half)")

    # Format: 1/2 BU (fraction before unit)
    bushel_match = re.search(r'(?:1/2|½)\s*(?:BU|BSH|BUSH)\b', product_upper)
    if bushel_match:
        return _done(0.5 * bushel_wt, "lb", "matched bushel (half)")

    # Format: BU 1/4 (unit before simple fraction)
    bushel_match = re.search(r'(?:BU|BSH|BUSH)\b\s*(\d+)/(\d+)', product_upper)
    if bushel_match:
        return _done((int(bushel_match.group(1))/int(bushel_match.group(2))) * bushel_wt, "lb",
                     "matched bushel (unit-first fraction)")

    # Format: 1/4 BU (simple fraction before unit)
    bushel_match = re.search(r'(\d+)/(\d+)\s*(?:BU|BSH|BUSH)\b', product_upper)
    if bushel_match:
        return _done((int(bushel_match.group(1))/int(bushel_match.group(2))) * bushel_wt, "lb",
                     "matched bushel (fraction)")

    # Format: BU 2 (unit before whole number)
    bushel_match = re.search(r'(?:BU|BSH|BUSH)\b\s*(\d+(?:\.\d+)?)', product_upper)
    if bushel_match:
        return _done(float(bushel_match.group(1)) * bushel_wt, "lb", "matched bushel (unit-first whole)")

    # Format: 2 BU (whole number before unit)
    bushel_match = re.search(r'\b(\d+(?:\.\d+)?)\s*(?:BU|BSH|BUSH)\b', product_upper)
    if bushel_match:
        return _done(float(bushel_match.group(1)) * bushel_wt, "lb", "matched bushel (whole)")

    # CASE PATTERNS (CS = case, typical produce case weights in lb)
    case_weights = {
        'berry': 12, 'berries': 12, 'strawberry': 12, 'strawberries': 12,
        'blueberry': 12, 'blueberries': 12, 'raspberry': 12, 'raspberries': 12,
        'blackberry': 12, 'blackberries': 12,
        'lettuce': 24, 'cabbage': 50, 'broccoli': 20, 'cauliflower': 20,
        'pepper': 25, 'peppers': 25, 'tomato': 25, 'tomatoes': 25,
        'cucumber': 24, 'cucumbers': 24, 'squash': 20, 'zucchini': 20,
        'onion': 50, 'onions': 50, 'carrot': 50, 'carrots': 50,
        'celery': 55, 'scallion': 15, 'scallions': 15, 'greens': 20,
        'okra': 15, 'watercress': 4,
    }

    def get_case_weight(product_text: str) -> float:
        text_lower = product_text.lower()
        for product_name, weight in case_weights.items():
            if product_name in text_lower:
                return weight
        return 25  # default case weight

    # Match "1/2 CS" or "CS" patterns
    case_match = re.search(r'(?:1/2|½)\s*CS\b', product_upper)
    if case_match:
        return _done(get_case_weight(product_text) * 0.5, "lb", "matched half-case")

    case_match = re.search(r'\bCS\b', product_upper)
    if case_match:
        return _done(get_case_weight(product_text), "lb", "matched full case")

    # WEIGHT RANGES
    range_match = re.search(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s+(LBS?|OZ|KG|G)\b', product_upper)
    if range_match:
        midpoint = (float(range_match.group(1)) + float(range_match.group(2))) / 2
        result = to_weight(midpoint, range_match.group(3))
        if result is not None:
            debug_print(f"'{product_text}' matched weight range → {result:.4f} {U}", row_sheet)
            return result, "weight_range"

    # CT + WEIGHT
    ct_weight_patterns = [
        (r'\b(\d+)\s*CT\s+(\d+(?:\.\d+)?)\s*LBS?\b', lambda m: float(m.group(2)), "lb"),
        (r'\b(\d+)\s*CT\s+(\d+(?:\.\d+)?)\s*KG\b', lambda m: float(m.group(2)), "kg"),
        (r'\b(\d+)\s*CT\s+(\d+(?:\.\d+)?)\s*OZ\b', lambda m: float(m.group(2)) / 16, "lb"),
        (r'\b(\d+)\s*CT\s+(\d+(?:\.\d+)?)\s*GR?\b', lambda m: float(m.group(2)) / 1000, "kg"),
    ]
    for pattern, calc_func, native in ct_weight_patterns:
        match = re.search(pattern, product_upper)
        if match:
            try:
                result = calc_func(match)
                if result and result > 0:
                    return _done(result, native, "matched CT+weight", "ct_total_weight")
            except:
                continue

    # PATTERNS: (regex, function, native unit of the function's result).
    # A native unit of None means the function returns (value, unit) itself.
    dens = lambda: liquid_kg_per_liter(product_text)
    patterns = [
        (r'\b(\d+(?:\.\d+)?)\s*DOZ\b', lambda m: float(m.group(1)) * 1.5, "lb"),
        (r'(\d+)X#(\d+)\b', lambda m: int(m.group(1)) * CAN_WEIGHTS.get(int(m.group(2)), 6.5), "lb"),
        (r'(\d+)X(\d*\.?\d+)\s*OZ\b', lambda m: int(m.group(1)) * float(m.group(2)) / 16, "lb"),
        (r'(\d+)X(\d*\.?\d+)\s*LBS?\b', lambda m: int(m.group(1)) * float(m.group(2)), "lb"),
        (r'(\d+)X(\d*\.?\d+)\s*KG\b', lambda m: int(m.group(1)) * float(m.group(2)), "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*GR?\b', lambda m: int(m.group(1)) * float(m.group(2)) / 1000, "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*L(?:T|TR?)?\b', lambda m: int(m.group(1)) * float(m.group(2)) * dens(), "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*CL\b', lambda m: int(m.group(1)) * float(m.group(2)) / 100 * dens(), "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*DL\b', lambda m: int(m.group(1)) * float(m.group(2)) / 10 * dens(), "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*L(?:T|TR?)?\b', lambda m: float(m.group(1)) * dens(), "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*ML\b', lambda m: int(m.group(1)) * float(m.group(2)) / 1000 * dens(), "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*ML\b', lambda m: float(m.group(1)) / 1000 * dens(), "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*CL\b', lambda m: float(m.group(1)) / 100 * dens(), "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*DL\b', lambda m: float(m.group(1)) / 10 * dens(), "kg"),
        (r'(\d+)X(\d*\.?\d+)\s*PT\b', lambda m: int(m.group(1)) * float(m.group(2)) * 1.0, "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*PT\b', lambda m: float(m.group(1)) * 1.0, "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*OZ\s*\((\d+)\)', lambda m: float(m.group(1)) * int(m.group(2)) / 16, "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*QT\s*\((\d+)\)', lambda m: float(m.group(1)) * int(m.group(2)) * get_liquid_density(product_text) * 0.25, "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*PT\s*\((\d+)\)', lambda m: float(m.group(1)) * int(m.group(2)) * get_liquid_density(product_text) * 0.125, "lb"),
        (r'(\d+)X(?:1/2|½)\s*GAL\b', lambda m: int(m.group(1)) * 0.5 * get_liquid_density(product_text), "lb"),
        (r'(\d+)X(\d*\.?\d+(?:/\d+)?)\s*GAL\b', lambda m: int(m.group(1)) * _parse_fraction(m.group(2)) * get_liquid_density(product_text), "lb"),
        (r'\b(\d+(?:\.\d+)?(?:/\d+)?)\s*GAL\b', lambda m: _parse_fraction(m.group(1)) * get_liquid_density(product_text), "lb"),
        (r'(\d+)X(\d*\.?\d+(?:/\d+)?)\s*QT\b', lambda m: int(m.group(1)) * _parse_fraction(m.group(2)) * get_liquid_density(product_text) * 0.25, "lb"),
        (r'\b(\d+(?:\.\d+)?(?:/\d+)?)\s*QT\b(?!\s*\()', lambda m: _parse_fraction(m.group(1)) * get_liquid_density(product_text) * 0.25, "lb"),
        (r'(?:^|\s)(\d*\.?\d+)\s*LBS?\b', lambda m: float(m.group(1)), "lb"),
        (r'\b(\d+)\s*LBS?\b(?!\s*MC|\s*ATL)', lambda m: float(m.group(1)), "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*KG\b', lambda m: float(m.group(1)), "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*GR?\b', lambda m: float(m.group(1)) / 1000, "kg"),
        (r'\b(\d+(?:\.\d+)?)\s*OZ\b', lambda m: float(m.group(1)) / 16, "lb"),
        (r'\b(\d+)\s*BUNCH(?:ES)?\b', lambda m: int(m.group(1)) * 0.25, "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*CT\b', lambda m: estimate_ct_weight(float(m.group(1)), product_text, row_sheet), None),
        (r'\b(\d+(?:\.\d+)?)\s*P(?:CS?|IECES?)\b', lambda m: estimate_piece_weight(float(m.group(1)), product_text, row_sheet), "lb"),
        (r'\b(\d+(?:\.\d+)?)\s*ST(?:K|Ü?CK|UECK)\b', lambda m: estimate_piece_weight(float(m.group(1)), product_text, row_sheet), "lb"),
    ]

    for pattern, calc_func, native in patterns:
        match = re.search(pattern, product_upper)
        if match:
            try:
                result = calc_func(match)
                if native is None:
                    result, native = result
                if result and result > 0:
                    return _done(result, native, "matched")
            except Exception as e:
                continue

    return None, "unknown"

def estimate_ct_weight(count: float, product_text: str, row_sheet: Optional[int] = None):
    """(total weight, unit) for a count of items: item_weights.json first, else rough lb defaults."""
    text_lower = product_text.lower()
    for pattern, weight in ITEM_WEIGHTS.items():
        if re.search(pattern, text_lower):
            total_weight = count * weight
            debug_print(f"CT: {count} × {weight} = {total_weight} {ITEM_WEIGHTS_UNIT}", row_sheet)
            return total_weight, ITEM_WEIGHTS_UNIT
    if re.search(r'\bsheets?\b|\bpapers?\b', text_lower):
        return count * 0.1, "lb"
    elif re.search(r'\bsmalls?\b|\bminis?\b', text_lower):
        return count * 0.16, "lb"
    else:
        return count * 0.25, "lb"

def estimate_piece_weight(count: float, product_text: str, row_sheet: Optional[int] = None) -> float:
    """Total weight in lb for a count of pieces."""
    text_lower = product_text.lower()
    if re.search(r'\bwatermelons?\b|\bpumpkins?\b', text_lower):
        return count * 15.0
    elif re.search(r'\bpineapples?\b|\bcantaloupes?\b|\bhoneydewe?\b', text_lower):
        return count * 3.0
    elif re.search(r'\bturkeys?\b|\bducks?\b', text_lower):
        return count * 12.0
    elif re.search(r'\bchickens?\b|\bfish\b', text_lower):
        return count * 4.0
    else:
        return count * 2.0

# ============================================================================
# AI INTEGRATION
# ============================================================================

def require_gpt():
    if "_call_gpt" not in globals() or not callable(globals()["_call_gpt"]):
        raise RuntimeError("[Cell 5] _call_gpt is not defined.")

def analyze_product_with_ai(product_text: str, estimated_weight=None, row_sheet: Optional[int] = None):
    require_gpt()
    weight_hint = f" (estimated total weight: {estimated_weight} {_unit_label()})" if estimated_weight else ""
    prompt = f"""Analyze this food item: {product_text}{weight_hint}

Rules:
1. Single products (cheese, butter, milk) → 100% of that food
2. Prepared foods → break into main ingredients
3. Think like a chef, not a chemist

{PRODUCT_FLAG_RULES}

Return JSON: {{"items": [{{"ingredient": "name", "percent": number}}], "animal_derived": false, "neither_plant_nor_animal": false, "diluted_beverage": false}}"""

    try:
        response = _call_gpt(messages=[
            {"role": "system", "content": "You identify main food ingredients in products as they appear in recipes."},
            {"role": "user", "content": prompt}
        ])
        response_text = str(response).strip()
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            response_text = json_match.group(0)
        data = json.loads(response_text)
        store_product_flags(product_text, data)   # whole-product answers for the plant-based judgment
        items = data.get("items", [])
        if not items:
            return get_fallback_ingredients(product_text)
        valid_items = []
        total_percent = 0
        for item in items:
            ingredient = str(item.get("ingredient", "")).strip().lower()
            if not ingredient or ingredient == "null":
                continue
            percent = float(item.get("percent", 0))
            if percent > 0:
                valid_items.append({"ingredient": ingredient, "percent": percent})
                total_percent += percent
        if valid_items and abs(total_percent - 100) > 0.1:
            for item in valid_items:
                item["percent"] = (item["percent"] / total_percent) * 100
        return valid_items if valid_items else get_fallback_ingredients(product_text)
    except Exception as e:
        return get_fallback_ingredients(product_text)

def get_fallback_ingredients(product_text: str):
    text_lower = product_text.lower()
    ingredient_map = {
        r'\b(butter)\b': 'butter', r'\b(cheese)\b': 'cheese', r'\b(milk)\b': 'milk',
        r'\b(cream)\b': 'cream', r'\b(yogurt)\b': 'yogurt', r'\b(bread)\b': 'bread',
        r'\b(chicken)\b': 'chicken', r'\b(beef)\b': 'beef', r'\b(pork)\b': 'pork',
        r'\b(fish)\b': 'fish', r'\b(oil)\b': 'oil', r'\b(flour)\b': 'flour',
    }
    for pattern, ingredient in ingredient_map.items():
        if re.search(pattern, text_lower):
            return [{"ingredient": ingredient, "percent": 100.0}]
    words = re.findall(r'\b[a-z]{3,}\b', text_lower)
    ingredient = words[0] if words else "food product"
    return [{"ingredient": ingredient, "percent": 100.0}]

def estimate_weight_with_ai(product_text: str, qty: float, row_sheet: Optional[int] = None) -> Optional[float]:
    """Total weight in the run's weight unit when no weight could be read from the text."""
    text_lower = product_text.lower()
    U = _unit_label()
    for fruit, each_lb in (("watermelon", 12.0), ("cantaloupe", 3.0), ("pineapple", 4.0)):
        if fruit in text_lower:
            result = _in_run_unit(each_lb * qty, "lb")
            debug_print(f"'{product_text}' AI estimate ({fruit}) → {result:.4f} {U}", row_sheet)
            return result
    metric = (WEIGHT_UNIT == "kg")
    try:
        require_gpt()
        if metric:
            prompt = f"""Estimate the total weight in kilograms for this produce/food item.

Product: {product_text}
Quantity: {qty}

Context:
- Case / box / carton (CS, CTN, KRT, colis) of produce typically weighs 5-25 kg depending on item
- Half case = half of that
- Berries in boxes typically weigh 2-6 kg per box
- Leafy greens in boxes typically weigh 5-12 kg per box
- Root vegetables in sacks or boxes typically weigh 10-25 kg
- Dense vegetables (peppers, tomatoes) in boxes typically weigh 5-10 kg
- PCS / STK / ST = pieces

Think step by step:
1. Identify the product type
2. Identify the packaging (case, box, sack, count, weight unit)
3. Estimate reasonable weight per unit
4. Calculate total weight

Provide only the final numeric answer in kilograms (e.g., "11.5")"""
            system = ("You are an expert in produce wholesale and food service packaging, including metric "
                      "packaging conventions (kg, g, L, cases/boxes, pieces).")
        else:
            prompt = f"""Estimate the total weight in pounds for this produce/food item.

Product: {product_text}
Quantity: {qty}

Context:
- CS = case (full case of produce, typically 15-50 lbs depending on item)
- 1/2 CS = half case
- BU/BUSH = bushel (typically 30-60 lbs depending on item)
- Berries in cases typically weigh 10-15 lbs per case
- Leafy greens in cases typically weigh 20-30 lbs per case
- Root vegetables in cases typically weigh 40-60 lbs per case
- Dense vegetables (peppers, tomatoes) in cases typically weigh 20-30 lbs per case

Think step by step:
1. Identify the product type
2. Identify the packaging (case, half-case, bushel, count, weight unit)
3. Estimate reasonable weight per unit
4. Calculate total weight

Provide only the final numeric answer in pounds (e.g., "24.5")"""
            system = ("You are an expert in produce wholesale and food service packaging. You understand industry "
                      "abbreviations like CS (case), BU (bushel), and typical weights for produce items in commercial foodservice.")
        response = _call_gpt(messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ])

        weight_match = re.search(r'(\d+(?:\.\d+)?)', str(response))
        if weight_match:
            weight = float(weight_match.group(1))
            if 0.1 <= weight <= 10000:
                result = round(weight, 4)
                debug_print(f"'{product_text}' AI estimate (GPT) → {result:.4f} {U}", row_sheet)
                return result
    except:
        pass
    result = _in_run_unit(3.0 * qty, "lb")
    debug_print(f"'{product_text}' AI estimate (fallback default) → {result:.4f} {U}", row_sheet)
    return result

# ============================================================================
# MAIN PROCESSING
# ============================================================================

def process_single_product(index: int, product_text: str, df_raw: pd.DataFrame, qty_col: Optional[str]):
    src_row = df_raw.loc[index, "__src_row"] if "__src_row" in df_raw.columns else index
    row_sheet = sheet_row(src_row)   # the line's row number in the worksheet (Cell 2)
    
    try:
        if qty_col and qty_col in df_raw.columns:
            qty_value = df_raw.loc[index, qty_col]
            if pd.notna(qty_value) and qty_value != '':
                qty_final = max(float(qty_value), 1.0)
            else:
                qty_final = 1.0
        else:
            qty_final = 1.0
    except:
        qty_final = 1.0
    
    if not product_text or not isinstance(product_text, str):
        return None
    product_text_clean = str(product_text).strip()
    if len(product_text_clean) < 3 or product_text_clean.lower() in ['type01', 'type02', 'type03', 'nan', 'null']:
        return None

    per_unit_weight, source_type = extract_weight_from_text(product_text_clean, row_sheet)
    if per_unit_weight:
        total_weight = round(per_unit_weight * qty_final, 4)
    else:
        debug_print(f"No weight pattern matched, estimating via AI", row_sheet)
        total_weight = estimate_weight_with_ai(product_text_clean, qty_final, row_sheet)
        source_type = "ai_estimated"
   
    items = analyze_product_with_ai(product_text_clean, total_weight, row_sheet)
    
    return {
        "index": index, "row_sheet": row_sheet, "product": product_text_clean, "qty": qty_final,
        "total_weight": total_weight, "items": items, "source_type": source_type,
        "attributes": product_attributes(src_row, product_text_clean, items),
    }

def run_stage_1_type03_gpt_base(input_filename: str, base_name: str) -> pd.DataFrame:
    with log_time("Loading input data"):
        reset_sheet_rows()
        df_raw = read_input_csv(input_filename)
        df_raw["__src_row"] = df_raw.index   # worksheet row, to find the row's INPUT_ATTRIBUTES
        unwanted = {"type01", "type02", "type03"}
        if len(df_raw.columns) > 0:
            mask1 = ~(df_raw[df_raw.columns[0]].notna() & df_raw[df_raw.columns[0]].astype(str).str.strip().str.lower().isin(unwanted))
        else:
            mask1 = pd.Series([True] * len(df_raw))
        if len(df_raw.columns) > 1:
            mask2 = ~(df_raw[df_raw.columns[1]].notna() & df_raw[df_raw.columns[1]].astype(str).str.strip().str.lower().isin(unwanted))
        else:
            mask2 = pd.Series([True] * len(df_raw))
        # "Total" / "Grand Total" lines are not products (Cell 2)
        _first = df_raw.columns[0] if len(df_raw.columns) else None
        _pc = next((c for c in df_raw.columns if str(c).lower().strip() in ("product", "description")), _first)
        mask3 = ~df_raw[_pc].astype(str).map(is_total_line) if _pc is not None else True
        df_raw = df_raw[mask1 & mask2 & mask3].reset_index(drop=True)

    columns = {c.lower().strip(): c for c in df_raw.columns}
    product_col = columns.get("product") or columns.get("description") or df_raw.columns[0]
    qty_col = columns.get("qty")
    
    product_series = df_raw[product_col].astype(str)

    # Decide the run's weight unit from the units written in the descriptions (and a unit column, if any)
    n_metric, n_us = count_unit_mentions(product_series)
    unit_col = columns.get("unit") or columns.get("stock unit") or columns.get("uom")
    if unit_col:
        _sys = df_raw[unit_col].map(uom_system)
        n_metric += int(_sys.eq("kg").sum())
        n_us += int(_sys.eq("lb").sum())
    resolve_weight_unit(n_metric, n_us, context=f"{Path(input_filename).name} (type03)")
    suf = weight_suffix()

    max_workers = int(os.getenv("FI_MAX_WORKERS", "4"))
    row_offset = int(os.getenv("FI_ROW_OFFSET", "2"))
    
    results = []
    with log_time(f"Processing {len(product_series)} products"):
        with cf.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_single_product, i, text, df_raw, qty_col): (i, text)
                      for i, text in product_series.items() if pd.notna(text) and str(text).strip()}
            for future in cf.as_completed(futures):
                result = future.result()
                if result:
                    results.append(result)
    
    output_rows = []
    for result in results:
        for item in result["items"]:
            ingredient = item.get("ingredient", "").strip() or "food product"
            percent = item.get("percent", 100.0)
            ingredient_weight = round((percent * result["total_weight"] / 100), 4) if result["total_weight"] and percent else None
            output_rows.append({
                "row_sheet": result["row_sheet"],
                "product": result["product"],
                "qty": result["qty"],
                f"product_weight_{suf}": result["total_weight"],
                "ingredient": ingredient,
                "percent": round(percent, 2) if percent else None,
                f"ingredient_weight_{suf}": ingredient_weight,
                "extract_origin": "ai_analysis",
                "source_uom_type": result["source_type"],
                **result["attributes"],
            })

    with log_time("Creating output file"):
        df_output = pd.DataFrame(output_rows)
        if not df_output.empty:
            df_output = df_output.sort_values("row_sheet")
            for col in [f"product_weight_{suf}", "percent", f"ingredient_weight_{suf}"]:
                if col in df_output.columns:
                    df_output[col] = df_output[col].round(2)
        output_path = result_out(f"ingredients_{base_name}.csv")
        df_output.to_csv(output_path, index=False, encoding="utf-8")
        print(f"[Stage 1] Wrote {len(df_output)} rows to {output_path}")
    return df_output

try:
    _SESSION_START_TIME
except NameError:
    import time
    _SESSION_START_TIME = time.time()

def run_stage_1_type03_gpt(*args, **kwargs):
    base_name = kwargs.get("base") or (args[1] if len(args) >= 2 else None)
    output_file = result_out(f"ingredients_{base_name}.csv") if base_name else None
    if output_file and output_file.exists():
        try:
            if output_file.stat().st_mtime >= _SESSION_START_TIME:
                return pd.read_csv(output_file)
        except:
            pass
    return run_stage_1_type03_gpt_base(*args, **kwargs)

print("=== CELL 5 v110b COMPLETE ===")
print("✓ All fixes applied")
print("✓ Ready to process")

In [ ]:
# === Cell 6 — Stage 1 (General) Runner ===
# Purpose: Execute AI ingredient extraction for type01 and type02 vendor formats.
# Reads the procurement table, extracts MAJOR ingredients (+ percents), writes ingredients_{BASE}.csv
# Provides log_time timings.
# Inputs:
#   - standardized DataFrame from Cell 3
#   - ai_extract_ingredients() from Cell 4
# Outputs:
#   - ingredients_{BASE}.csv (weights in the worksheet's unit: product_weight_kg / ingredient_weight_kg,
#     or product_weight_lbs / ingredient_weight_lbs)
# Behavior:
#   - Loops through all products, calling GPT for ingredient prediction.
#   - Applies label safeguards: vegan/plant-based (no animal ingredients), dairy-free/non-dairy (no dairy
#     only -- dairy-free is NOT assumed to be vegan), vegetarian/meatless (no meat or fish); normalization.
#   - Judges each product plant-based or not and copies the worksheet's certification columns
#     (product_attributes, Cell 4) onto every ingredient row of the product.
#   - Writes results to disk.
# Notes:
#   - This cell mirrors Cell 5 but omits item_weights.json dependency.
#   - Overwrites existing ingredients file if START_MODE='full'.

from __future__ import annotations
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import concurrent.futures as cf
import pandas as pd
import numpy as np
import json, re, os


if file_type == 'type03':
    print("Skipping this cell - type03 uses different processing")
    # You could also just use a pass statement or return if it's in a function
else:

    # ----------------------------
    # Timing helper (no-op fallback)
    # ----------------------------
    try:
        log_time
    except NameError:
        from contextlib import contextmanager
        @contextmanager
        def log_time(_label: str):
            yield
    
    # ----------------------------
    # Constants / configuration
    # ----------------------------
    # Product labels, matched on normalized text (lower case, hyphens -> spaces, so "non-dairy" = "non dairy").
    # Each label rules out only what it says: dairy-free is not assumed to be vegan, and vegetarian
    # products may contain dairy and eggs.
    # (Own names: Stage 2 defines a different VEGAN_TERMS later in the notebook.)
    S1_VEGAN_TERMS = ("vegan", "plant based", "plant derived")              # no animal ingredients at all
    S1_DAIRY_FREE_TERMS = ("dairy free", "non dairy", "nondairy", "milk free") # no dairy
    S1_VEGETARIAN_TERMS = ("vegetarian", "meatless", "meat free")            # no meat, poultry, fish, seafood

    VEGAN_INGREDIENT_REPLACEMENTS = {
    # Dairy products
    "yogurt": "non-dairy yogurt",
    "milk": "plant milk",
    "cheese": "vegan cheese",
    "cream": "plant-based cream",
    "butter": "vegan butter",
    "sour cream": "vegan sour cream",
    "cream cheese": "vegan cream cheese",
    "ice cream": "non-dairy ice cream",
    
    # Meat products (for vegan meat substitutes)
    "chicken": "plant-based chicken",
    "beef": "plant-based beef", 
    "pork": "plant-based pork",
    "turkey": "plant-based turkey",
    "sausage": "plant-based sausage",
    "bacon": "plant-based bacon",
    
    # Other animal products
    "egg": "egg substitute",
    "gelatin": "plant-based gelling agent",
    "honey": "plant-based sweetener",
}
    
    ANIMAL_INGREDIENT_TOKENS = {
        # dairy
        "milk","dairy milk","cow milk","goat milk","sheep milk","cheese","whey","casein","lactose",
        "butter","ghee","cream","yogurt","buttermilk","skim milk","whole milk","milk powder","whey powder","sour cream",
        # eggs & honey & gelatin
        "egg","eggs","egg white","egg whites","egg yolk","honey","gelatin",
        # terrestrial meats
        "beef","pork","chicken","turkey","lamb","mutton","bacon","sausage","venison","veal","ham",
        # fish & seafood
        "fish","salmon","tuna","cod","haddock","pollock","anchovy","sardine","halibut","mahi",
        "catfish","tilapia","trout","bass","shrimp","prawn","crab","lobster","oyster","clam","mussel",
    }
    
    # Fish species tokens for annotation
    FISH_SPECIES = {
        "fish","salmon","tuna","cod","haddock","pollock","anchovy","sardine","halibut","mahi",
        "catfish","tilapia","trout","bass","hake","sole","flounder","mackerel","herring","anchovies","sardines"
    }
    
    # “Do not decompose” labels (top-level categories kept intact if name matches exactly)
    DO_NOT_DECOMPOSE = set(globals().get("ALLOWED_CATEGORIES", []))
    
    # Protected multi-word items that shouldn’t be split into subcomponents
    PROTECTED_INGREDIENTS = {
    "almond milk","soy milk","oat milk","rice milk","pea milk","coconut milk","cashew milk",
    "peanut butter","ice cream","tofu","tempeh","seitan","vegan cheese alternative",
    "vegan mozzarella cheese alternative","non-dairy yogurt","coconut yogurt",
    "almond yogurt","soy yogurt","cashew yogurt","plant-based cream","vegan butter",
    "plant-based chicken","plant-based beef","plant-based pork","vegan sausage",
}
    
    # Single-item detectors (collapse obvious single-ingredient staples)
    SINGLE_ITEM_DETECTORS: List[Tuple[re.Pattern,str]] = [
        (re.compile(r"\bcheese\b", re.I), "cheese"),
        (re.compile(r"\bwater\b", re.I), "water"),
        (re.compile(r"\bsugar\b", re.I), "sugar"),
        (re.compile(r"\bsalt\b", re.I), "salt"),
        (re.compile(r"\bflour\b", re.I), "flour"),
        (re.compile(r"\bbutter\b", re.I), "butter"),
        (re.compile(r"\bcream\b", re.I), "cream"),
        (re.compile(r"\bmilk\b", re.I), "milk"),
        (re.compile(r"\byogurt\b", re.I), "yogurt"),
        (re.compile(r"\b(oil|olive oil|canola oil|vegetable oil|shortening)\b", re.I), "oil"),
        (re.compile(r"\brice\b", re.I), "rice"),
        (re.compile(r"\boats?\b", re.I), "oats"),
        (re.compile(r"\bpasta\b|\bspaghetti\b|\bpenne\b", re.I), "pasta"),
    ]
    
    # Words allowed next to each staple in the product's main name for the single-ingredient shortcut.
    # The shortcut is used only when the main name (text before the first comma, e.g. "MILK" in
    # "MILK, 1% LOW FAT...") consists of the staple plus such words -- so "HAM, ... WATER-ADDED",
    # "CHEESE, MOZZARELLA ... WHOLE-MILK" or "HONEY OAT GRANOLA" go to GPT instead.
    SINGLE_STAPLE_WORDS = {
        "cheese": {"cheese", "cheddar", "mozzarella", "ricotta", "parmesan", "parmigiano", "reggiano", "romano",
                   "provolone", "swiss", "american", "jack", "monterey", "pepper", "colby", "feta", "gouda", "brie",
                   "blue", "bleu", "gorgonzola", "muenster", "havarti", "asiago", "fontina", "gruyere", "mascarpone",
                   "queso", "fresco", "cotija", "manchego", "cream", "cottage", "goat", "string", "shredded",
                   "shred", "sliced", "block", "grated", "blend", "sharp", "mild", "aged", "smoked"},
        "water":  {"water", "spring", "sparkling", "mineral", "purified", "bottled", "distilled", "drinking", "still"},
        "sugar":  {"sugar", "cane", "granulated", "powdered", "confectioners", "superfine", "turbinado", "raw"},
        "salt":   {"salt", "sea", "kosher", "iodized", "table", "fine", "coarse"},
        "flour":  {"flour", "wheat", "all", "purpose", "bread", "pastry", "cake", "unbleached", "bleached",
                   "enriched", "semolina", "durum"},
        "butter": {"butter", "salted", "unsalted", "sweet", "cultured", "european", "style", "solid", "whipped"},
        "cream":  {"cream", "heavy", "whipping", "light", "sour", "ultra", "pasteurized", "half"},
        "milk":   {"milk", "whole", "skim", "lowfat", "nonfat", "reduced", "chocolate", "vitamin", "d", "ultra",
                   "pasteurized", "homogenized", "lactose", "free"},
        "yogurt": {"yogurt", "greek", "plain", "vanilla", "strawberry", "blueberry", "fruit", "lowfat", "nonfat"},
        "oil":    {"oil", "olive", "canola", "vegetable", "soybean", "sunflower", "corn", "peanut", "avocado",
                   "sesame", "coconut", "rice", "bran", "grapeseed", "safflower", "blend", "frying", "salad",
                   "pomace", "virgin", "shortening", "liquid", "clear", "fry", "pan", "spray"},
        "rice":   {"rice", "jasmine", "basmati", "arborio", "long", "short", "medium", "grain", "parboiled",
                   "converted", "instant", "sushi", "calrose", "wild"},
        "oats":   {"oats", "oat", "rolled", "steel", "cut", "quick", "old", "fashioned", "instant"},
        "pasta":  {"pasta", "spaghetti", "penne", "rigatoni", "rotini", "fusilli", "linguine", "fettuccine",
                   "macaroni", "elbow", "elbows", "ziti", "farfalle", "orzo", "shells", "angel", "hair", "lasagna",
                   "noodles", "durum", "semolina"},
    }
    # Descriptive, packaging and unit words ignored when checking the main name
    SINGLE_FILLER_WORDS = {
        "organic", "natural", "pure", "plain", "fresh", "frozen", "raw", "white", "brown", "regular", "original",
        "extra", "fancy", "grade", "a", "aa", "usda", "premium", "select", "choice", "bulk", "dry", "dried",
        "low", "fat", "unsweetened", "sweetened", "unflavored", "domestic", "imported", "gluten", "gmo", "non",
        "kosher", "halal", "the", "and", "of", "with", "w", "in", "for", "pack", "case", "cs", "bag", "box",
        "jug", "bottle", "bottles", "carton", "can", "canned", "tub", "pail", "plastic", "paper", "glass",
        "gal", "gallon", "gallons", "qt", "quart", "pt", "pint", "lb", "lbs", "oz", "fl", "kg", "g", "l", "ml",
        "ct", "count", "ea", "each", "ss", "ref", "tff", "shelf", "stable", "iw", "single", "serve",
    }

    def _single_shortcut_ok(title: str, name: str) -> bool:
        """True if the product's main name is just the staple (plus descriptive/packaging words)."""
        head = title.split(",")[0]
        words = [w for w in _norm(head).replace("/", " ").split()
                 if not re.search(r"\d", w) and w not in SINGLE_FILLER_WORDS]
        allowed = SINGLE_STAPLE_WORDS.get(name, {name})
        return bool(words) and all(w in allowed or w.rstrip("s") in allowed for w in words)

    # Packaging-water context (avoid collapsing to “water” just because “packed in water”)
    PACKAGING_WATER_CONTEXT_RE = re.compile(r"\b(in|with)\s+water\b|\bwater\s*pack(ed)?\b", re.I)
    
    # Half & Half canonical rule
    HALF_AND_HALF_RE = re.compile(r"\bhalf\s*&\s*half\b|\bhalf\s+and\s+half\b", re.I)
    
    # Ratios like 75/25
    RATIO_RE = re.compile(r'(\d{1,3})\s*/\s*(\d{1,3})(?:\s*/\s*(\d{1,3}))?')
    
    # Threshold to drop micro-ingredients post-normalization
#    MIN_KEEP_PCT = float(os.getenv("FI_MIN_KEEP_PCT", "5")) old version deleted
    MIN_KEEP_PCT = CATEGORY_PERCENT_MIN
    
    # Vegan meat default mix (when GPT returns weak/unknown for vegan meat substitutes)
    VEGAN_MEAT_DEFAULT = [
        {"ingredient": "legumes",          "percent": 30.0},
        {"ingredient": "wheat",            "percent": 30.0},
        {"ingredient": "other vegetables", "percent": 30.0},
        {"ingredient": "vegetable oil",    "percent": 10.0},
    ]
    
    # ----------------------------
    # Helpers
    # ----------------------------
    def _norm(s: str) -> str:
        try:
            return norm_text(s)  # use your notebook's normalizer if present
        except Exception:
            t = str(s).strip().lower()
            return re.sub(r"\s+", " ", t)

    def _is_blank_product(s: Any) -> bool:
        if s is None:
            return True
        t = str(s).strip()
        if t == "":
            return True
        tl = t.lower()
        return tl in ("nan", "none")
    
    def is_vegan_labeled(name: str) -> bool:
        """True only for fully plant-based labels (vegan / plant-based), not dairy-free or vegetarian."""
        n = _norm(name)
        return any(re.search(rf"\b{t}\b", n) for t in S1_VEGAN_TERMS)

    def label_exclusions(name: str) -> set:
        """Ingredient groups a product's label rules out: 'dairy', 'meat', 'fish', 'eggs', 'honey', 'gelatin'."""
        n = _norm(name)
        has = lambda terms: any(re.search(rf"\b{t}\b", n) for t in terms)
        ex = set()
        if has(S1_VEGAN_TERMS):
            ex |= {"dairy", "meat", "fish", "eggs", "honey", "gelatin"}
        if has(S1_DAIRY_FREE_TERMS):
            ex |= {"dairy"}
        if has(S1_VEGETARIAN_TERMS):
            ex |= {"meat", "fish", "gelatin"}
        return ex

    # Word lists for removing ruled-out ingredients (whole words, so "eggplant" and "graham" are safe)
    _EXCLUDE_WORDS = {
        "dairy": r"milk|cream|butter|buttermilk|cheese|cheddar|mozzarella|parmesan|yogurt|yoghurt|whey|casein|ghee|lactose",
        "meat": r"beef|pork|chicken|turkey|lamb|mutton|bacon|sausage|ham|veal|venison|duck|goat|meat",
        "fish": r"fish|salmon|tuna|shrimp|prawns?|cod|crab|lobster|anchov\w*|sardines?|tilapia|clams?|oysters?|mussels?|scallops?|squid",
        "eggs": r"eggs?|egg whites?|egg yolks?",
        "honey": r"honey",
        "gelatin": r"gelatin|gelatine",
    }
    # Plant-based versions of those words are kept ("almond milk", "peanut butter", "vegan cheese", "coconut cream")
    _PLANT_VARIANT_RX = re.compile(r"\b(?:vegan|plant|plant based|non dairy|dairy free|nondairy|coconut|almond|oat|oats|"
                                   r"soy|soya|cashew|rice|peanut|cocoa|cacao|shea|nut|nuts|seed|seeds|hemp|imitation|"
                                   r"substitute|alternative|meatless|meat free)\b")

    def _plant_version(ing: str) -> str:
        """Plant-based stand-in for a ruled-out ingredient (keeps its share of the product)."""
        n = _norm(ing)
        for key, new in VEGAN_INGREDIENT_REPLACEMENTS.items():
            if n == _norm(key):
                return new
        if re.search(r"\b(?:eggs?|egg whites?|egg yolks?)\b", n):
            return "plant-based egg substitute"
        if re.search(r"\bhoney\b", n):
            return "plant-based sweetener"
        if re.search(r"\bgelatine?\b", n):
            return "plant-based gelling agent"
        return "plant-based " + str(ing).strip()

    def replace_excluded_items(items: List[Dict[str, Any]], excluded: set) -> List[Dict[str, Any]]:
        """Swap ingredients the label rules out for plant-based versions (e.g. 'chicken' in a meatless
        product -> 'plant-based chicken'), keeping their percentages; plant versions already named are kept."""
        if not items or not excluded:
            return items
        rx = re.compile(r"\b(?:" + "|".join(_EXCLUDE_WORDS[g] for g in excluded if g in _EXCLUDE_WORDS) + r")\b")
        out = []
        for it in items:
            ing = _norm(it.get("ingredient", ""))
            if rx.search(ing) and not _PLANT_VARIANT_RX.search(ing):
                it = dict(it)
                it["ingredient"] = _plant_version(it.get("ingredient", ""))
            out.append(it)
        return out

    def remove_excluded_items(items: List[Dict[str, Any]], excluded: set) -> List[Dict[str, Any]]:
        """Drop ingredients in the excluded groups, keeping plant-based versions (e.g. 'almond milk')."""
        if not items or not excluded:
            return items
        rx = re.compile(r"\b(?:" + "|".join(_EXCLUDE_WORDS[g] for g in excluded if g in _EXCLUDE_WORDS) + r")\b")
        out = []
        for it in items:
            ing = _norm(it.get("ingredient", ""))
            if rx.search(ing) and not _PLANT_VARIANT_RX.search(ing):
                continue
            out.append(it)
        return out
    
    def contains_animal_token(name_lc: str) -> bool:
        return any(tok in name_lc for tok in ANIMAL_INGREDIENT_TOKENS)
    
    def is_allowed_plant_variant(name_lc: str) -> bool:
        # allow explicit plant milks etc.
        for prot in PROTECTED_INGREDIENTS:
            if prot in name_lc:
                return True
        return False
    
    def sanitize_vegan_dairy_naming(s: str) -> str:
        # avoid returning bare “mozzarella” for vegan cheese; keep it generic
        if "mozzarella" in s.lower() and any(k in s.lower() for k in ("vegan","plant")):
            return "vegan mozzarella cheese alternative"
        return s
    
    def is_protected(ing: str) -> bool:
        ing_lc = ing.lower()
        return any(p in ing_lc for p in PROTECTED_INGREDIENTS)
    
    # ------------------------------------------------------------------
    # Species keywords used by detect_bone_in() / detect_fish_cut() below.
    #
    # _LAND_ANIMAL_RX: the land-animal species Cell 8's EDIBLE_YIELD_DEFAULTS
    #   applies a bone-in discount to (chicken/turkey/beef/pork/lamb/goat).
    # _FISH_SPECIES_RX: finfish names. "\w*fish" also catches compound names
    #   (swordfish, monkfish, rockfish, whitefish, ...), and common market
    #   fish that are not factors.csv categories (snapper, grouper, ...) are
    #   listed so their cut language is still read.
    # _SHRIMP_RX / _CRAB_RX / _LOBSTER_RX / _CEPHALOPOD_RX: the shellfish and
    #   cephalopod categories in factors.csv. They have their own
    #   FISH_CUT_YIELDS curves in Cell 8 but use different trade vocabulary
    #   from finfish (head-on/peeled, clusters/meat, tails, tubes/rings), so
    #   detect_fish_cut() handles each with its own rules.
    # Keep these in sync with Cell 8's _SPECIES_PATTERNS.
    # ------------------------------------------------------------------
    _LAND_ANIMAL_RX = re.compile(r"\b(?:chicken|turkey|beef|bovine|cow|pork|swine|lamb|mutton|goat)\b")
    _FISH_SPECIES_RX = re.compile(r"""
        \b(?:\w*fish|salmon|tuna|cod|haddock|pollock|pollack|scrod|anchov\w*|sardine\w*|halibut|mahi\w*|
             tilapia|trout|bass|hake|sole|flounder|whiting|
             carp|congers?|mullets?|sauries?|smelt|amberjack|hamachi|yellowtail|jacks|unagi|eels?|barramundi|
             mackerel|herring|branzino|swai|basa|pangasius|
             snapper|grouper|pompano|perch|pike|walleye|cobia|wahoo|bream|dory|opah|roughy|
             shad|sturgeon|turbot|plaice)\b
    """, re.VERBOSE)
    _SHRIMP_RX     = re.compile(r"\b(?:shrimps?|prawns?)\b")
    _CRAB_RX       = re.compile(r"\bcrabs?\b")
    _LOBSTER_RX    = re.compile(r"\blobsters?\b")
    _OCTOPUS_RX    = re.compile(r"\b(?:octopus\w*|octopi)\b")
    _SQUID_RX      = re.compile(r"\b(?:squids?|calamari)\b")

    def _has_seafood_word(t: str) -> bool:
        return any(rx.search(t) for rx in
                   (_FISH_SPECIES_RX, _SHRIMP_RX, _CRAB_RX, _LOBSTER_RX, _OCTOPUS_RX, _SQUID_RX))

    # ------------------------------------------------------------------
    # Which ingredient rows get the product's bone_in / fish_cut flags.
    # Both flags are read from the product text, so without this check every
    # ingredient of a product got them ("CLOVES, WHOLE" -> cloves "Whole / Round",
    # "FRIES, STEAK CUT" -> potatoes "Steaks / Cutlets"). A row keeps fish_cut
    # only if its ingredient names a fish or seafood, and bone_in only if it
    # names a meat or seafood. The brine/solution injected into a fish or meat
    # product keeps them too (bone_in is only set for meat/fish products, and
    # fish_cut only when the product names a fish/seafood). Stage 2 also clears
    # both on any row whose category is not one Stage 3 applies edible yields to.
    # ------------------------------------------------------------------
    _SEAFOOD_ING_RX = re.compile(r"\b(?:seafood|fillets?|filets?|ahi|lox|gravlax|sashimi|scampi|langostinos?|"
                                 r"crayfish|crawfish|surimi|roe|caviar)\b")
    _MEAT_ING_RX = re.compile(r"\b(?:veal|hams?|bacon|sausages?|poultry|hens?|capons?|duck|meats?|wings?|"
                              r"drumsticks?|drumm?ettes?|thighs?|legs?|breasts?|ribs?|chops?|shanks?|loins?|"
                              r"brisket|roast|steaks?|cutlets?|tenderloins?|carcass|bones?|skins?|giblets?|"
                              r"necks?|feet|gizzards?|livers?)\b")
    _CARRIER_ING_RX = re.compile(r"\b(?:solution|brine|marinade|injection)\b")

    def _keeps_fish_cut(ingredient: str, product: str = "") -> bool:
        t = _norm(ingredient)
        if _has_seafood_word(t) or _SEAFOOD_ING_RX.search(t):
            return True
        return bool(_CARRIER_ING_RX.search(t)) and _has_seafood_word(_species_text(_norm(product)))

    def _keeps_bone_in(ingredient: str) -> bool:
        t = _norm(ingredient)
        return bool(_LAND_ANIMAL_RX.search(t) or _MEAT_ING_RX.search(t) or _CARRIER_ING_RX.search(t)
                    or _has_seafood_word(t) or _SEAFOOD_ING_RX.search(t))

    # ------------------------------------------------------------------
    # Dairy handling. "goat cheese", "cow's milk", etc. contain a species
    # word but are dairy, so those phrases are blanked out before species
    # matching. Dairy words elsewhere in the text are NOT a veto by
    # themselves -- "Buttermilk Fried Chicken, Bone-In" or "Salmon Fillet
    # with Dill Butter" are still meat/fish. If dairy words are present and
    # the only species word left is goat/cow (e.g. "Cheese, Goat, Log"),
    # the item is treated as dairy.
    # ------------------------------------------------------------------
    _DAIRY_TERMS_RX = re.compile(r"\b(?:milk|cheese|chevre|yogurt|yoghurt|cream|butter|ghee|whey|buttermilk|kefir)\b")
    _DAIRY_PHRASE_RX = re.compile(
        r"\b(?:goats?|cows?|sheep|ewes?|buffalo)(?:\s+s)?\s+"
        r"(?:milk|cheese|chevre|yogurt|yoghurt|cream|butter|ghee|whey|kefir)\b"
    )

    def _species_text(t: str) -> str:
        """Normalized text with dairy species-phrases removed, for species matching only."""
        t = _DAIRY_PHRASE_RX.sub(" ", t)
        if _DAIRY_TERMS_RX.search(t):
            t = re.sub(r"\b(?:goats?|cows?)\b", " ", t)
        return t

    # "whole" as a cut word, excluding "whole" that modifies something other
    # than the animal: whole wheat/grain/kernel/oats, whole milk, whole
    # egg(s), whole bean(s), whole foods, and "whole muscle" -- a deli/
    # processing term for intact (usually boneless) meat, e.g. "Turkey
    # Breast, Whole Muscle".
    _WHOLE_CUT = r"\bwhole\b(?!\s*(?:wheat|grains?|kernels?|oats?|milk|muscle|eggs?|beans?|foods?)\b)"

    def detect_bone_in(text: str) -> bool:
        """
        Flags product/ingredient text implying a substantial inedible portion
        (bones, head, guts, shell, etc.) is still present in the purchased weight,
        so Stage 3's edible-yield shim (Cell 8) can shrink it toward the boneless/
        edible-only basis the factors.csv d_lives/t_lives figures were built on.

        Restricted to animal proteins (chicken, turkey, beef, pork, lamb, goat,
        fish/seafood) via _LAND_ANIMAL_RX / _has_seafood_word(). Dairy phrases
        such as "goat cheese" or "cow's milk" are removed before that check
        (see _species_text), so they never count as goat/beef -- but other dairy
        words in the text don't block a real meat item ("Buttermilk Fried
        Chicken, Bone-In" is still bone-in).

        Explicit "bone-in"/"with bones" always wins. Otherwise, text that says
        "boneless", "deboned" or "fillet" is not bone-in, even if it also says
        "whole" ("Whole Turkey Breast, Boneless"). After that, it also catches
        common seafood/poultry trade terms for whole or minimally-processed
        product: "whole", "dressed"/"pan-dressed", "pan-ready", "head-on",
        "headed and gutted"/"H&G", "gutted", and "drawn" (gutted-only, a
        seafood-specific term). "whole" excludes whole wheat/grain/kernel/oats/
        milk/eggs/beans/foods and "whole muscle" (see _WHOLE_CUT).

        For fish/seafood rows Cell 8 uses detect_fish_cut() instead; this flag
        only matters there as a fallback when no cut was recognized.
        """
        t = _norm(text)
        t_sp = _species_text(t)
        if not (_LAND_ANIMAL_RX.search(t_sp) or _has_seafood_word(t_sp)):
            return False
        if re.search(r"\bbone[-\s]?in\b|\bwith\s+bones?\b", t):
            return True
        if re.search(r"\bboneless\b|\bde[-\s]?boned\b|\bfillets?\b|\bfilets?\b", t):
            return False
        pattern = r"""
            """ + _WHOLE_CUT + r"""                              # whole <animal/fish>
            | \bdressed\b                                        # dressed / pan-dressed / oven-dressed (gutted, usually bone-in)
            | \bpan[-\s]?ready\b                                 # pan-ready
            | \bhead[-\s]?on\b                                   # head-on (shrimp/fish)
            | \bheaded\s+and\s+gutted\b | \bh\s?g\b              # headed and gutted / H&G
            | \bgutted\b                                         # gutted
            | \bdrawn\b(?!\s*butter)                             # drawn (seafood: gutted only; excludes "drawn butter")
        """
        return bool(re.search(pattern, t, re.VERBOSE))

    # ------------------------------------------------------------------
    # Cut classifiers. Each returns one of the eight FISH_CUT_YIELDS keys in
    # Cell 8 ("Whole / Round", "Drawn", "H&G", "Dressed", "Steaks / Cutlets",
    # "Bone-In Fillet", "Skin-On Fillet", "Skinless Fillet") or None. For
    # shellfish/cephalopods the key names are just positions on that
    # species' curve (least to most edible), not literal descriptions --
    # e.g. tail-on peeled shrimp uses the "Skin-On Fillet" point.
    # ------------------------------------------------------------------
    def _finfish_cut(t: str, allow_bare_round: bool) -> Optional[str]:
        has_boneless_skinless = bool(re.search(r"\bboneless\b|\bskinless\b", t))
        has_bone_in           = bool(re.search(r"\bbone[-\s]?in\b|\bwith\s+bones?\b", t))
        has_fillet            = bool(re.search(r"\bfillets?\b|\bfilets?\b", t))
        has_skin_on           = bool(re.search(r"\bskin[-\s]?on\b", t))
        has_steak_cutlet      = bool(re.search(r"\bsteaks?\b|\bcutlets?\b", t))
        has_dressed           = bool(re.search(r"\bdressed\b|\bpan[-\s]?ready\b", t))
        has_hg                = bool(re.search(r"\bheaded\s+and\s+gutted\b|\bh\s?g\b", t))
        has_drawn             = bool(re.search(r"\bdrawn\b(?!\s*butter)|\bgutted\b|\bhead[-\s]?on\b|\bheaded\b", t))
        has_whole_round       = bool(re.search(_WHOLE_CUT, t)) or (allow_bare_round and bool(re.search(r"\bround\b", t)))

        if has_boneless_skinless:
            return "Skinless Fillet"
        if has_bone_in and has_fillet:
            return "Bone-In Fillet"
        if has_bone_in:
            return "Steaks / Cutlets"
        if has_skin_on or has_fillet:
            return "Skin-On Fillet"
        if has_steak_cutlet:
            return "Steaks / Cutlets"
        if has_dressed:
            return "Dressed"
        if has_hg:
            return "H&G"
        if has_drawn:
            return "Drawn"
        if has_whole_round:
            return "Whole / Round"
        return None

    def _shrimp_cut(t: str) -> Optional[str]:
        # peeled (incl. P&D, PUD, PDTO) -> edible; tail-on keeps a little shell
        if re.search(r"\bpeeled\b|\bp\s?d\b|\bpud\b|\bpdto\b|\bmeat\b", t):
            if re.search(r"\btail[-\s]?on\b|\bpdto\b", t):
                return "Skin-On Fillet"
            return "Skinless Fillet"
        # head-on shell-on = the whole animal
        if re.search(_WHOLE_CUT + r"|\bhead[-\s]?on\b|\bhoso\b", t):
            return "Whole / Round"
        # headless but shell still on
        if re.search(r"\bshell[-\s]?on\b|\bin\s+(?:the\s+)?shells?\b|\bheadless\b|\bhlso\b|\b(?:ez|easy)\s?peel\b|\bunpeeled\b", t):
            return "Dressed"
        if re.search(r"\btail[-\s]?on\b", t):
            return "Skin-On Fillet"
        if re.search(r"\btail[-\s]?off\b|\bdeveined\b", t):
            return "Skinless Fillet"
        return None

    def _crab_cut(t: str) -> Optional[str]:
        # picked meat, imitation/surimi, crab cakes, and soft-shell crab (eaten whole) -> edible
        if re.search(r"\bmeat\b|\blump\b|\bbackfin\b|\bpicked\b|\bimitation\b|\bsurimi\b|\bcakes?\b|\bsoft[-\s]?shell\b", t):
            return "Skinless Fillet"
        if re.search(_WHOLE_CUT + r"|\blive\b", t):
            return "Whole / Round"
        # legs/clusters/claws still in shell
        if re.search(r"\bclusters?\b|\blegs?\b|\bclaws?\b|\bsections?\b|\bsplit\b|\bknuckles?\b|\bshell[-\s]?on\b|\bin\s+(?:the\s+)?shells?\b", t):
            return "H&G"
        return None

    def _lobster_cut(t: str) -> Optional[str]:
        if re.search(r"\bmeat\b|\bpicked\b", t):
            return "Skinless Fillet"
        if re.search(_WHOLE_CUT + r"|\blive\b", t):
            return "Whole / Round"
        # tails/claws still in shell
        if re.search(r"\btails?\b|\bclaws?\b|\bknuckles?\b|\bsplit\b|\bshell[-\s]?on\b|\bin\s+(?:the\s+)?shells?\b", t):
            return "Steaks / Cutlets"
        return None

    def _octopus_cut(t: str) -> Optional[str]:
        # cut pieces (tentacles/legs/arms, sliced) -> edible
        if re.search(r"\btentacles?\b|\blegs?\b|\barms?\b|\bslice[ds]?\b|\bdiced\b|\bcut\b", t):
            return "Skinless Fillet"
        # whole octopus with beak/ink sac/viscera removed
        if re.search(r"\bcleaned\b|\bgutted\b|\bdrawn\b", t):
            return "Drawn"
        if re.search(_WHOLE_CUT, t):
            return "Whole / Round"
        return None

    def _squid_cut(t: str) -> Optional[str]:
        # cleaned squid = tubes + tentacles, all edible
        if re.search(r"\bcleaned\b|\btubes?\b|\brings?\b|\btentacles?\b|\bsteaks?\b|\bstrips?\b|\bmantles?\b", t):
            return "Skinless Fillet"
        if re.search(_WHOLE_CUT, t):
            return "Whole / Round"
        return None

    def detect_fish_cut(text: str) -> Optional[str]:
        """
        Classifies fish/seafood product text into one of the eight cut stages
        FISH_CUT_YIELDS (Cell 8) is keyed on: "Whole / Round", "Drawn", "H&G",
        "Dressed", "Steaks / Cutlets", "Bone-In Fillet", "Skin-On Fillet",
        "Skinless Fillet". Returns None when nothing recognized -- Cell 8 then
        falls back to its flat bone-in/boneless yield and lists the row in
        unclassified_fish_cuts_{BASE}.csv.

        Cell 8 only uses this for rows whose *category* is fish/seafood, so a
        stray match on an unrelated item is harmless. That lets it run for
        text that names no known species at all (e.g. "Snapper, Whole" if
        snapper weren't listed, or "Fillet, 6 oz"), as long as the text names
        no land animal and has no dairy words. In that no-species case a bare
        "round" is ignored (too easily "round bun", "round patty").

        Finfish text uses _finfish_cut(). Shrimp, crab, lobster, octopus and
        squid text uses its own rules (_shrimp_cut() etc.), because their trade
        terms differ and the finfish rules would misplace them (e.g. head-on
        shrimp is the whole animal, not "Drawn"). If a finfish word is present
        it takes priority over a shellfish word. Shellfish/cephalopod results
        are returned as readable labels from SHELLFISH_CUT_LABELS (Cell 2),
        e.g. "Shrimp: peeled, tail-on"; Cell 8 maps them back to curve points.

        Finfish rules: "Bone-In Fillet" wins whenever both bone-in and fillet
        language are present, regardless of order or what's between them
        ("Bone-In Salmon Fillet"). Bare "bone-in" (no fillet language) is
        treated as "Steaks / Cutlets" -- a deliberate middle estimate for an
        unqualified bone-in fish cut. A bare, unqualified "fillet"/"filet" is
        treated as "Skin-On Fillet", not "Skinless Fillet" -- another
        deliberately cautious middle estimate, since a "fillet" purchase often
        still carries skin (or pin bones) even when not explicitly labeled.
        """
        t = _norm(text)
        t_sp = _species_text(t)
        has_fish = bool(_FISH_SPECIES_RX.search(t_sp))
        if not has_fish:
            for sp, rx, fn in (("shrimp", _SHRIMP_RX, _shrimp_cut), ("crab", _CRAB_RX, _crab_cut),
                               ("lobster", _LOBSTER_RX, _lobster_cut), ("octopus", _OCTOPUS_RX, _octopus_cut),
                               ("squid", _SQUID_RX, _squid_cut)):
                if rx.search(t_sp):
                    key = fn(t)
                    return SHELLFISH_CUT_LABELS.get(sp, {}).get(key, key) if key else None
            if _LAND_ANIMAL_RX.search(t_sp) or _DAIRY_TERMS_RX.search(t):
                return None
        return _finfish_cut(t, allow_bare_round=has_fish)

    def detect_fish_source(text: str) -> Optional[str]:
        t = _norm(text)
        if re.search(r"\bwild\s*[- ]?\s*caught\b|\bwild\b", t):
            return "wild-caught"
        if re.search(r"\bfarm(ed|[- ]raised)?\b|\baquaculture\b|\braised\b", t):
            return "farmed"
        return None
    
    def annotate_fish_items(items: List[Dict[str,Any]], product_text: str) -> List[Dict[str,Any]]:
        src = detect_fish_source(product_text)
        if not src:
            return items
        out = []
        for it in items:
            name = str(it.get("ingredient",""))
            nlc = name.lower()
            # Skip sauces & non-fish
            if "sauce" in nlc:
                out.append(it); continue
            # If already annotated, keep
            if "(farmed" in nlc or "(wild" in nlc:
                out.append(it); continue
            # Annotate fish/seafood tokens
            token = None
            for sp in FISH_SPECIES:
                if re.search(rf"\b{re.escape(sp)}\b", nlc):
                    token = sp; break
            if token:
                it = dict(it)
                it["ingredient"] = f"{name} ({src})"
            out.append(it)
        return out

    def canonical_single_ingredient(title: str) -> Optional[str]:
        t = title or ""
        # Vegan/plant-based products: don't use single-ingredient rules that could return animal products
        if is_vegan_labeled(t):
            return None
        excl = label_exclusions(t)
        
        # Respect do-not-decompose if name exactly equals an allowed category
        n = _norm(t)
        if n in {_norm(x) for x in DO_NOT_DECOMPOSE}:
            return t.strip()
        # Otherwise single-ingredient staples
        for rx, name in SINGLE_ITEM_DETECTORS:
            if rx.search(t):
                # only when the staple is the product's main word (otherwise GPT lists the ingredients)
                if not _single_shortcut_ok(t, name):
                    continue
                # a dairy-free product is not plain milk/cream/butter/yogurt
                if "dairy" in excl and remove_excluded_items([{"ingredient": name}], excl) == []:
                    continue
                # Never collapse to water if it's clearly just packaging context
                if name == 'water' and PACKAGING_WATER_CONTEXT_RE.search(t):
                    continue
                return name
        return None
        
    def normalize_percents(items: List[Dict[str,Any]], product_text: str) -> List[Dict[str,Any]]:
        # If explicit ratios are in the name (e.g., 75/25), honor them if two items
        m = RATIO_RE.search(product_text or "")
        if m and len(items) == 2:
            nums = [int(x) for x in m.groups() if x]
            if len(nums) >= 2:
                total = sum(nums[:2])
                if total > 0:
                    return [
                        {"ingredient": items[0]["ingredient"], "percent": 100.0 * nums[0] / total},
                        {"ingredient": items[1]["ingredient"], "percent": 100.0 * nums[1] / total},
                    ]
    
        # Otherwise, if percents missing or don’t sum ~100, scale majors sensibly
        vals = [it.get("percent") for it in items if it.get("percent") is not None]
        if not vals:
            return items
        total = sum(p for p in vals if p is not None)
        if total <= 0:
            return items
        scale = 100.0 / total
        out = []
        for it in items:
            pct = it.get("percent")
            if pct is None:
                out.append({"ingredient": it["ingredient"], "percent": None})
            else:
                out.append({"ingredient": it["ingredient"], "percent": pct * scale})
        return out
    
    def drop_minors(items: List[Dict[str,Any]]) -> List[Dict[str,Any]]:
        majors = []
        for it in items:
            pct = it.get("percent")
            if pct is None:
                majors.append(it)  # keep if unsure
            elif pct >= MIN_KEEP_PCT:
                majors.append(it)
        return majors or items  # never drop everything
    
    def looks_like_vegan_meat(text: str) -> bool:
        # meat-style names count as substitutes only for vegan/plant-based or vegetarian/meatless labels
        if "meat" not in label_exclusions(text):
            return False
        t = _norm(text)
        return bool(re.search(
            r"\b(chicken|beef|pork|turkey|sausage|nugget|tender|patty|burger|meatball|ground|crumbles|hot\s*dog|rib|steak|cutlet)\b",
            t))
    
    def vegan_meat_fallback_needed(items: List[Dict[str,Any]]) -> bool:
        if not items:
            return True
        # Too generic or single generic item like "patty"
        generic = {"patty","burger","protein","meat substitute","meatless patty","tender","nugget"}
        if len(items) == 1 and _norm(items[0]["ingredient"]) in generic:
            return True
        # All unknown?
        if all(_norm(it["ingredient"]) == "unknown" for it in items):
            return True
        return False




    # --------------------------------------------------------------------
# --- Vegan safeguard patch (Option A) ---
# Provides a minimal, safe definition of enhanced_vegan_filter_and_convert()
# and guards its invocation to prevent "error"/"unknown" failures for vegan items.
# --------------------------------------------------------------------

    def enhanced_vegan_filter_and_convert(items, product_text):
        """
        Simple vegan/meatless filter and converter.
    
        - Removes animal-derived ingredients (meat, fish, eggs, dairy, honey, gelatin).
        - Keeps plant-based or neutral ingredients.
        - Returns the cleaned list; if everything is removed, returns an empty list.
        """
        if not items:
            return []
    
        animal_terms = (
            "beef","pork","chicken","turkey","lamb","fish","salmon","tuna","shrimp",
            "egg","cheese","milk","butter","cream","yogurt","honey","gelatin"
        )
    
        safe_items = []
        for it in items:
            ing = str(it.get("ingredient","")).lower()
            # skip clear animal items
            if any(tok in ing for tok in animal_terms):
                continue
            safe_items.append(it)
    
        return safe_items
    
    
    # --- Wrap vegan-filter invocation safely ---
    _old_ai_extract_ingredients = None
    if 'ai_extract_ingredients' in globals():
        _old_ai_extract_ingredients = ai_extract_ingredients  # preserve old symbol if rerun

    #----------------------------
    # ----------------------------
    # ai_extract_ingredients (returns items, origin)
    # ----------------------------
    def ai_extract_ingredients(product_text: str, max_ings: int = 8) -> Tuple[List[Dict[str, Any]], str]:
        """
        Return (items, origin) where items = [{"ingredient": str, "percent": float|None}, ...]
        origin in {"rule_halfhalf","rule_single","rule_fallback_vegan_meat","gpt1","error","skip"}
        """
        if _is_blank_product(product_text):
            return [], "skip"
    
        product_text = (product_text or "").strip()
        vegan_flag = is_vegan_labeled(product_text)       # vegan / plant-based only
        excl = label_exclusions(product_text)             # what the product's label(s) rule out
    
        # EXACT RULE: Half & Half is 50/50 milk/cream (ignore butterfat % range)
        if "dairy" not in excl and HALF_AND_HALF_RE.search(product_text):
            return [{"ingredient": "milk", "percent": 50.0},
                    {"ingredient": "cream", "percent": 50.0}], "rule_halfhalf"
    
        # Do-not-decompose (exact category match) OR single staples
        single = canonical_single_ingredient(product_text)
        if single:
            return [{"ingredient": single, "percent": 100.0}], "rule_single"
    
        # --- Build the LLM prompt (your established guidance) ---
        base_rules = [
            "Return ingredients for the product below as a JSON object with key `items`.",
            "Each item must be {\"ingredient\": <string>, \"percent\": <number or null>}.",
            f"Limit to at most {max_ings} items.",
            "Report ONLY major ingredients by mass. OMIT micro-ingredients likely under "
            f"{MIN_KEEP_PCT}% (e.g., salt, acids, leaveners, stabilizers, spices, flavors), unless the product has only 1–2 ingredients.",
            "If the product name specifies blend ratios (e.g., 75/25), reflect those percents exactly.",
            "If unsure of exact amounts, provide realistic approximate percents for MAJOR ingredients; do NOT evenly split many items.",
            "Do NOT decompose protected items; keep them intact (e.g., 'almond milk', 'ice cream', 'peanut butter', 'tofu', 'tempeh', 'seitan').",
            "IF the product name denotes a single-ingredient staple (milk, cream, butter, yogurt, a single specific cheese, plain oil, sugar, flour, salt, rice, oats, pasta, water, vinegar), output EXACTLY one item at 100 and do NOT list additives.",
            "IF the product name includes 'half & half' or 'half and half', output exactly two items: 'milk' 50% and 'cream' 50%.",
            

            "When a product is a manufactured multi-ingredient food, identify its "
            "underlying constituent ingredients when they can be reasonably inferred "
            "from the product description. Do not invent a detailed formulation when "
            "the description provides insufficient information. "
            "For plant-based meat substitutes (such as vegan burgers, sausages, "
            "chorizo, ground meat, or chicken substitutes), use the single ingredient "
            "'plant-based meat' at 100 percent when the underlying formulation is "
            "unknown and no distinct components are clearly identified. "
            "If distinct components such as breading, a bun, or sauce are explicitly "
            "identified, estimate their shares separately where reasonable. "
            "Do not infer components that are not indicated by the description. "
            "For other manufactured products, including vegan cheese alternatives, "
            "continue to estimate major underlying ingredients when reasonably "
            "supported by the product description. "
            "Apply stated blend ratios when their meaning is clear; do not automatically "
            "treat a blend of finished products as its final ingredient breakdown.",
            # === added to reduce unknowns ===
            "If the product clearly is a non-food item (e.g. napkins, paper towels, cups, containers, foil, plastic wrap), return one item {\"ingredient\": \"unknown\", \"percent\": 100}.  Otherwise, never return 'unknown'—instead infer the most typical major ingredients for that food product.",
        ]
        vegan_rules = [
            "IMPORTANT: The product is labeled vegan/plant-based.",
            "Do NOT include any animal-derived ingredients (meat, fish/seafood, eggs, dairy, gelatin, honey).",
            "If a 'milk' is implied, interpret as a plant milk and name explicitly (e.g., 'almond milk').",
            "Avoid naming dairy cheeses directly; if needed, use 'vegan cheese alternative' terminology.",
            "IF the product name indicates vegan or imitation cheese and GPT would otherwise return 'unknown', "
            "infer a plausible base composition such as water (~45%), vegetable oil (~23%), potato starch (~22%), "
            "and legumes (~10%), or similar plant-based ingredients that form vegan cheese.",
        ] if vegan_flag else []
        if not vegan_flag and "dairy" in excl:
            vegan_rules += [
                "IMPORTANT: The product is labeled dairy-free/non-dairy.",
                "Do NOT include dairy ingredients (milk, cream, butter, cheese, yogurt, whey, casein, ghee).",
                "If a dairy-style ingredient is implied, name the plant-based alternative explicitly "
                "(e.g., 'coconut cream', 'oat milk', 'vegan cheese alternative').",
                "Dairy-free does NOT mean vegan: include meat, fish, eggs or honey if the product contains them.",
            ]
        if not vegan_flag and "meat" in excl:
            vegan_rules += [
                "IMPORTANT: The product is labeled vegetarian/meatless.",
                "Do NOT include meat, poultry, fish, seafood or gelatin; meat-style names (e.g., 'chicken nuggets') "
                "refer to plant-based substitutes.",
                "Dairy and eggs may be included if the product contains them.",
            ]
    
        protected_note = "Protected examples:\n" + ", ".join(sorted(PROTECTED_INGREDIENTS))
        system_msg = (
            "You extract food ingredients as structured JSON for downstream life-cycle analysis. "
            "Prioritize MAJOR ingredients and realistic percents; omit micro-ingredients under the threshold."
        )
        user_msg = "\n".join(base_rules + vegan_rules + [
            "", protected_note, "", PRODUCT_FLAG_RULES, "", "Product text:", product_text, "",
            "Respond ONLY with JSON like: {\"items\":[{\"ingredient\":\"...\",\"percent\":12.3}, ...], "
            "\"animal_derived\": false, \"neither_plant_nor_animal\": false, \"diluted_beverage\": false}",
        ])
    
        gpt_called = True
        try:
            # NOTE: uses the notebook's existing _call_gpt defined earlier
            raw = _call_gpt(
                messages=[{"role": "system", "content": system_msg},
                          {"role": "user", "content": user_msg}]
            )

        except Exception as e:
            print(f"[GPT ERROR] {type(e).__name__}: {e}")
            raw = ""
            gpt_called = False
    
        # Parse JSON
        items: List[Dict[str, Any]] = []
        if raw:
            try:
                obj = json.loads(raw)
                store_product_flags(product_text, obj)   # whole-product answers for the plant-based judgment
                cand = obj.get("items", [])
                if isinstance(cand, list):
                    for it in cand:
                        ing = str(it.get("ingredient", "")).strip()
                        if not ing:
                            continue
                        pct = it.get("percent", None)
                        try:
                            pct = float(pct) if pct is not None else None
                        except Exception:
                            pct = None
                        # Snap protected phrases
                        if is_protected(ing):
                            for prot in PROTECTED_INGREDIENTS:
                                if prot in ing.lower():
                                    ing = prot
                                    break
                        items.append({"ingredient": ing, "percent": pct})
            except Exception:
                items = []
    
        origin = "gpt1" if gpt_called else "error"

        def _is_all_unknown(lst):
            return (not lst) or all(_norm(it.get("ingredient","")) == "unknown" for it in lst)
        
        if _is_all_unknown(items):
            try:
                fb_user = (
                    "Return realistic MAJOR ingredients (avoid 'unknown'). "
                    f"Limit to 5 items, omit micro-ingredients under {MIN_KEEP_PCT}%. "
                    "If the product is vegan/plant-based/imitation, use typical substitutes. "
                    f"Product text: {product_text}\n"
                    'Respond ONLY as JSON like {"items":[{"ingredient":"...","percent":12.3}, ...]}'
                )
                raw2 = _call_gpt(messages=[
                    {"role": "system", "content": "You infer typical ingredient compositions for institutional foods."},
                    {"role": "user", "content": fb_user}
                ])
                try:
                    obj2 = json.loads(raw2)
                except Exception:
                    m = re.search(r"\{[\s\S]*\}", raw2 or "")
                    obj2 = json.loads(m.group(0)) if m else {}
                cand2 = obj2.get("items", [])
                new_items = []
                if isinstance(cand2, list):
                    for it in cand2:
                        ing = str(it.get("ingredient","")).strip()
                        if not ing:
                            continue
                        pct = it.get("percent", None)
                        try:
                            pct = float(pct) if pct is not None else None
                        except Exception:
                            pct = None
                        # snap protected phrases
                        if is_protected(ing):
                            for prot in PROTECTED_INGREDIENTS:
                                if prot in ing.lower():
                                    ing = prot
                                    break
                        new_items.append({"ingredient": ing, "percent": pct})
                if new_items and not _is_all_unknown(new_items):
                    items = new_items
                    origin = "gpt_fallback"
            except Exception:
                # keep prior items/origin if fallback fails
                pass


        # === added fallback to avoid "unknown" ===
        def _is_all_unknown(lst):
            return all(_norm(it["ingredient"]) == "unknown" for it in lst)

        if (not items) or _is_all_unknown(items):
            try:
                fallback_prompt = f"""
You are a food product analyst estimating typical compositions for institutional foods.

The following product name was too vague for prior analysis:
"{product_text}"

Task:
Infer the most typical MAJOR ingredients and their approximate percentages by mass.
Avoid "unknown" or "other". If the product name is generic (e.g., 'Chicken Patty', 'Fish Stick'),
use realistic, generic ingredients (e.g., 'chicken', 'breading', 'oil').
Limit to 5 ingredients.

Respond ONLY as JSON:
{{"items":[{{"ingredient":"...","percent":...}},...]}}
"""
                raw2 = _call_gpt(
                    messages=[
                        {"role": "system", "content": "You estimate typical ingredient compositions for common institutional food products."},
                        {"role": "user", "content": fallback_prompt}
                    ]
                )
                obj2 = json.loads(raw2)
                cand2 = obj2.get("items", [])
                if isinstance(cand2, list) and cand2:
                    items = []
                    for it in cand2:
                        ing = str(it.get("ingredient", "")).strip()
                        pct = it.get("percent", None)
                        try:
                            pct = float(pct) if pct is not None else None
                        except Exception:
                            pct = None
                        if ing:
                            items.append({"ingredient": ing, "percent": pct})
                    origin = "gpt_fallback"
            except Exception:
                pass
        # === end added fallback ===
    
        # Label post-filter: swap only what the label rules out for a plant-based version, keeping its share
        # (dairy-free keeps real meat, fish and eggs; meatless 'chicken' becomes 'plant-based chicken')
        if excl and items:
            try:
                items = replace_excluded_items(items, excl)
            except Exception as e:
                print(f"[warn] label filter failed for: {product_text[:80]} ({e})")
            if not items and looks_like_vegan_meat(product_text):
                return VEGAN_MEAT_DEFAULT.copy(), "rule_fallback_vegan_meat"
    
        # Vegan meat fallback if GPT weak/unknown for vegan-meatlike item
        if looks_like_vegan_meat(product_text) and vegan_meat_fallback_needed(items):
            return VEGAN_MEAT_DEFAULT.copy(), "rule_fallback_vegan_meat"
    
        # Normalize, annotate fish, and drop minors
        items = normalize_percents(items, product_text)
        items = annotate_fish_items(items, product_text)
        items = drop_minors(items)

        if not items and "dairy" in excl and "cheese" in product_text.lower():
            return [
                {"ingredient": "water", "percent": 45.0},
                {"ingredient": "vegetable oil", "percent": 23.0},
                {"ingredient": "potato starch", "percent": 22.0},
                {"ingredient": "legumes", "percent": 10.0},
            ], "rule_fallback_vegan_cheese"

        if not items:
            return [{"ingredient": "unknown", "percent": 100.0}], origin
        return items, origin
    
    # ----------------------------
    # Weight extraction helper
    # ----------------------------
    def _extract_row_weight(src_row: pd.Series) -> Optional[float]:
        """Product weight (in the run's weight unit) if present on the standardized row."""
        suf = weight_suffix()
        for cand in (f"ingredient_weight_{suf}", f"product_weight_{suf}", f"weight_{suf}"):
            if cand in src_row.index and pd.notna(src_row[cand]):
                try:
                    return float(src_row[cand])
                except Exception:
                    pass
        return None

    # ----------------------------
    # Stage 1 runner (with log_time)
    # ----------------------------
    def _find_product_col(df: pd.DataFrame) -> str:
        for c in df.columns:
            if _norm(c) in ("product","product name","item","description","name"):
                return c
        return df.columns[0]
    
    def run_stage_1(input_filename: str, file_type: str, base: str) -> pd.DataFrame:
        with log_time("S1.0 load + standardize"):
            try:
                std = load_and_standardize(input_filename, file_type)
                df = std
                prod_col = "product"
                series = df[prod_col].astype(str)
            except Exception:
                df = read_input_csv(input_filename)
                prod_col = _find_product_col(df)
                series = df[prod_col].astype(str)

        # Filter out unwanted type markers before processing
        unwanted = {"type01", "type02", "type03"}
        
        # Create masks for first two columns (typically product and description)
        if len(df.columns) > 0:
            mask1 = ~(df[df.columns[0]].notna() & 
                     df[df.columns[0]].astype(str).str.strip().str.lower().isin(unwanted))
        else:
            mask1 = pd.Series([True] * len(df))
        
        if len(df.columns) > 1:
            mask2 = ~(df[df.columns[1]].notna() & 
                     df[df.columns[1]].astype(str).str.strip().str.lower().isin(unwanted))
        else:
            mask2 = pd.Series([True] * len(df))
        
        # Apply both masks, and leave out "Total" / "Grand Total" lines (Cell 2)
        mask3 = ~df[prod_col].astype(str).map(is_total_line)
        df = df[mask1 & mask2 & mask3].reset_index(drop=True)
        series = df[prod_col].astype(str)

        # === added: skip NaN/blank product rows BEFORE submitting to GPT ===
        keep_mask = df[prod_col].notna() & df[prod_col].astype(str).str.strip().ne("")
        df = df.loc[keep_mask]
        series = df[prod_col].astype(str)
        # === end added ===
        # --- Skip empty, blank, or obvious non-values before submitting to GPT ---
        mask_valid = (
            series.notna()
            & series.str.strip().ne("")
            & ~series.str.lower().isin(["nan", "none"])
        )
        series = series[mask_valid]
        
        # Settings
        max_workers = int(os.getenv("FI_MAX_WORKERS", "5"))
        max_ings = int(os.getenv("FI_MAX_INGS", "8"))
        ROW_OFFSET = int(os.getenv("FI_ROW_OFFSET", "2"))  # header row=1, first product=2
    
        def _one(idx_val: int, text: str) -> Tuple[List[Dict[str, Any]], str]:
            try:
                return ai_extract_ingredients(text, max_ings=max_ings)
            except Exception:
                return [{"ingredient": "unknown", "percent": 100.0}], "error"
    
        rows: List[Dict[str, Any]] = []
        suf = weight_suffix()   # 'kg' or 'lbs' -- the worksheet's unit, decided in load_and_standardize

        with log_time(f"S1.1 submit {len(series)} items to pool (max_workers={max_workers})"):
            with cf.ThreadPoolExecutor(max_workers=max_workers) as ex:
                futures = {
                    ex.submit(_one, i, t): (i, t)
                    for i, t in series.items()
                    if (pd.notna(t) and str(t).strip())
                }
    
                with log_time("S1.2 collect results from pool"):
                    for fut in cf.as_completed(futures):
                        i, t = futures[fut]
                        items, origin = fut.result()
    
                        src_row = df.loc[i] if i in df.index else pd.Series(dtype=object)
                        p_wt = _extract_row_weight(src_row)
                        bone_in = detect_bone_in(str(t))
                        fish_cut = detect_fish_cut(str(t))
                        qty_val = None
                        if "qty" in src_row.index and pd.notna(src_row["qty"]):
                            qty_val = src_row["qty"]
    
                        # Prefer standardized product name if available
                        prod_name = (str(src_row["product"])
                                     if ("product" in src_row.index and pd.notna(src_row["product"]))
                                     else str(t))
    
                        base_fields = {
                            "__row": i,
                            "row_sheet": sheet_row(src_row_no(src_row, i)),   # the worksheet's row number (Cell 2)
                            "product": prod_name,
                            "extract_origin": origin,
#                            "bone_in": bool(bone_in),
                            "bone_in": "True" if bone_in else "",
                            "fish_cut": fish_cut or "",
                        }
                        if qty_val is not None:
                            base_fields["qty"] = qty_val
                        if p_wt is not None:
                            base_fields[f"product_weight_{suf}"] = p_wt
                        # plant-based judgment and the worksheet's certification columns (Cell 4)
                        src = src_row.get("__src_row", i)
                        base_fields.update(product_attributes(i if pd.isna(src) else src, str(t), items))
    
                        for it in items:
                            rec = dict(base_fields)
                            rec["ingredient"] = it["ingredient"]
                            # product-level flags apply only to the fish/meat ingredient rows
                            if rec["fish_cut"] and not _keeps_fish_cut(str(it["ingredient"]), str(t)):
                                rec["fish_cut"] = ""
                            if rec["bone_in"] and not _keeps_bone_in(str(it["ingredient"])):
                                rec["bone_in"] = ""
#                            rec["percent"] = it.get("percent", None)
                            val = it.get("percent", None)
                            rec["percent"] = round(val, 2) if isinstance(val, (int, float)) else val
                            rows.append(rec)


        with log_time("S1.3 build DataFrame"):
            df_ing = pd.DataFrame(rows)
    
            # Ensure row_sheet exists (fallback to __row + offset)
            if "row_sheet" not in df_ing.columns:
                if "__row" in df_ing.columns:
                    df_ing["row_sheet"] = df_ing["__row"].astype(int) + ROW_OFFSET
                else:
                    df_ing = df_ing.reset_index(drop=False).rename(columns={"index": "__row"})
                    df_ing["row_sheet"] = df_ing["__row"].astype(int) + ROW_OFFSET
    
            # Calculate ingredient weight (worksheet's unit) and format decimal columns
            pw_col, iw_col = f"product_weight_{suf}", f"ingredient_weight_{suf}"
            if 'percent' in df_ing.columns and pw_col in df_ing.columns:
                df_ing[iw_col] = df_ing['percent'] * df_ing[pw_col] / 100

                # Round to 2 decimal places
                df_ing['percent'] = df_ing['percent'].round(2)
                df_ing[iw_col] = df_ing[iw_col].round(2)
                df_ing[pw_col] = df_ing[pw_col].round(2)

            # Column order (keep one 'product' column; hide __row in file) - add ingredient weight
# optional add qty (remove next line)           preferred = ["row_sheet","product","qty",pw_col,"ingredient","percent",iw_col,"bone_in","extract_origin"]
            preferred = ["row_sheet","product",pw_col,"ingredient","percent",iw_col,"extract_origin","bone_in","fish_cut"]
            ordered = [c for c in preferred if c in df_ing.columns]
            ordered += [c for c in df_ing.columns if c not in set(ordered + ["__row"])]
            df_ing = df_ing[ordered].sort_values("row_sheet", kind="stable")
    
        with log_time("S1.4 write ingredients CSV"):
            out_path = result_out(f"ingredients_{base}.csv")
            df_ing.drop(columns=["__row"], errors="ignore").to_csv(out_path, index=False)
            print(f"[Stage 1] Wrote {len(df_ing)} rows to {out_path} (submitted {len(series)} products)")
    
        return df_ing


In [ ]:
# === Cell 7 — Stage 2: Categorization (Modified for FOODCAT_UPDATE_MODE) ===
# Purpose: Map each ingredient to a standardized food category using a mix of deterministic and AI-assisted logic.
# Inputs:
#   - ingredients_{BASE}.csv (from Stage 1), factors.csv (to derive ALLOWED_CATEGORIES if not provided)
#   - foodcategories.json (category dictionary)
# Dependencies:
#   - AI fallback function (optional)
#   - normalization utilities (_norm)
#   - FOODCAT_UPDATE_MODE (from Cell 1): "append", "disabled", or "separate"
# Outputs:
#   - categories_{BASE}.csv — categorized ingredient data
#   - updated foodcategories.json OR newcategories.json (based on FOODCAT_UPDATE_MODE)
# Behavior:
#   - Performs exact and fuzzy matching.
#   - Calls GPT for unmatched ingredients if available.
#   - Logs all newly learned ingredient–category pairs for review.
#   - Respects FOODCAT_UPDATE_MODE for persistence strategy
# Notes:
#   - Overwrites existing categories_{BASE}.csv if START_MODE='full' or 'after_ingredients'.
#   - Skips if START_MODE='after_categories'.


from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
import json, re, os
from difflib import SequenceMatcher

# --- Optional timing (no-op if you didn't add it in Utilities) ---
try:
    log_time
except NameError:
    from contextlib import contextmanager
    @contextmanager
    def log_time(_): 
        yield

# ----------------------------
# Config & helpers
# ----------------------------
VEGAN_TERMS = (
    "vegan","plant-based","plant based","meatless","vegetarian",
    "dairy-free","dairy free","non-dairy","nondairy"
)

DAIRY_CATEGORIES = {
    "dairy milk","skim milk","whole milk","concentrated milk","milk powder","whey powder",
    "butter","ghee","cream","sour cream","ice cream","cheese","dairy cheese",
    "yogurt","low fat yogurt","buttermilk","lactose powder"
}

PLANT_MILK_WORDS = {
    "almond","soy","oat","rice","pea","coconut","cashew","hemp","hazelnut","macadamia"
}

FUZZY_CUTOFF = 0.92  # high cutoff to avoid wrong hits

def _norm(s: str) -> str:
    """Use your norm_text if defined, else lowercase/strip/collapse spaces."""
    try:
        return norm_text(s)
    except Exception:
        t = str(s).strip().lower()
        return re.sub(r"\s+", " ", t)

def _load_allowed_categories(factors_csv: Optional[str] = None) -> Tuple[List[str], Dict[str,str]]:
    """Return (allowed_list, norm->display map)."""
    allowed = None
    if "ALLOWED_CATEGORIES" in globals():
        try:
            allowed = list(ALLOWED_CATEGORIES)
        except Exception:
            allowed = None
    if allowed is None:
        cand = factors_csv or globals().get("factors_filename") or Path(globals().get("base_dir", ".")) / "factors.csv"
        fac = pd.read_csv(cand)
        if "category" not in fac.columns:
            fac = fac.rename(columns={fac.columns[0]: "category"})
        allowed = fac["category"].dropna().astype(str).tolist()
    key_map = {_norm(c): c for c in allowed}
    return allowed, key_map

def _vegan_hint(product_key: str) -> bool:
    return any(t in product_key for t in VEGAN_TERMS)

def _is_dairy(cat_display: str) -> bool:
    return _norm(cat_display) in {_norm(x) for x in DAIRY_CATEGORIES}

def _milk_heuristic(ingredient_key: str, allowed_key2disp: Dict[str,str]) -> Optional[str]:
    """If ingredient looks like '<plant> milk' and plant is known, map to that milk category if allowed."""
    m = re.search(r"\b([a-z][a-z ]+)\s+milk\b", ingredient_key)
    if not m:
        return None
    plant = m.group(1).strip().split()[-1]
    if plant in PLANT_MILK_WORDS:
        key = _norm(f"{plant} milk")
        return allowed_key2disp.get(key)
    return None

# ---- Plant-based foods: fixed rules + animal-category guard ----
# Text is normalized first (lower case, punctuation removed, hyphens -> spaces), so "non-dairy" is "non dairy".
_PLANT_SRC = r"(?:coconut|almond|oat|oats|soy|soya|cashew|rice|pea|hemp|macadamia|hazelnut)"
_VEGAN_RX = re.compile(r"\b(?:vegan|plant based|plant derived|non animal|animal free)\b")
_DAIRY_FREE_RX = re.compile(r"\b(?:dairy free|non dairy|nondairy|milk free)\b")
_VEGETARIAN_RX = re.compile(r"\b(?:vegetarian|meatless|meat free)\b")
_DAIRY_WORDS = r"(?:milk|cream|creamer|yogurt|yoghurt|butter|cheese|ice cream|frozen dessert)"
_PLANT_DAIRY_RX = re.compile(rf"\b{_PLANT_SRC}\s+(?:\w+\s+)?{_DAIRY_WORDS}\b|\bmargarine\b")
_NONDAIRY_CHEESE_WORDS = r"(?:yogurt|yoghurt|cheese|cheddar|mozzarella|parmesan|feta)"

def _plant_rule(ik: str) -> Optional[str]:
    """Fixed category for common plant-based foods with dairy- or meat-style names (ingredient text only)."""
    t = ik
    # stand-ins Stage 1 uses when a label rules out honey, egg or gelatin
    if re.search(r"\bplant based (?:sweeteners?|honey)\b", t):
        return "sugars and sweeteners"
    if re.search(r"\bplant based egg (?:substitutes?|replacers?)\b", t):
        return "legumes"
    if re.search(r"\bplant based gelling agents?\b", t):
        return "additive"
    labelled = bool(_VEGAN_RX.search(t) or _DAIRY_FREE_RX.search(t))
    if (labelled and re.search(rf"\b{_NONDAIRY_CHEESE_WORDS}\b", t)) \
            or re.search(rf"\b{_PLANT_SRC}\s+(?:\w+\s+)?{_NONDAIRY_CHEESE_WORDS}\b", t) \
            or (re.search(r"\bcultured\b", t) and re.search(rf"\b{_PLANT_SRC}\s+(?:\w+\s+)?{_DAIRY_WORDS}\b", t)):
        return "non-dairy cheese or yogurt"
    if re.search(r"\bmargarine\b", t):
        return "vegetable oils"
    if re.search(r"\bpeanut butter\b", t):
        return "peanuts/groundnuts"
    if re.search(r"\b(?:cocoa|cacao) butter\b", t):
        return "cocoa"
    if re.search(r"\bapple butter\b", t):
        return "fruits"
    if re.search(r"\b(?:almond|cashew|hazelnut|pistachio|macadamia|walnut|pecan|sunflower|sunflower seed|sesame|"
                 r"pumpkin seed|seed|shea) butter\b", t):
        return "tree nuts and seeds"
    if labelled and re.search(r"\bbutter\b", t) and not re.search(r"\bbuttermilk\b", t):
        return "vegetable oils"
    if re.search(r"\bcoconut (?:milk|cream|creamer)\b|\bcream of coconut\b", t):
        return "tree nuts and seeds"
    m = re.search(r"\b(almond|oat|soy|soya|rice)(?: non dairy)? (?:milk|drink|beverage)\b", t)
    if m:
        return {"almond": "almond milk", "oat": "oat milk", "soy": "soy milk", "soya": "soy milk",
                "rice": "rice milk"}[m.group(1)]
    if re.search(r"\b(?:vegan|plant based|meatless|meat free|impossible|beyond)\b", t) \
            and re.search(r"\b(?:burgers?|patty|patties|chicken|beef|pork|sausages?|meat|nuggets?|crumbles|"
                          r"meatballs?|bacon|hot dogs?)\b", t) \
            and not re.search(r"\bbeans?\b", t):
        return "plant-based meat"
    return None

# ---- Coconut milk / cream: only the coconut content counts as tree nuts and seeds; the rest is water ----
# Shares are dried-coconut equivalents (tree nuts and seeds factors are per kg of dry shelled nuts), judged from
# fat content (dried coconut ~65% fat). Change these numbers to adjust.
COCONUT_SHARE_DRINK = 0.04    # coconut milk drinks / beverages (cartons, "milk sub"): ~2-2.5% fat
COCONUT_SHARE_LIGHT = 0.12    # light coconut milk, coconut creamer: ~7-10% fat
COCONUT_SHARE_MILK  = 0.25    # canned / cooking coconut milk (default): ~17-22% fat
COCONUT_SHARE_CREAM = 0.40    # coconut cream, cream of coconut: ~25-30% fat
_COCONUT_RX = re.compile(r"\bcoconut (?:milk|cream|creamer)\b|\bcream of coconut\b")
_COCONUT_SKIP_RX = re.compile(r"\b(?:powder|powdered|cultured|yogurt|yoghurt|oil|water|flour|flakes?|shreds?)\b")
_DRINK_RX = re.compile(r"\b(?:beverage|drink|milk sub|aseptic|carton|half gallon|gallon|quart|milk alternative|"
                       r"barista|unsweetened original|original unsweetened)\b")

def _coconut_share(ingredient_key: str, product_key: str) -> Optional[float]:
    """Share of a coconut milk/cream ingredient that is coconut (rest = water), or None if not split."""
    t = ingredient_key
    if not _COCONUT_RX.search(t) or _COCONUT_SKIP_RX.search(t):
        return None
    if re.search(r"\bcoconut cream\b|\bcream of coconut\b", t):
        return COCONUT_SHARE_CREAM
    if re.search(r"\b(?:light|lite|reduced fat)\b|\bcreamer\b", t) or re.search(r"\b(?:light|lite)\b", product_key):
        return COCONUT_SHARE_LIGHT
    if _DRINK_RX.search(t) or _DRINK_RX.search(product_key):
        return COCONUT_SHARE_DRINK
    return COCONUT_SHARE_MILK

# Products whose main name (before the first comma) is a seasoning: never meat, poultry, fish or seafood
_SEASONING_HEAD_RX = re.compile(r"^(?:seasoning|seasonings|spice|spices|rub|herb|herbs|spice blend|seasoning blend)$")

def _guard_groups(factors_csv, allowed_key2disp: Dict[str, str]) -> Dict[str, set]:
    """Normalized category keys for dairy, eggs, and meat/fish/seafood (categories with lives figures, + bivalves)."""
    try:
        fac = pd.read_csv(factors_csv or globals().get("factors_filename") or "factors.csv")
        lives = pd.to_numeric(fac.get("d_lives"), errors="coerce").fillna(0) + \
                pd.to_numeric(fac.get("t_lives"), errors="coerce").fillna(0)
        animal = {_norm(c) for c, l in zip(fac["category"], lives) if l > 0} | {_norm("bivalves")}
    except Exception:
        animal = set()
    dairy = {_norm(x) for x in DAIRY_CATEGORIES}
    eggs = {_norm("eggs")}
    animal = (animal | dairy | eggs) & set(allowed_key2disp)
    return {"all": animal, "dairy": dairy & animal, "meatfish": animal - dairy - eggs}

def _excluded(text: str, groups: Dict[str, set], ingredient_level: bool) -> set:
    """Animal categories ruled out by the wording of an ingredient (or, with explicit labels only, a product)."""
    ex = set()
    if _VEGAN_RX.search(text) or (ingredient_level and _PLANT_DAIRY_RX.search(text)):
        ex |= groups["all"]
    if _DAIRY_FREE_RX.search(text):
        ex |= groups["dairy"]
    if _VEGETARIAN_RX.search(text):
        ex |= groups["meatfish"]
    return ex

def _guard_message(ex: set, groups: Dict[str, set]) -> str:
    if not ex:
        return ""
    if groups["all"] and groups["all"] <= ex:
        return ("Do NOT choose any animal-derived category (meat, poultry, fish, seafood, eggs, or dairy such as milk, "
                "cream, butter, cheese, yogurt); the ingredient or product is plant-based.")
    parts = []
    if groups["dairy"] and groups["dairy"] <= ex:
        parts.append("Do NOT choose any dairy category (milk, cream, butter, cheese, yogurt, whey); the product is "
                     "labelled dairy-free/non-dairy.")
    if groups["meatfish"] and groups["meatfish"] <= ex:
        parts.append("Do NOT choose any meat, poultry, fish or seafood category; the product is vegetarian/meatless "
                     "or a seasoning/spice blend.")
    return " ".join(parts)

def _fuzzy_best(label_norm: str, allowed_norm: List[str]) -> Tuple[Optional[str], float]:
    best = 0.0
    hit = None
    for cand in allowed_norm:
        r = SequenceMatcher(None, label_norm, cand).ratio()
        if r > best:
            best = r; hit = cand
    return hit, best

# ---- Minimal JSON dict cache (ingredient -> category), append-only ----
def _load_foodcats_map(path: Path) -> Dict[str, str]:
    """Load simple dict {ingredient -> category}. If missing/invalid, return {}. Keys normalized."""
    if not path.exists():
        return {}
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            return {}
        out = {}
        for k, v in data.items():
            nk = _norm(k)
            if nk and str(v).strip():
                out.setdefault(nk, str(v).strip())  # keep first; do not overwrite
        return out
    except Exception:
        return {}

def _append_foodcats_map(path: Path, new_rows: pd.DataFrame, mode: str) -> int:
    """
    Conditionally append new {ingredient_key -> category} based on mode.
    
    Args:
        path: Path to foodcategories.json
        new_rows: DataFrame with 'ingredient_key' and 'category' columns
        mode: One of "append", "disabled", or "separate"
    
    Returns:
        Number of keys that would be/were appended
    
    Behavior:
        - "append": Write to foodcategories.json (persistent)
        - "disabled": Return 0, no action taken
        - "separate": Write to newcategories.json (for review before merging)
    """
    if new_rows is None or new_rows.empty:
        return 0
    
    # Count how many would be added
    cur = _load_foodcats_map(path)
    before = len(cur)
    candidates = {}
    
    # only categories that exist in factors.csv may be saved
    allowed_keys = {_norm(c) for c in (globals().get("ALLOWED_CATEGORIES") or [])}

    for _, r in new_rows.iterrows():
        ik = _norm(r.get("ingredient_key", ""))
        cat = str(r.get("category", "")).strip()
        if not ik or not cat or cat.lower() == "unknown":
            continue
        if allowed_keys and _norm(cat) not in allowed_keys:
            continue
        if ik not in cur:
            candidates[ik] = cat
    
    count = len(candidates)
    
    if mode == "disabled":
        return 0
    elif mode == "separate":
        # Write to separate file (newcategories.json)
        if count > 0:
            separate_path = Path(path).parent / "newcategories.json"   # next to foodcategories.json
            existing_new = _load_foodcats_map(separate_path)
            existing_new.update(candidates)
            separate_path.write_text(json.dumps(existing_new, ensure_ascii=False, indent=2), encoding="utf-8")
            # copy into the output folder too, so the web service returns it with the results
            result_out("newcategories.json").write_text(json.dumps(existing_new, ensure_ascii=False, indent=2), encoding="utf-8")
        return count
    elif mode == "append":
        # Write to main file (original behavior)
        if count > 0:
            cur.update(candidates)
            path.write_text(json.dumps(cur, ensure_ascii=False, indent=2), encoding="utf-8")
            # copy into the output folder too, so the web service returns the updated file
            result_out("foodcategories.json").write_text(json.dumps(cur, ensure_ascii=False, indent=2), encoding="utf-8")
        return count
    else:
        print(f"[WARNING] Unknown FOODCAT_UPDATE_MODE '{mode}', defaulting to 'append'")
        if count > 0:
            cur.update(candidates)
            path.write_text(json.dumps(cur, ensure_ascii=False, indent=2), encoding="utf-8")
        return count


# ---- Per-run CSV cache for (ingredient_key, product_key) ----
def _load_run_cache(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        need = ["ingredient_key","product_key","category","match_method"]
        for c in need:
            if c not in df.columns:
                df[c] = ""
        return df[need].drop_duplicates()
    return pd.DataFrame(columns=["ingredient_key","product_key","category","match_method"])

def _update_run_cache(path: Path, rows: pd.DataFrame) -> None:
    if rows is None or rows.empty:
        return
    cur = _load_run_cache(path)
    merged = pd.concat([cur, rows[["ingredient_key","product_key","category","match_method"]]], ignore_index=True)
    merged = merged.drop_duplicates(subset=["ingredient_key","product_key"], keep="last")
    merged.to_csv(path, index=False)

# ---- Strict GPT categorizer (must return exactly one allowed label) ----
def _call_gpt_categorize(ingredient: str, product: str, allowed: List[str], excluded: set = frozenset(),
                         guard_msg: str = "") -> Optional[str]:
    """Requires `_call_gpt` (defined in Utilities/Cell 5). Accepts only an allowed label that is not excluded."""
    if "_call_gpt" not in globals():
        return None
    used = [c for c in allowed if _norm(c) not in excluded] or allowed
    if not used:
        return None
    allowed_map = {c.lower(): c for c in used}

    system_msg = (
        "You classify one ingredient into exactly one category from a provided list. "
        "Reply with ONLY the chosen category string—no extra words."
    )
    user_msg = (
        "Choose exactly ONE category from the list that best matches the ingredient.\n"
        "Rules:\n"
        "  • Reply with ONLY the chosen category string (character-for-character).\n"
        "  • If none is perfect, choose the closest reasonable option from the list.\n"
        f"  • {guard_msg}\n\n"
        f"Ingredient: {ingredient}\n"
        f"Product context: {product}\n\n"
        "Allowed categories:\n" + "\n".join(f"- {c}" for c in used)
    )
    try:
        out = _call_gpt(
            messages=[{"role":"system","content":system_msg},
                      {"role":"user","content":user_msg}],
            temperature=0
        )
    except Exception:
        return None

    if not out:
        return None
    reply = out.strip().strip('"').strip("'")
    return allowed_map.get(reply.lower())  # None if not exact allowed label

# ----------------------------
# Main: Stage 2
# ----------------------------
def run_stage_2_base(base: str, factors_csv: Optional[str] = None):
    # Get the update mode from global config (defined in Cell 1)
    update_mode = globals().get("FOODCAT_UPDATE_MODE", "append")
    
    with log_time("S2.0 load ingredients"):
        ing_path = result_in(f"ingredients_{base}.csv")
        df = pd.read_csv(ing_path)

        # Ensure 'product' and 'row_sheet'
        if "product" not in df.columns and "product_text" in df.columns:
            df = df.rename(columns={"product_text":"product"})
        if "product" not in df.columns:
            # last resort fallback to first column
            df["product"] = df.iloc[:,0].astype(str)
        df["product"] = df["product"].astype(str)

        if "row_sheet" not in df.columns:
            df = df.reset_index(drop=False).rename(columns={"index":"__row"})
            row_offset = int(os.getenv("FI_ROW_OFFSET","2"))
            df["row_sheet"] = df["__row"].astype(int) + row_offset

        # Normalize once
        df["ingredient"] = df["ingredient"].astype(str)
        df["ingredient_key"] = df["ingredient"].map(_norm)
        df["product_key"]    = df["product"].map(_norm)

        print(f"[S2] rows={len(df):,}  unique (ingredient,product)={df[['ingredient_key','product_key']].drop_duplicates().shape[0]:,}")

    with log_time("S2.1 allowed & maps"):
        allowed, allowed_key2disp = _load_allowed_categories(factors_csv)
        allowed_norm = list(allowed_key2disp.keys())
        # Optional curated exact map
        global_map = {}
        if "CATEGORY_MAP" in globals():
            try:
                # skip entries whose category is not in factors.csv; those ingredients
                # go on to the later layers (fuzzy match, GPT) instead
                global_map = {_norm(k): v for k, v in dict(CATEGORY_MAP).items()
                              if _norm(v) in allowed_key2disp}
            except Exception:
                pass

    with log_time("S2.2 load caches"):
        json_cache_path = Path(globals().get("categories_filename")
                               or Path(globals().get("base_dir", ".")) / "foodcategories.json")
        json_map = _load_foodcats_map(json_cache_path)  # {ingredient_key -> category}
        json_map = {k: v for k, v in json_map.items() if _norm(v) in allowed_key2disp}  # factors.csv categories only
        csv_cache_path  = result_out(f"category_cache_{base}.csv")
        run_cache = _load_run_cache(csv_cache_path)
        appended_json_total = 0

    # Work copy
    df_cat = df.copy()
    df_cat["category"] = pd.NA
    df_cat["match_method"] = pd.NA

    # Coconut milk / cream: split into its coconut content (tree nuts and seeds) and water, by the shares
    # set at the top of this cell (drinks are far more dilute than canned coconut milk or coconut cream).
    with log_time("S2.2a coconut milk split"):
        _new_rows = []
        _w_cols = [c for c in df_cat.columns if c.startswith("ingredient_weight_")]
        for _, r in df_cat.iterrows():
            share = _coconut_share(r["ingredient_key"], r["product_key"])
            if share is None or "tree nuts and seeds" not in allowed_key2disp.values() or "water" not in allowed_key2disp.values():
                _new_rows.append(r)
                continue
            for part, frac, cat in (("coconut", share, "tree nuts and seeds"), ("water", 1 - share, "water")):
                s = r.copy()
                s["ingredient"] = f"{part} (in {r['ingredient']})"
                s["ingredient_key"] = _norm(s["ingredient"])
                for c in ["percent"] + _w_cols:
                    if c in s.index and pd.notna(s[c]):
                        s[c] = round(float(s[c]) * frac, 4)
                s["category"] = cat
                s["match_method"] = "rule_coconut"
                _new_rows.append(s)
        if len(_new_rows) != len(df_cat):
            df_cat = pd.DataFrame(_new_rows).reset_index(drop=True)

    # Plant-based guard: animal categories ruled out for each (ingredient, product) pair.
    # Ingredient wording counts fully; a product counts only through explicit labels (vegan, plant-based,
    # dairy-free/non-dairy, vegetarian/meatless). Results that depend on the product label alone are not
    # saved to foodcategories.json / newcategories.json, since those files are keyed by ingredient only.
    groups = _guard_groups(factors_csv, allowed_key2disp)
    _head = {pk: _norm(str(p).split(",")[0]) for pk, p in zip(df_cat["product_key"], df_cat["product"])}
    guard = {}
    for ik, pk in df_cat[["ingredient_key", "product_key"]].drop_duplicates().itertuples(index=False):
        ex_i = _excluded(ik, groups, ingredient_level=True)
        ex_p = _excluded(pk, groups, ingredient_level=False)
        if _SEASONING_HEAD_RX.match(_head.get(pk, "")):
            ex_p = ex_p | groups["meatfish"]   # e.g. "SEASONING, SEAFOOD": a spice blend, not fish
        guard[(ik, pk)] = (ex_i | ex_p, bool(ex_p - ex_i))
    def _ex(ik, pk):
        return guard.get((ik, pk), (set(), False))[0]
    def _product_only(ik, pk):
        return guard.get((ik, pk), (set(), False))[1]
    def _drop_guarded():
        """Undo earlier matches that fall in a category ruled out for that pair (they go on to fuzzy/GPT)."""
        bad = [i for i, (ik, pk, c) in enumerate(zip(df_cat["ingredient_key"], df_cat["product_key"], df_cat["category"]))
               if isinstance(c, str) and _norm(c) in _ex(ik, pk)]
        if bad:
            df_cat.loc[df_cat.index[bad], ["category", "match_method"]] = pd.NA

    # Layer 0: fixed rules for common plant-based foods (coconut milk/cream, margarine, plant milks,
    # nut/seed butters, vegan cheese/yogurt, plant-based meat). Not saved; they apply on every run.
    with log_time("S2.2b plant-based rules"):
        rule = {ik: _plant_rule(ik) for ik in df_cat["ingredient_key"].drop_duplicates()}
        rule = {ik: allowed_key2disp[_norm(c)] for ik, c in rule.items() if c and _norm(c) in allowed_key2disp}
        if rule:
            hit = df_cat["ingredient_key"].map(rule)
            m = hit.notna() & df_cat["category"].isna()
            df_cat.loc[m, "category"] = hit[m]
            df_cat.loc[m, "match_method"] = "rule_plant"

    # Layer 1: exact CATEGORY_MAP (curated)
    with log_time("S2.3 exact CATEGORY_MAP"):
        if global_map:
            hit = df_cat["ingredient_key"].map(global_map)
            m = hit.notna() & df_cat["category"].isna()
            df_cat.loc[m, "category"] = hit[m]
            df_cat.loc[m, "match_method"] = "map_exact"

    # Layer 2: resolve_exact_category (if provided), on unique unknowns
    with log_time("S2.3b resolve_exact_category"):
        if "resolve_exact_category" in globals():
            mask = df_cat["category"].isna()
            uniq = df_cat.loc[mask, "ingredient_key"].drop_duplicates()
            if not uniq.empty:
                res = {k: resolve_exact_category(k) for k in uniq}
                hit = df_cat["ingredient_key"].map(res)
                m = mask & hit.notna()
                df_cat.loc[m, "category"] = hit[m]
                df_cat.loc[m, "match_method"] = "map_resolve"

    # Layer 3: foodcategories.json dict (ingredient-only)
    with log_time("S2.4 json cache (ingredient only)"):
        if json_map:
            mask = df_cat["category"].isna()
            if mask.any():
                hit = df_cat.loc[mask, "ingredient_key"].map(json_map)
                m = hit.notna()
                idx = hit.index[m]
                df_cat.loc[idx, "category"] = hit.loc[idx]
                df_cat.loc[idx, "match_method"] = "cache_json"

    # Layer 4: plant milk heuristic
    with log_time("S2.5 milk heuristic"):
        mask = df_cat["category"].isna()
        if mask.any():
            cand = df_cat.loc[mask, "ingredient_key"].map(lambda k: _milk_heuristic(k, allowed_key2disp))
            m = cand.notna()
            idx = cand.index[m]
            df_cat.loc[idx, "category"] = cand.loc[idx]
            df_cat.loc[idx, "match_method"] = "milk_heuristic"

    # Matches from the lists above that are ruled out by the plant-based guard go on to fuzzy/GPT instead
    _drop_guarded()

    # Layer 5: fuzzy to allowed (high cutoff), with vegan guard (dedup pairs)
    with log_time("S2.6 fuzzy (unique pairs)"):
        mask = df_cat["category"].isna()
        fuzzy_df = pd.DataFrame(columns=["ingredient_key","product_key","category","match_method"])
        if mask.any():
            pairs = df_cat.loc[mask, ["ingredient_key","product_key"]].drop_duplicates()
            rows = []
            for _, r in pairs.iterrows():
                ik, pk = r["ingredient_key"], r["product_key"]
                best_norm, score = _fuzzy_best(ik, allowed_norm)
                if best_norm and score >= FUZZY_CUTOFF:
                    cat = allowed_key2disp[best_norm]
                    if best_norm in _ex(ik, pk):
                        continue
                    rows.append({"ingredient_key": ik, "product_key": pk, "category": cat, "match_method": "fuzzy"})
            if rows:
                fuzzy_df = pd.DataFrame(rows)
                df_cat = df_cat.merge(fuzzy_df, on=["ingredient_key","product_key"], how="left", suffixes=("","_fzy"))
                fill = df_cat["category"].isna() & df_cat["category_fzy"].notna()
                df_cat.loc[fill, "category"] = df_cat.loc[fill, "category_fzy"]
                df_cat.loc[fill, "match_method"] = "fuzzy"
                df_cat.drop(columns=[c for c in ("category_fzy","match_method_fzy") if c in df_cat.columns], inplace=True)
                # Persist fuzzy hits using the mode
                keep = [not _product_only(ik, pk) for ik, pk in zip(fuzzy_df["ingredient_key"], fuzzy_df["product_key"])]
                add_fuzzy = (fuzzy_df[keep][["ingredient_key","category"]]
                             .drop_duplicates("ingredient_key"))
                appended_json_total += _append_foodcats_map(json_cache_path, add_fuzzy, update_mode)


    # Layer 6: per-run CSV cache
    with log_time("S2.7 run CSV cache"):
        if not run_cache.empty:
            cache_hit = df_cat[["ingredient_key","product_key"]].merge(run_cache, on=["ingredient_key","product_key"], how="left")
            has = df_cat["category"].isna() & cache_hit["category"].notna()
            df_cat.loc[has, "category"] = cache_hit.loc[has, "category"]
            df_cat.loc[has, "match_method"] = cache_hit.loc[has, "match_method"].fillna("gpt_cached")
            _drop_guarded()

    # Layer 7: GPT fallback (dedup pairs; strict to allowed)
    with log_time("S2.8 GPT (unique unknown pairs; constrained)"):
        need = df_cat["category"].isna() | df_cat["category"].eq("unknown")
        if need.any():
            if "_call_gpt" not in globals():
                print("[S2.8] WARNING: _call_gpt not defined; skipping GPT fallback.")
                gpt_df = pd.DataFrame(columns=["ingredient_key","product_key","category","match_method"])
            else:
                pairs = df_cat.loc[need, ["ingredient_key","product_key"]].drop_duplicates()
                allowed_list = list(allowed_key2disp.values())
    
                rows = []
                for _, r in pairs.iterrows():
                    ik, pk = r["ingredient_key"], r["product_key"]
                    ex = _ex(ik, pk)
                    cat = _call_gpt_categorize(ik, pk, allowed_list, excluded=ex,
                                               guard_msg=_guard_message(ex, groups))
                    if cat:
                        rows.append({
                            "ingredient_key": ik,
                            "product_key": pk,
                            "category": cat,
                            "match_method": "gpt"
                        })

                gpt_df = pd.DataFrame(rows, columns=["ingredient_key","product_key","category","match_method"])
                
                # --- After GPT: persist using the mode ---
                if not gpt_df.empty:
                    _update_run_cache(csv_cache_path, gpt_df)  # keep your per-run cache
                    keep = [not _product_only(ik, pk) for ik, pk in zip(gpt_df["ingredient_key"], gpt_df["product_key"])]
                    add_gpt = (gpt_df[pd.Series(keep, index=gpt_df.index) & (gpt_df["category"].astype(str).str.lower() != "unknown")]
                               [["ingredient_key","category"]]
                               .drop_duplicates("ingredient_key"))
                    appended_json_total += _append_foodcats_map(json_cache_path, add_gpt, update_mode)
                
                # Keep your existing merge-back block unchanged:
                if not gpt_df.empty:
                    df_cat = df_cat.merge(
                        gpt_df, on=["ingredient_key","product_key"], how="left", suffixes=("","_gpt")
                    )
                    fill = (df_cat["category"].isna() | df_cat["category"].eq("unknown")) & df_cat["category_gpt"].notna()
                    df_cat.loc[fill, "category"] = df_cat.loc[fill, "category_gpt"]
                    df_cat.loc[fill, "match_method"] = "gpt"
                    df_cat.drop(columns=[c for c in ("category_gpt","match_method_gpt") if c in df_cat.columns], inplace=True)

    # Finalize unknowns
    with log_time("S2.9 finalize unknowns"):
        still = df_cat["category"].isna()
        df_cat.loc[still, "category"] = "unknown"
        df_cat.loc[still, "match_method"] = df_cat.loc[still, "match_method"].fillna("unknown")

    # Canned / cooked beans, rice and pasta: dry equivalent + water; broth / stock: a small share of its category
    # + water (cooked_split, COOKED_DRY_SHARE and BROTH_EQUIVALENT_SHARE, Cell 2)
    with log_time("S2.9a canned / cooked / broth split"):
        df_cat, _n_cooked = cooked_split(df_cat)
        if _n_cooked:
            print(f"[S2] {_n_cooked} canned or cooked bean, rice or pasta, broth or soup base ingredient(s) or "
                  f"product(s) adjusted (match_method rule_cooked / rule_broth / rule_base)")

    # bone_in / fish_cut only matter for the categories Stage 3 applies edible
    # yields to (Cell 8's _SPECIES_PATTERNS -- keep these in sync): clear them on
    # every other row, so e.g. whole cloves or steak-cut fries carry no fish cut.
    _FISH_CAT_RX = re.compile(
        r"\bfish|salmon|tuna|cod|haddock|pollock|anchov|sardine|halibut|mahi|"
        r"catfish|tilapia|trout|bass|hake|sole|flounder|whiting|"
        r"carp|congers|milkfish|mullets|sauries|smelt|amberjack|unagi|barramundi|"
        r"jacks|pangasius|shrimp|crab|lobster|octopus|squid|"
        r"mackerel|herring\b", re.I)
    _YIELD_CAT_RXS = [re.compile(p, re.I) for p in (r"\bchicken\b", r"\bturkey\b", r"\bbeef|bovine|cow\b",
                                                    r"\bpork|swine\b", r"\blamb|mutton\b", r"\blamb|goat\b")]
    _cat_txt = df_cat["category"].astype(str)
    _is_fish_cat = _cat_txt.str.contains(_FISH_CAT_RX)
    _is_yield_cat = _is_fish_cat.copy()
    for _rx in _YIELD_CAT_RXS:
        _is_yield_cat |= _cat_txt.str.contains(_rx)
    if "fish_cut" in df_cat.columns:
        df_cat["fish_cut"] = df_cat["fish_cut"].where(_is_fish_cat, "")
    if "bone_in" in df_cat.columns:
        df_cat["bone_in"] = df_cat["bone_in"].where(_is_yield_cat, "")

    # ----------------- Outputs (preserve row_sheet; sorted) -----------------
    with log_time("S2.A write outputs"):

        # Categories file: keep key columns up front; keep the rest as-is
        front = [c for c in ["row_sheet","product","qty","ingredient","percent","category","match_method"] if c in df_cat.columns]
        rest  = [c for c in df_cat.columns if c not in front and c not in ("ingredient_key","product_key")]
        df_out = df_cat[front + rest].copy()
        if "row_sheet" in df_out.columns:
            df_out["row_sheet"] = pd.to_numeric(df_out["row_sheet"], errors="coerce")
            df_out = df_out.sort_values(["row_sheet","product","ingredient"], kind="stable")
        out_path = result_out(f"categories_{base}.csv")
        df_out.to_csv(out_path, index=False)

    with log_time("S2.B summary"):
        print(f"[Stage 2] Wrote: {out_path.name} ({len(df_out):,} rows)")
        if update_mode == "disabled":
            print(f"[S2.cache] Mode='{update_mode}': {appended_json_total} new pairs found but NOT stored")
        elif update_mode == "separate":
            print(f"[S2.cache] Mode='{update_mode}': {appended_json_total} new pairs appended to newcategories.json")
        else:  # append
            print(f"[S2.cache] Mode='{update_mode}': {appended_json_total} new pairs appended to foodcategories.json")

    with log_time("S2.Z cleanup"):
        try:
            if csv_cache_path.exists():
                csv_cache_path.unlink()
                print(f"[S2.cache] Deleted per-run cache {csv_cache_path.name}")
        except Exception as e:
            print(f"[S2.cache] WARN: could not delete {csv_cache_path.name}: {e}")

    return df_out

# --- merged categories column order for run_stage_2 (v100f) ---
from pathlib import Path
import pandas as pd
def run_stage_2(*args, **kwargs):
    # Policy (v100g):
    # - Reorders the categories DataFrame so 'category' appears immediately after 'ingredient'.
    # - Persists that order to categories_{BASE}.csv when base is available.
    df = run_stage_2_base(*args, **kwargs)
    try:
        cols = list(df.columns)
        if "ingredient" in cols and "category" in cols:
            new_cols = []
            for c in cols:
                if c == "ingredient":
                    new_cols.append("ingredient"); new_cols.append("category")
                elif c == "category":
                    continue
                else:
                    new_cols.append(c)
            df = df[new_cols]
        base = kwargs.get("base") or kwargs.get("base_filename")
        if base is None and len(args) >= 1:
            base = args[0]
        if base:
            df.to_csv(result_out(f"categories_{base}.csv"), index=False)
    except Exception as e:
        print(f"[v100f] Could not reorder/persist categories output: {e}")
    return df


In [ ]:
# === Cell 8 — Stage 3: Impact Calculation ===
# Purpose: Quantify environmental and animal-welfare impacts for each categorized ingredient.
# Inputs:
#   - categories_{BASE}.csv (Stage 2 output)
#   - factors.csv (impact conversion factors, METRIC ONLY: all per kg —
#       kcal, kg CO2e, kg carbon_opp, lives, liters blue water, liters green water, m² land, g PO₄³⁻ eq)
#   - optional CATEGORY_WEIGHT_MULTIPLIERS (from config)
# Outputs (OUTPUT_FILES in Cell 1 chooses which are written):
#   - impacts_metric_{BASE}.csv — impacts in kg, m², L, etc.
#   - impacts_lbs_{BASE}.csv — impacts in lb, ft², gal, etc.
# Behavior:
#   - Joins category data with factors table.
#   - Reads weights in kg (columns ending _kg, from a metric worksheet) or lbs (columns ending
#     _lbs); converts lbs to kg only when the weights are in lbs.
#   - Computes all impacts from metric (per-kg) factors × edible kg,
#     then converts to Imperial units for impacts_lbs_{BASE}.csv.
#   - Applies category multipliers.
#   - Canned / cooked beans, rice and pasta count as their dry equivalent plus water (cooked_split, Cell 2;
#     Stage 2 writes the split rows, and a categories file from an older version is split here for the run).
#   - Aggregates impacts by product and category.
# Notes:
#   - Requires that categories_{BASE}.csv exists (raises FileNotFoundError otherwise).
#   - Respects USE_CATEGORY_WEIGHT_MULTIPLIERS flag.

from pathlib import Path
import pandas as pd
import numpy as np
import re  # (added) for species pattern matching in bone-in shim

# ---- Category weight multipliers (toggle + map) ----
# Multiplies weights by a category-specific factor in addition to any existing logic.
# set in Cell 1 not here: USE_CATEGORY_WEIGHT_MULTIPLIERS = True or set False to disable

# Keys are matched using the same normalization your cell uses (via _norm at call-time).

USE_CATEGORY_WEIGHT_MULTIPLIERS = bool(globals().get("USE_CATEGORY_WEIGHT_MULTIPLIERS", False))
CATEGORY_WEIGHT_MULTIPLIERS = globals().get("CATEGORY_WEIGHT_MULTIPLIERS", {})  # empty if not set

def _mult_for_category(cat: str) -> float:
    if not USE_CATEGORY_WEIGHT_MULTIPLIERS:
        return 1.0
    try:
        key = _norm(cat)  # uses your notebook normalizer if present
    except Exception:
        key = " ".join(str(cat).strip().lower().split())
    return CATEGORY_WEIGHT_MULTIPLIERS.get(key, 1.0)


def calculate_impacts(df_cat: pd.DataFrame, factors_csv: str, base: str):
    """
    Inputs:
      df_cat: categories_{BASE}.csv already loaded into a DataFrame
      factors_csv: path to factors.csv
      base: BASE string for file naming

    Outputs:
      Writes impacts_metric_{BASE}.csv and/or impacts_lbs_{BASE}.csv (OUTPUT_FILES in Cell 1)
      Returns (df_lbs_out, df_metric_out) formatted as written to disk (both always calculated)
    """
    LB_TO_KG = 0.45359237
    KG_TO_LB = 1.0 / LB_TO_KG

    # ---- Helpers ----
    def _norm(x: str) -> str:
        try:
            return norm_text(x)  # use your notebook's normalizer if present
        except Exception:
            s = str(x).strip().lower()
            s = " ".join(s.split())
            return s

    # ---- Load/normalize factors ----
    fac = pd.read_csv(factors_csv)
    # factors.csv is metric-only (per kg). The old per-lb file had water_US and
    # land_acres columns and per-lb lives; using it here would understate lives ~2.2x.
    _legacy = [c for c in fac.columns if c.strip().lower() in ("water_us", "land_acres")]
    if _legacy:
        raise ValueError(f"{factors_csv} has legacy Imperial columns {_legacy} — it looks like "
                         f"the old per-lb factors file. Use the metric-only factors.csv.")
    # From factorsb.csv on, water is split into blue_water and green_water columns.
    if "blue_water" not in fac.columns or "green_water" not in fac.columns:
        _old = " (it still has the old 'water' column)" if "water" in fac.columns else ""
        raise ValueError(f"{factors_csv} needs blue_water and green_water columns{_old}. "
                         f"Use factors.csv from factorsb.csv or later.")
    if "category" not in fac.columns:
        fac = fac.rename(columns={fac.columns[0]: "category"})
    # Remember which categories have a green-water factor (blank = no factor) before blanks become 0
    fac["__green_known"] = pd.to_numeric(fac["green_water"], errors="coerce").notna()
    fac["__in_factors"] = True
    # Coerce factor columns to numeric, create missing as zeros
    need_cols = ["kcal","co2","carbon_opp","d_lives","t_lives","blue_water","green_water","land_metric","eutro"]
    for c in need_cols:
        if c not in fac.columns:
            fac[c] = 0.0
        fac[c] = pd.to_numeric(fac[c], errors="coerce").fillna(0.0)

    # Build normalized keys for merge
    fac = fac.copy()
    fac["__category_key"] = fac["category"].astype(str).map(_norm)

    # ---- Normalize df_cat columns and weights ----
    cats = df_cat.copy()
    cols = {c.lower(): c for c in cats.columns}

    if "category" not in cols:
        raise KeyError("categories file is missing a 'category' column.")
    cat_col = cols["category"]
    cats["__category_key"] = cats[cat_col].astype(str).map(_norm)

    # Weights are in kg (metric worksheet: *_kg columns) or lbs (US worksheet: *_lbs columns).
    # Priority within a unit: ingredient_weight > weight > product_weight * percent
    def _colnum(name):
        return pd.to_numeric(cats[name], errors="coerce") if name in cats.columns else None

    def _pick_weight(suf):
        w = None
        if f"ingredient_weight_{suf}" in cols:
            w = _colnum(cols[f"ingredient_weight_{suf}"])
        if w is None or w.isna().all():
            if f"weight_{suf}" in cols:
                w = _colnum(cols[f"weight_{suf}"])
        if w is None or w.isna().all():
            if (f"product_weight_{suf}" in cols) and (("percent" in cols) or ("pct" in cols)):
                pct_col = cols.get("percent") or cols.get("pct")
                p_w = _colnum(cols[f"product_weight_{suf}"])
                pct = pd.to_numeric(cats[pct_col], errors="coerce")
                w = p_w * (pct / 100.0)
        return w

    has_kg  = any(f"{c}_kg" in cols for c in ("ingredient_weight", "weight", "product_weight"))
    has_lbs = any(f"{c}_lbs" in cols for c in ("ingredient_weight", "weight", "product_weight"))
    if has_kg and has_lbs:
        raise ValueError("The categories file has both _kg and _lbs weight columns. Keep only one set "
                         "(e.g. delete the _lbs columns of a metric worksheet) and re-run.")
    w_unit = "kg" if has_kg else "lb"
    w_in = _pick_weight("kg" if has_kg else "lbs")

    if w_in is None:
        raise KeyError("No usable weight found (need ingredient_weight_kg/_lbs OR weight_kg/_lbs OR "
                       "product_weight_kg/_lbs + percent).")
    print(f"[impacts] Reading weights in {w_unit} from the categories file.")

    # ======= BEGIN bone-in / fish-cut edible-weight shim =======
    # Ensure the flags exist
    if "bone_in" not in cats.columns:
        cats["bone_in"] = False
    if "fish_cut" not in cats.columns:
        cats["fish_cut"] = ""

    # Species patterns + yields (land animals: unchanged binary bone-in/boneless model)
    EDIBLE_YIELD_DEFAULTS = {
        ("chicken", False): 1,   ("chicken", True): 0.7708,
        ("turkey",  False): 1,   ("turkey",  True): 0.7708,
        ("beef",    False): 1,   ("beef",    True): 0.8667,
        ("pork",    False): 1,   ("pork",    True): 0.8784,
        ("lamb",    False): 1,   ("lamb",    True): 0.6643,
        ("goat",    False): 1,   ("goat",    True): 0.6643,
        ("fish",    False): 1,   ("fish",    True): 0.57,  # fallback only: used for a fish row with no
                                                             # recognized cut (fish_cut empty) whose text
                                                             # Cell 6's detect_bone_in() still flagged
    }
    _SPECIES_PATTERNS = [
        ("chicken", re.compile(r"\bchicken\b", re.I)),
        ("turkey",  re.compile(r"\bturkey\b",  re.I)),
        ("beef",    re.compile(r"\bbeef|bovine|cow\b", re.I)),
        ("pork",    re.compile(r"\bpork|swine\b", re.I)),
        ("lamb",    re.compile(r"\blamb|mutton\b", re.I)),
        ("goat",    re.compile(r"\blamb|goat\b", re.I)),
        ("fish",    re.compile(
            r"\bfish|salmon|tuna|cod|haddock|pollock|anchov|sardine|halibut|mahi|"
            r"catfish|tilapia|trout|bass|hake|sole|flounder|whiting|"
            r"carp|congers|milkfish|mullets|sauries|smelt|amberjack|unagi|barramundi|"
            r"jacks|pangasius|shrimp|crab|lobster|octopus|squid|"  # factors.csv categories with
                                                                     # FISH_CUT_YIELDS curves below
            r"mackerel|herring\b", re.I)),
    ]
    def _species_key(cat_str: str):
        c = str(cat_str or "")
        for species, rx in _SPECIES_PATTERNS:
            if rx.search(c):
                return species
        return None

    # ---------------------------------------------------------------------
    # Fish/seafood: species- AND cut-specific edible yield.
    #
    # Replaces the old flat ("fish", True): 0.57 constant for every fish row
    # whose category we can resolve. Each species' curve is anchored at both
    # ends to figures we trust -- "Whole / Round" = that species' own
    # round-weight edible fraction (Gephart et al., via FishPage.csv, the
    # same basis d_lives/t_lives were built from), "Skinless Fillet" = 1.0
    # (already-edible, matching this code's longstanding default) -- with
    # the cuts in between placed proportionally using fish_yield_matrix.json
    # (chefs-resources.com) for SHAPE only, not as absolute values, since
    # that source's own absolute scale doesn't share the live-weight basis
    # the rest of this notebook uses.
    #
    # "__generic_fish__" is the fallback for a resolved species with no
    # matching curve (e.g. the catch-all "fish (finfish)" category, or any
    # future category not in this table): the same anchor-and-rescale
    # method, but using the average shape across all curved species and
    # anchored to "fish (finfish)"'s own Gephart edible fraction (0.5578) --
    # which is, not coincidentally, almost exactly the old flat 0.57.
    #
    # Source data: fish_yield_matrix.json (chefs-resources.com-derived cut
    # curve) x FishPage.csv (Gephart et al. live-weight/edible% anchors).
    # Recompute this table if either source is revised.
    #
    # Hand-corrected: "tuna", "skipjack tuna" and "yellowfin tuna" had H&G at
    # or below Drawn (H&G removes more waste, so its yield must be higher).
    # Their H&G values are set to the midpoint of Drawn and Dressed until the
    # curves are recomputed from source. A check below warns about any curve
    # that decreases.
    # ---------------------------------------------------------------------
    FISH_CUT_YIELDS = {
        "anchovies": {"Whole / Round": 0.6019, "Drawn": 0.6815, "H&G": 0.7213, "Dressed": 0.7611, "Steaks / Cutlets": 0.8408, "Bone-In Fillet": 0.9204, "Skin-On Fillet": 0.9602, "Skinless Fillet": 1.0},
        "barramundi": {"Whole / Round": 0.5305, "Drawn": 0.6062, "H&G": 0.735, "Dressed": 0.7728, "Steaks / Cutlets": 0.8485, "Bone-In Fillet": 0.9091, "Skin-On Fillet": 0.9697, "Skinless Fillet": 1.0},
        "bass": {"Whole / Round": 0.4726, "Drawn": 0.5396, "H&G": 0.6819, "Dressed": 0.7656, "Steaks / Cutlets": 0.8493, "Bone-In Fillet": 0.9079, "Skin-On Fillet": 0.9665, "Skinless Fillet": 1.0},
        "carp": {"Whole / Round": 0.54, "Drawn": 0.6108, "H&G": 0.7169, "Dressed": 0.7877, "Steaks / Cutlets": 0.8585, "Bone-In Fillet": 0.9151, "Skin-On Fillet": 0.9646, "Skinless Fillet": 1.0},
        "carp silver or bighead carp": {"Whole / Round": 0.54, "Drawn": 0.6076, "H&G": 0.7159, "Dressed": 0.7835, "Steaks / Cutlets": 0.8512, "Bone-In Fillet": 0.9053, "Skin-On Fillet": 0.9662, "Skinless Fillet": 1.0},
        "catfish": {"Whole / Round": 0.5305, "Drawn": 0.6088, "H&G": 0.7105, "Dressed": 0.7887, "Steaks / Cutlets": 0.867, "Bone-In Fillet": 0.8983, "Skin-On Fillet": 0.9687, "Skinless Fillet": 1.0},
        "cod": {"Whole / Round": 0.4962, "Drawn": 0.5581, "H&G": 0.6641, "Dressed": 0.7525, "Steaks / Cutlets": 0.8232, "Bone-In Fillet": 0.8851, "Skin-On Fillet": 0.9646, "Skinless Fillet": 1.0},
        "congers": {"Whole / Round": 0.4726, "Drawn": 0.5685, "H&G": 0.6644, "Dressed": 0.7603, "Steaks / Cutlets": 0.8562, "Bone-In Fillet": 0.9041, "Skin-On Fillet": 0.9712, "Skinless Fillet": 1.0},
        "crab": {"Whole / Round": 0.391, "Drawn": 0.4722, "H&G": 0.594, "Dressed": 0.6752, "Steaks / Cutlets": 0.797, "Bone-In Fillet": 0.8782, "Skin-On Fillet": 0.9188, "Skinless Fillet": 1.0},
        "european sea bass (branzino)": {"Whole / Round": 0.51, "Drawn": 0.5722, "H&G": 0.7044, "Dressed": 0.7822, "Steaks / Cutlets": 0.86, "Bone-In Fillet": 0.9144, "Skin-On Fillet": 0.9689, "Skinless Fillet": 1.0},
        "flounder": {"Whole / Round": 0.5275, "Drawn": 0.595, "H&G": 0.7637, "Dressed": 0.7975, "Steaks / Cutlets": 0.8515, "Bone-In Fillet": 0.8987, "Skin-On Fillet": 0.9663, "Skinless Fillet": 1.0},
        "haddock": {"Whole / Round": 0.4962, "Drawn": 0.5382, "H&G": 0.6641, "Dressed": 0.7481, "Steaks / Cutlets": 0.8321, "Bone-In Fillet": 0.9076, "Skin-On Fillet": 0.9664, "Skinless Fillet": 1.0},
        "hake": {"Whole / Round": 0.4962, "Drawn": 0.5777, "H&G": 0.637, "Dressed": 0.7407, "Steaks / Cutlets": 0.837, "Bone-In Fillet": 0.8889, "Skin-On Fillet": 0.963, "Skinless Fillet": 1.0},
        "halibut": {"Whole / Round": 0.5275, "Drawn": 0.615, "H&G": 0.6937, "Dressed": 0.7812, "Steaks / Cutlets": 0.825, "Bone-In Fillet": 0.895, "Skin-On Fillet": 0.965, "Skinless Fillet": 1.0},
        "herring": {"Whole / Round": 0.6019, "Drawn": 0.6815, "H&G": 0.7452, "Dressed": 0.801, "Steaks / Cutlets": 0.8567, "Bone-In Fillet": 0.9204, "Skin-On Fillet": 0.9682, "Skinless Fillet": 1.0},
        "jacks": {"Whole / Round": 0.5409, "Drawn": 0.616, "H&G": 0.7078, "Dressed": 0.783, "Steaks / Cutlets": 0.8497, "Bone-In Fillet": 0.8998, "Skin-On Fillet": 0.9583, "Skinless Fillet": 1.0},
        "japanese amberjack (hamachi)": {"Whole / Round": 0.5409, "Drawn": 0.616, "H&G": 0.7078, "Dressed": 0.783, "Steaks / Cutlets": 0.8497, "Bone-In Fillet": 0.8998, "Skin-On Fillet": 0.9583, "Skinless Fillet": 1.0},
        "lobster": {"Whole / Round": 0.2, "Drawn": 0.3111, "H&G": 0.5, "Dressed": 0.6111, "Steaks / Cutlets": 0.7222, "Bone-In Fillet": 0.8333, "Skin-On Fillet": 0.9111, "Skinless Fillet": 1.0},
        "mahi-mahi": {"Whole / Round": 0.6174, "Drawn": 0.687, "H&G": 0.7148, "Dressed": 0.7913, "Steaks / Cutlets": 0.8609, "Bone-In Fillet": 0.8957, "Skin-On Fillet": 0.9652, "Skinless Fillet": 1.0},
        "milkfish": {"Whole / Round": 0.61, "Drawn": 0.6772, "H&G": 0.7445, "Dressed": 0.8117, "Steaks / Cutlets": 0.8655, "Bone-In Fillet": 0.9193, "Skin-On Fillet": 0.9731, "Skinless Fillet": 1.0},
        "mullets": {"Whole / Round": 0.5409, "Drawn": 0.6174, "H&G": 0.6939, "Dressed": 0.7705, "Steaks / Cutlets": 0.847, "Bone-In Fillet": 0.9082, "Skin-On Fillet": 0.9617, "Skinless Fillet": 1.0},
        "octopus": {"Whole / Round": 0.6667, "Drawn": 0.8334, "H&G": 0.9167, "Dressed": 0.9667, "Steaks / Cutlets": 1.0, "Bone-In Fillet": 1.0, "Skin-On Fillet": 1.0, "Skinless Fillet": 1.0},
        "pangasius (basa or swai)": {"Whole / Round": 0.5305, "Drawn": 0.5666, "H&G": 0.675, "Dressed": 0.7472, "Steaks / Cutlets": 0.8411, "Bone-In Fillet": 0.8917, "Skin-On Fillet": 0.9639, "Skinless Fillet": 1.0},
        "pollock": {"Whole / Round": 0.4962, "Drawn": 0.542, "H&G": 0.6565, "Dressed": 0.7328, "Steaks / Cutlets": 0.8321, "Bone-In Fillet": 0.8855, "Skin-On Fillet": 0.9618, "Skinless Fillet": 1.0},
        "salmon": {"Whole / Round": 0.587, "Drawn": 0.6664, "H&G": 0.722, "Dressed": 0.7776, "Steaks / Cutlets": 0.857, "Bone-In Fillet": 0.9047, "Skin-On Fillet": 0.9523, "Skinless Fillet": 1.0},
        "sardines": {"Whole / Round": 0.6019, "Drawn": 0.68, "H&G": 0.719, "Dressed": 0.7814, "Steaks / Cutlets": 0.8439, "Bone-In Fillet": 0.9219, "Skin-On Fillet": 0.9688, "Skinless Fillet": 1.0},
        "sauries": {"Whole / Round": 0.5409, "Drawn": 0.6365, "H&G": 0.6939, "Dressed": 0.7609, "Steaks / Cutlets": 0.8278, "Bone-In Fillet": 0.9044, "Skin-On Fillet": 0.9617, "Skinless Fillet": 1.0},
        "shrimp": {"Whole / Round": 0.57, "Drawn": 0.6775, "H&G": 0.785, "Dressed": 0.8387, "Steaks / Cutlets": 0.8925, "Bone-In Fillet": 0.9463, "Skin-On Fillet": 0.9785, "Skinless Fillet": 1.0},
        "smelt": {"Whole / Round": 0.5933, "Drawn": 0.6837, "H&G": 0.7469, "Dressed": 0.8192, "Steaks / Cutlets": 0.8644, "Bone-In Fillet": 0.9277, "Skin-On Fillet": 0.9729, "Skinless Fillet": 1.0},
        "sole": {"Whole / Round": 0.5275, "Drawn": 0.594, "H&G": 0.7005, "Dressed": 0.7671, "Steaks / Cutlets": 0.8336, "Bone-In Fillet": 0.9002, "Skin-On Fillet": 0.9601, "Skinless Fillet": 1.0},
        "squid": {"Whole / Round": 0.6667, "Drawn": 0.7778, "H&G": 0.8334, "Dressed": 0.8889, "Steaks / Cutlets": 0.9445, "Bone-In Fillet": 1.0, "Skin-On Fillet": 1.0, "Skinless Fillet": 1.0},
        "tilapia": {"Whole / Round": 0.37, "Drawn": 0.4358, "H&G": 0.5769, "Dressed": 0.6709, "Steaks / Cutlets": 0.7649, "Bone-In Fillet": 0.859, "Skin-On Fillet": 0.9436, "Skinless Fillet": 1.0},
        "trout": {"Whole / Round": 0.585, "Drawn": 0.6588, "H&G": 0.8063, "Dressed": 0.8524, "Steaks / Cutlets": 0.8893, "Bone-In Fillet": 0.917, "Skin-On Fillet": 0.9539, "Skinless Fillet": 1.0},
        "tuna": {"Whole / Round": 0.6174, "Drawn": 0.691, "H&G": 0.7278, "Dressed": 0.7646, "Steaks / Cutlets": 0.8381, "Bone-In Fillet": 0.8896, "Skin-On Fillet": 0.9485, "Skinless Fillet": 1.0},
        "albacore tuna": {"Whole / Round": 0.6174, "Drawn": 0.7357, "H&G": 0.7496, "Dressed": 0.7565, "Steaks / Cutlets": 0.833, "Bone-In Fillet": 0.8957, "Skin-On Fillet": 0.9513, "Skinless Fillet": 1.0},
        "bluefin tuna": {"Whole / Round": 0.6174, "Drawn": 0.6763, "H&G": 0.7351, "Dressed": 0.794, "Steaks / Cutlets": 0.8352, "Bone-In Fillet": 0.8999, "Skin-On Fillet": 0.9529, "Skinless Fillet": 1.0},
        "skipjack tuna": {"Whole / Round": 0.6174, "Drawn": 0.687, "H&G": 0.7253, "Dressed": 0.7635, "Steaks / Cutlets": 0.8261, "Bone-In Fillet": 0.8887, "Skin-On Fillet": 0.9513, "Skinless Fillet": 1.0},
        "yellowfin tuna": {"Whole / Round": 0.6174, "Drawn": 0.6939, "H&G": 0.7284, "Dressed": 0.7628, "Steaks / Cutlets": 0.8317, "Bone-In Fillet": 0.8852, "Skin-On Fillet": 0.9541, "Skinless Fillet": 1.0},
        "unagi (freshwater eel)": {"Whole / Round": 0.5305, "Drawn": 0.6348, "H&G": 0.7392, "Dressed": 0.7913, "Steaks / Cutlets": 0.8748, "Bone-In Fillet": 0.9165, "Skin-On Fillet": 0.9583, "Skinless Fillet": 1.0},
        "whiting": {"Whole / Round": 0.4962, "Drawn": 0.5737, "H&G": 0.6512, "Dressed": 0.7287, "Steaks / Cutlets": 0.8062, "Bone-In Fillet": 0.8837, "Skin-On Fillet": 0.9535, "Skinless Fillet": 1.0},
        "__generic_fish__": {"Whole / Round": 0.5578, "Drawn": 0.636, "H&G": 0.7124, "Dressed": 0.7831, "Steaks / Cutlets": 0.8524, "Bone-In Fillet": 0.9091, "Skin-On Fillet": 0.9631, "Skinless Fillet": 1.0},
    }

    # category text -> FISH_CUT_YIELDS key, for names that don't match directly
    # (wild/farmed variants collapse to the base species; the mixed retail
    # category maps to the JSON's generic "tuna" curve; "fish (finfish)"
    # maps to the generic fallback curve).
    FISH_CATEGORY_ALIASES = {
        "silver or bighead carp": "carp silver or bighead carp",
        "bass (wild-caught)": "bass",
        "bass (farmed)": "bass",
        "catfish (wild-caught)": "catfish",
        "catfish (farmed)": "catfish",
        "milkfish (wild-caught)": "milkfish",
        "milkfish (farmed)": "milkfish",
        "salmon (wild-caught)": "salmon",
        "salmon (farmed)": "salmon",
        "shrimp (wild-caught)": "shrimp",
        "shrimp (farmed)": "shrimp",
        "tilapia (wild-caught)": "tilapia",
        "tilapia (farmed)": "tilapia",
        "trout (wild-caught)": "trout",
        "trout (farmed)": "trout",
        "tuna (skipjack, albacore, yellowfin)": "tuna",
        "fish (finfish)": "__generic_fish__",
    }

    _FISH_CUT_ORDER = ["Whole / Round", "Drawn", "H&G", "Dressed", "Steaks / Cutlets",
                       "Bone-In Fillet", "Skin-On Fillet", "Skinless Fillet"]
    for _sp, _curve in FISH_CUT_YIELDS.items():
        _vals = [_curve[k] for k in _FISH_CUT_ORDER]
        if any(b < a for a, b in zip(_vals, _vals[1:])):
            print(f"[fish-cut] WARNING: yield curve for '{_sp}' decreases somewhere along "
                  f"{_FISH_CUT_ORDER} -- check FISH_CUT_YIELDS.")

    _FISH_CUT_YIELDS_NORM = {_norm(k): v for k, v in FISH_CUT_YIELDS.items()}
    # readable shellfish labels (SHELLFISH_CUT_LABELS, Cell 2) -> curve point
    _CUT_LABEL_TO_KEY = {_norm(lbl): key
                         for _labels in globals().get("SHELLFISH_CUT_LABELS", {}).values()
                         for key, lbl in _labels.items()}
    _FISH_CATEGORY_ALIASES_NORM = {_norm(k): _norm(v) for k, v in FISH_CATEGORY_ALIASES.items()}

    def _fish_edible_yield(category_text: str, cut: str) -> Optional[float]:
        """Species+cut yield, or None when no recognized cut (caller keeps the flat yield)."""
        if not cut:
            return None
        cut = _CUT_LABEL_TO_KEY.get(_norm(cut), cut)
        key = _norm(category_text)
        key = _FISH_CATEGORY_ALIASES_NORM.get(key, key)
        # keys were normalized with _norm, which strips the underscores from
        # "__generic_fish__", so look the fallback up the same way
        table = _FISH_CUT_YIELDS_NORM.get(key) or _FISH_CUT_YIELDS_NORM[_norm("__generic_fish__")]
        return table.get(cut)

    # Apply edible yield row-by-row
    # Vectorized edible-yield using existing _SPECIES_PATTERNS + EDIBLE_YIELD_DEFAULTS
    cat_txt = cats[cat_col].astype(str).str.lower()
#    bone = cats["bone_in"].fillna(False).astype(bool) if "bone_in" in cats.columns else pd.Series(False, index=cats.index)
    bone = (
        cats["bone_in"].astype(str).str.strip().str.lower().eq("true")
    ) if "bone_in" in cats.columns else pd.Series(False, index=cats.index)

    
    # Assign first-matching species according to the order in _SPECIES_PATTERNS
    species = pd.Series(pd.NA, index=cats.index, dtype="object")
    for sp, rx in _SPECIES_PATTERNS:  # reuses your compiled patterns
        m = cat_txt.str.contains(rx)
        species.loc[species.isna() & m] = sp
    
    # Build yield maps from the single source of truth (no hardcoded numbers here)
    boneless_map = {sp: y for (sp, b), y in EDIBLE_YIELD_DEFAULTS.items() if not b}
    bone_map     = {sp: y for (sp, b), y in EDIBLE_YIELD_DEFAULTS.items() if b}
    
    yield_boneless = species.map(boneless_map).fillna(1.0)
    yield_bone     = species.map(bone_map)
    
    # Choose bone-in yield when bone=True, otherwise boneless; default to 1.0
    edible_yield = yield_boneless.where(~bone, yield_bone).fillna(yield_boneless)

    # Fish rows: override with the species+cut-specific yield above instead of
    # the flat bone-in/boneless pair (land animals are untouched by this).
    # Rows with no recognized cut keep the flat yield computed above: 1.0, or
    # the ("fish", True) 0.57 fallback when Cell 6 flagged the text bone-in.
    fish_mask = species.eq("fish")
    if fish_mask.any():
        # categories_{BASE}.csv is re-read with pd.read_csv, so an empty
        # fish_cut arrives as NaN ("nan" after astype(str)) -- normalize those
        # back to "" so "no cut" is recognized.
        if "fish_cut" in cats.columns:
            fish_cut_col = (cats["fish_cut"].fillna("").astype(str).str.strip()
                            .replace({"nan": "", "NaN": "", "None": ""}))
        else:
            fish_cut_col = pd.Series("", index=cats.index)
        fish_yield = pd.Series(
            [
                _fish_edible_yield(cat, cut)
                for cat, cut in zip(cats.loc[fish_mask, cat_col].astype(str), fish_cut_col.loc[fish_mask])
            ],
            index=cats.index[fish_mask],
            dtype="float64",
        ).reindex(cats.index)
        edible_yield = edible_yield.where(fish_yield.isna(), fish_yield)

        # Flag fish rows where Cell 6's detect_fish_cut() found no recognized
        # cut language (fish_cut == ""). Those rows fell back to the flat
        # yield above -- write them out so they can be spot-checked instead
        # of silently assumed correct.
        fish_cut_for_mask = fish_cut_col.loc[fish_mask]
        unclassified_fish = fish_cut_for_mask.eq("")
        n_unclassified = int(unclassified_fish.sum())
        if n_unclassified:
            flagged_idx = fish_cut_for_mask.index[unclassified_fish.values]
            flag_cols = []
            for _key in ("row_sheet", "product", "ingredient"):
                if _key in cols:
                    flag_cols.append(cols[_key])
            flag_cols.append(cat_col)
            for _key in ("bone_in", "fish_cut"):
                if _key in cats.columns:
                    flag_cols.append(_key)
            flag_cols = list(dict.fromkeys(flag_cols))  # de-dupe, preserve order
            flagged_df = cats.loc[flagged_idx, flag_cols].copy()
            flagged_df["edible_yield_applied"] = edible_yield.loc[flagged_idx].values
            _msg = (f"[fish-cut] {n_unclassified} fish row(s) had no recognized cut language "
                    f"in the invoice text -- yield defaulted to 1.0 (100% edible), or "
                    f"{EDIBLE_YIELD_DEFAULTS[('fish', True)]} where the text was flagged bone-in.")
            try:
                flagged_df.to_csv(result_out(f"unclassified_fish_cuts_{base}.csv"), index=False, encoding="utf-8")
                print(f"{_msg} Flagged rows written to unclassified_fish_cuts_{base}.csv for review.")
            except Exception as _e:
                print(f"{_msg} (Could not write unclassified_fish_cuts_{base}.csv: {_e})")
    

    edible_in = pd.to_numeric(w_in, errors="coerce").fillna(0.0) * edible_yield.fillna(1.0)
    if w_unit == "kg":
        edible_kg  = edible_in
        edible_lbs = edible_kg * KG_TO_LB
    else:
        edible_lbs = edible_in
        edible_kg  = edible_lbs * LB_TO_KG
    # ======= END bone-in / fish-cut shim =======

    # Store per-row edible weights under the same internal names v90 expects
    cats["__weight_lbs"] = edible_lbs
    cats["__weight_kg"]  = edible_kg

    # ---- Apply category multipliers to produce/extend effective weights ----
    # Start from existing effective weights if your bone-in/edible-yield logic already created them,
    # otherwise start from the base weights. We only apply the category multiplier here to avoid
    # touching your existing edible-yield math.
    base_lbs = cats["__weight_lbs_eff"] if "__weight_lbs_eff" in cats.columns else cats["__weight_lbs"]
    base_kg  = cats["__weight_kg_eff"]  if "__weight_kg_eff"  in cats.columns else cats["__weight_kg"]
    
    # Per-row category multiplier
    cat_mult = cats["category"].astype(str).map(_mult_for_category) if "category" in cats.columns else \
               cats[cat_col].astype(str).map(_mult_for_category)
    
    cats["__weight_lbs_eff"] = pd.to_numeric(base_lbs, errors="coerce").fillna(0.0) * cat_mult
    cats["__weight_kg_eff"]  = pd.to_numeric(base_kg,  errors="coerce").fillna(0.0) * cat_mult


    # ---- DEBUG: are edible weights changing? ----
    w_label = "kg" if w_unit == "kg" else "lbs"
    try:
        base_sum = float(pd.to_numeric(w_in, errors="coerce").fillna(0).sum())
    except Exception:
        base_sum = float("nan")
    edible_sum = float(pd.to_numeric(edible_in, errors="coerce").fillna(0).sum())
    print(f"[debug] base {w_label} sum={base_sum:,.2f}   edible {w_label} sum={edible_sum:,.2f}   delta={edible_sum - base_sum:,.2f}")
    
    # Check a few chicken rows and their yields/flags
    chix = cats[cats[cat_col].astype(str).str.contains("chicken", case=False, regex=True)]
    print(f"[debug] chicken rows={len(chix)}   bone_in True count={int(chix.get('bone_in', pd.Series()).isin([True,'True',1,'1']).sum())}")
    print(chix[[cat_col, "bone_in", f"__weight_{w_label}"]].head(8).to_string(index=False))

    # ---- Group by category ----
    grp = (
        cats.groupby("__category_key", as_index=False)
            .agg(total_weight_lbs=("__weight_lbs_eff", "sum"),
                 total_weight_kg=("__weight_kg_eff",  "sum"))
    )


    # ---- Merge display names (majority label per key) ----
    disp = (
        cats.assign(__category_disp=cats[cat_col].astype(str))
            .groupby("__category_key")["__category_disp"]
            .agg(lambda s: s.value_counts(dropna=False).idxmax())
            .reset_index()
    )
    out = grp.merge(disp, on="__category_key", how="left")
    out.rename(columns={"__category_disp":"category"}, inplace=True)

    # ---- Merge factors ----
    out = out.merge(fac, on="__category_key", how="left", suffixes=("","_f"))

    # Purchased weight with no green-water factor (blank in factors.csv, or a category not in factors.csv)
    # Categories with no row in factors.csv are counted as zero for every impact -- say so once
    _not_in_fac = out["__in_factors"].isna()
    _kg_missing = float(out.loc[_not_in_fac, "total_weight_kg"].sum())
    if _kg_missing > 0:
        _total_kg = float(out["total_weight_kg"].sum())
        _miss = ", ".join(sorted(out.loc[_not_in_fac, "category"].astype(str)))
        print(f"Note: {_kg_missing:,.1f} kg ({100 * _kg_missing / max(_total_kg, 1e-9):.1f}% of purchases) had no "
              f"impact factors and is counted as zero: {_miss}")

    _no_green = out["__in_factors"].notna() & ~out["__green_known"].fillna(False).astype(bool)
    _kg_no_green = float(out.loc[_no_green, "total_weight_kg"].sum())
    _kg_all = float(out["total_weight_kg"].sum())
    if _kg_no_green > 0 and _kg_all > 0:
        _cats = ", ".join(sorted(out.loc[_no_green, "category"].astype(str)))
        print(f"[impacts] Green water: no factor for {_kg_no_green:,.1f} kg "
              f"({100 * _kg_no_green / _kg_all:.1f}% of weight) -- {_cats}")

    # ---- Compute impacts (unit aware; see your factor definitions) ----
    # Metric-side (per kg factors)
    out["kcal"]         = out["kcal"] * out["total_weight_kg"]
    out["co2_kg"]       = out["co2"] * out["total_weight_kg"]
    out["carbon_opp_kg"]   = out["carbon_opp"] * out["total_weight_kg"]
    out["blue_water_liters"]  = out["blue_water"] * out["total_weight_kg"]
    out["green_water_liters"] = out["green_water"] * out["total_weight_kg"]
    out["land_sqm"]     = out["land_metric"] * out["total_weight_kg"]
    out["eutro_g"]      = out["eutro"] * out["total_weight_kg"]

    # Imperial-side conversions (all factors are metric; convert results)
    L_TO_GAL    = 0.264172     # liters → US gallons
    SQM_TO_SQFT = 10.7639      # m² → ft²
    G_PER_LB    = 453.59237    # grams per lb

    out["kcal_lbs"]       = out["kcal"]  # same scalar value, separate column for selection
    out["co2_lbs"]        = out["co2_kg"] * KG_TO_LB
    out["carbon_opp_lbs"] = out["carbon_opp_kg"] * KG_TO_LB
    out["blue_water_gal"]  = out["blue_water_liters"] * L_TO_GAL
    out["green_water_gal"] = out["green_water_liters"] * L_TO_GAL
    out["land_sqft"]      = out["land_sqm"] * SQM_TO_SQFT
    out["eutro_lb"]       = out["eutro_g"] / G_PER_LB

    # Lives (per kg factors). A life count is the same number in both output files.
    out["d_lives_imp"] = out["d_lives"] * out["total_weight_kg"]
    out["t_lives_imp"] = out["t_lives"] * out["total_weight_kg"]

    # ---- Build outputs (pre-format numeric frames) ----
    metric = out[[
        "category","total_weight_kg","kcal","co2_kg","carbon_opp_kg","d_lives_imp","t_lives_imp","blue_water_liters","green_water_liters","land_sqm","eutro_g"
    ]].rename(columns={"d_lives_imp":"d_lives","t_lives_imp":"t_lives"}).copy()

    lbs = out[[
        "category","total_weight_lbs","kcal_lbs","co2_lbs","carbon_opp_lbs","d_lives_imp","t_lives_imp","blue_water_gal","green_water_gal","land_sqft","eutro_lb"
    ]].rename(columns={"kcal_lbs":"kcal","d_lives_imp":"d_lives","t_lives_imp":"t_lives"}).copy()

    # ---- Add totals (Dairy & All) BEFORE formatting ----
    # Dairy set (normalized)
    DAIRY_CATEGORIES = {
        "butter","dairy cheese","cream","dairy milk","buttermilk","ice cream","low fat yogurt",
        "milk powder","yogurt","concentrated milk","ghee","lactose powder","skim milk","whey powder"
    }
    metric["__key"] = metric["category"].map(_norm)
    lbs["__key"]    = lbs["category"].map(_norm)

    is_dairy = metric["__key"].isin(DAIRY_CATEGORIES)

    def _sum_rows(df):
        return {
            "total_weight": pd.to_numeric(df.filter(like="total_weight").sum(numeric_only=True), errors="coerce").max(skipna=True),
            "kcal": df["kcal"].sum(skipna=True),
            "co2":  (df["co2_kg"].sum(skipna=True) if "co2_kg" in df.columns else df["co2_lbs"].sum(skipna=True)),
            "carbon_opp": (df["carbon_opp_kg"].sum(skipna=True) if "carbon_opp_kg" in df.columns else df["carbon_opp_lbs"].sum(skipna=True)), 
            "d_lives": df["d_lives"].sum(skipna=True),
            "t_lives": df["t_lives"].sum(skipna=True),
            "blue_water": df.filter(regex=r"^blue_water_").sum(numeric_only=True).max(skipna=True),
            "green_water": df.filter(regex=r"^green_water_").sum(numeric_only=True).max(skipna=True),
            "land": df.filter(regex=r"^land_").sum(numeric_only=True).max(skipna=True),
            "eutro": df.filter(regex=r"^eutro_").sum(numeric_only=True).max(skipna=True),
        }

    # Metric totals
    m_d = metric[is_dairy]
    m_a = metric
    m_d_s = _sum_rows(m_d)
    m_a_s = _sum_rows(m_a)

    metric_totals = pd.DataFrame([
        {"category":"TOTAL — Dairy",
         "total_weight_kg": m_d["total_weight_kg"].sum(skipna=True),
         "kcal": m_d_s["kcal"], "co2_kg": m_d_s["co2"], "carbon_opp_kg": m_d_s["carbon_opp"],
         "d_lives": m_d_s["d_lives"], "t_lives": m_d_s["t_lives"],
         "blue_water_liters": m_d_s["blue_water"], "green_water_liters": m_d_s["green_water"], "land_sqm": m_d_s["land"], "eutro_g": m_d_s["eutro"]},
        {"category":"TOTAL — All",
         "total_weight_kg": m_a["total_weight_kg"].sum(skipna=True),
         "kcal": m_a_s["kcal"], "co2_kg": m_a_s["co2"], "carbon_opp_kg": m_a_s["carbon_opp"],
         "d_lives": m_a_s["d_lives"], "t_lives": m_a_s["t_lives"],
         "blue_water_liters": m_a_s["blue_water"], "green_water_liters": m_a_s["green_water"], "land_sqm": m_a_s["land"], "eutro_g": m_a_s["eutro"]},
    ])

    metric = pd.concat([metric.drop(columns=["__key"]), metric_totals], ignore_index=True)

    # LBS totals
    l_d = lbs[lbs["__key"].isin(DAIRY_CATEGORIES)]
    l_a = lbs
    l_d_s = _sum_rows(l_d)
    l_a_s = _sum_rows(l_a)

    lbs_totals = pd.DataFrame([
        {"category":"TOTAL — Dairy",
         "total_weight_lbs": l_d["total_weight_lbs"].sum(skipna=True),
         "kcal": l_d_s["kcal"], "co2_lbs": l_d_s["co2"], "carbon_opp_lbs": l_d_s["carbon_opp"],
         "d_lives": l_d_s["d_lives"], "t_lives": l_d_s["t_lives"],
         "blue_water_gal": l_d_s["blue_water"], "green_water_gal": l_d_s["green_water"], "land_sqft": l_d_s["land"], "eutro_lb": l_d_s["eutro"]},
        {"category":"TOTAL — All",
         "total_weight_lbs": l_a["total_weight_lbs"].sum(skipna=True),
         "kcal": l_a_s["kcal"], "co2_lbs": l_a_s["co2"], "carbon_opp_lbs": l_a_s["carbon_opp"],
         "d_lives": l_a_s["d_lives"], "t_lives": l_a_s["t_lives"],
         "blue_water_gal": l_a_s["blue_water"], "green_water_gal": l_a_s["green_water"], "land_sqft": l_a_s["land"], "eutro_lb": l_a_s["eutro"]},
    ])

    lbs = pd.concat([lbs.drop(columns=["__key"]), lbs_totals], ignore_index=True)

    def _round_numeric_only(df: pd.DataFrame, decimals: int = 6) -> pd.DataFrame:
        """Return a copy of df with numeric columns rounded; non-numeric columns unchanged."""
        df_rounded = df.copy()
        numeric_cols = df_rounded.select_dtypes(include=["number"]).columns
        df_rounded[numeric_cols] = df_rounded[numeric_cols].round(decimals)
        return df_rounded
    
    metric = _round_numeric_only(metric, 6)
    lbs    = _round_numeric_only(lbs,    6)

    # ---- Strict formatting (write numbers as strings with exact decimals) ----
    metric_fmt = {
        "total_weight_kg": "{:.1f}",
        "kcal":            "{:.0f}",
        "co2_kg":          "{:.0f}",
        "carbon_opp_kg":      "{:.0f}",
        "d_lives":         "{:.1f}",
        "t_lives":         "{:.1f}",
        "blue_water_liters":  "{:.0f}",
        "green_water_liters": "{:.0f}",
        "land_sqm":        "{:.0f}",
        "eutro_g":         "{:.0f}",
    }
    lbs_fmt = {
        "total_weight_lbs": "{:.1f}",
        "kcal":             "{:.0f}",
        "co2_lbs":          "{:.0f}",
        "carbon_opp_lbs":   "{:.0f}",
        "d_lives":          "{:.1f}",
        "t_lives":          "{:.1f}",
        "blue_water_gal":   "{:.0f}",
        "green_water_gal":  "{:.0f}",
        "land_sqft":        "{:.0f}",
        "eutro_lb":         "{:.0f}",
    }

    def _apply_fmt_strict(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
        out = df.copy()
        for col, fmt in spec.items():
            if col in out.columns:
                vals = pd.to_numeric(out[col], errors="coerce")
                out[col] = vals.map(lambda x: "" if pd.isna(x) else fmt.format(x)).astype("object")
        return out

    metric_cols = ["category","total_weight_kg","kcal","co2_kg","carbon_opp_kg","d_lives","t_lives","blue_water_liters","green_water_liters","land_sqm","eutro_g"]
    lbs_cols    = ["category","total_weight_lbs","kcal","co2_lbs","carbon_opp_lbs","d_lives","t_lives","blue_water_gal","green_water_gal","land_sqft","eutro_lb"]

    metric = metric[metric_cols]
    lbs    = lbs[lbs_cols]

    metric_out = _apply_fmt_strict(metric, metric_fmt)
    lbs_out    = _apply_fmt_strict(lbs,    lbs_fmt)

    which = str(globals().get("OUTPUT_FILES", "both")).lower()
    written = []
    if which in ("both", "metric"):
        metric_out.to_csv(result_out(f"impacts_metric_{base}.csv"), index=False, encoding="utf-8")
        written.append(f"impacts_metric_{base}.csv")
    if which in ("both", "imperial"):
        lbs_out.to_csv(result_out(f"impacts_lbs_{base}.csv"), index=False, encoding="utf-8")
        written.append(f"impacts_lbs_{base}.csv")
    print(f"[impacts] wrote {' and '.join(written)} in {result_out('x').parent}")

    return lbs_out, metric_out


# --- If your runner calls this function, keep it as-is ---
def run_stage_3(base_filename: str, factors_filename: str):
    cats_path = result_in(f"categories_{base_filename}.csv")
    df_cat = pd.read_csv(cats_path)
    df_cat, n = cooked_split(df_cat)
    if n:
        print(f"[impacts] {n} canned or cooked bean, rice or pasta, broth or soup base row(s) or product(s) of "
              f"{cats_path.name} adjusted for this run (the file is from an older version; Stage 2 writes the "
              f"adjustment)")
    return calculate_impacts(df_cat, factors_filename, base_filename)


In [ ]:
# === Cell 8b — Shares of purchased weight: certified, plant-based, local, social impact ===
# Purpose: Report what share of total purchased weight is certified, plant-based, local, or bought from social
#   impact suppliers, scored for one of two systems, chosen by the worksheet's headings:
#   - Anchors in Action (AW / Env / Labor columns): AW, Env and Labor shares, weight by GFPP level (AW by food
#     group), Local by GFPP level, social / economic impact and POC suppliers as shares of local food, no
#     routine antibiotics (chicken; other animal products), and GFPP targets for the workbook's Program year
#   - AASHE STARS OP-07 v3.0 (AASHE Organic / Animal Welfare / Environment / Labor / Small Producers / Seafood
#     columns): the 7.1 sustainably or ethically produced share and the share for each AASHE category, the 7.2
#     social impact supplier share, estimated points, and an inventory of qualifying purchases
#   Plant-based and social impact shares are reported for both.
# Inputs:
#   - categories_{BASE}.csv (Stage 2; edits made there are used): weights, categories, and the product
#     columns written by Stage 1 -- plant_based, pb_basis, drink_water and, when the worksheet had them,
#     aw_food_group, aw, env, antibiotics, labor, the six aashe_* columns, local, social_impact, explanation
#   - certifiers.ods (CERTIFIERS_FILES in Cell 1): AIA sheet (Code | Certifier | Food group | GFPP Level |
#     No routine antibiotics) for Anchors in Action; STARS sheet (Section | Dropdown option | STARS name | ...)
#     for AASHE STARS
#   - Anchors in Action: the input worksheet's Program year ("Year N") and Nutrition checklist points (%)
#     (0-100) columns -- one value per file, chosen in the first data row
# Outputs (output folder):
#   - shares_{BASE}.csv: one row per measure with weight (kg and lb), % of total, calories (kcal), % of
#     total calories and number of products
#     (AASHE STARS: also estimated points)
#   - gfpp_levels_{BASE}.csv (Anchors in Action): AW (by food group) / Env / Labor / Local weight by GFPP level
#   - gfpp_targets_{BASE}.csv (Anchors in Action): shares with a GFPP target, and the target for the Program year
#   - stars_inventory_{BASE}.csv (AASHE STARS): the products that count for 7.1
#   - product_shares_{BASE}.csv: every product with its weight, calories and attributes
#   - shares_report_{BASE}.txt: those tables, how unclear cases were handled, and lists to review
# Rules:
#   - A product that has an attribute counts with its whole weight (no partial credit), once per measure.
#   - Calories (for information; neither system uses them): each ingredient's purchased weight x its
#     category's kcal per kg in factors.csv; canned / cooked beans, rice and pasta as dry equivalent + water
#     (cooked_split, Cell 2). The report also shows plant-based without cooking oils and impacts per 1,000
#     kcal (from impacts_*_{BASE}.csv).
#   - Total weight leaves out the water in bottled water and diluted drinks such as soda (rows in the
#     'water' category of products with drink_water = y); water in soups, juices, plant milks etc. stays in.
#   - Plant-based products have a processing category 1-4 (PLANT_CATEGORIES, Cell 4); the share is reported
#     for categories 1, 1-2, 1-3 and 1-4, and "Plant-based" means categories 1 to PLANT_BASED_MAX_CATEGORY
#     (Cell 1). A worksheet y counts in every category.
#   - A certification entry of "Other" counts and is listed with its Explanation. Seafood Watch: Avoid does not.
#   - Anchors in Action: worksheet entries read "Level N: <name>". The level is looked up in the AIA sheet
#     (AW by the worksheet's AW food group, Env / Labor / Local by name); an entry that is not on the list, or
#     an AW entry with no food group, is not scored and is listed. Fair Trade International counts only for
#     coffee, sugar and cocoa (GFPP Standards 3.0 p. 28). No routine antibiotics: y-cert is worked out again
#     from the AW / Env certifier (AIA column E); a typed y counts; only animal products count (chicken, and
#     other meat, dairy and eggs, each a share of its own purchases). Social / economic impact and POC
#     suppliers count as shares of local food (p. 18).
#   - AASHE STARS: a user fills in every AASHE column that applies. An entry qualifies if it is on that
#     column's list (the STARS sheet rows in its Section plus the rows STARS marks "Yes" for its category, as
#     add_certifier_dropdowns.ipynb builds the dropdowns) or is "Other". A product with any qualifying entry
#     counts for 7.1 once; each category's share is reported too (a product can count in several). Entries
#     not on the column's list, and Seafood Watch: Avoid, do not count and are listed. STARS v3.0 gives no credit for proximity alone, so a Local Credit column
#     (STARS_INCLUDE_LOCAL in add_certifier_dropdowns.ipynb) is reported for tracking only, with no points.

SHARES_DAIRY = {"butter", "dairy cheese", "cream", "dairy milk", "buttermilk", "ice cream", "low fat yogurt",
                "milk powder", "yogurt", "concentrated milk", "ghee", "lactose powder", "skim milk", "whey powder"}
AW_FOOD_GROUPS = {"Dairy": SHARES_DAIRY, "Eggs": {"eggs"}, "Poultry": {"chicken", "turkey"},
                  "Beef": {"beef"}, "Pork": {"pork"}}
AW_FOOD_GROUP_NAME_RX = [("Poultry", re.compile(r"\b(?:chicken|turkey|poultry)\b")),
                         ("Beef", re.compile(r"\bbeef\b")), ("Pork", re.compile(r"\b(?:pork|ham|bacon)\b")),
                         ("Eggs", re.compile(r"\beggs?\b")),
                         ("Dairy", re.compile(r"\b(?:milk|cheese|yogh?urt|butter|cream|dairy)\b"))]
# Animal categories of factors.csv, used to cross-check products marked plant-based
SHARES_ANIMAL_CATEGORIES = SHARES_DAIRY | {"animal fats", "beef", "chicken", "eggs", "goat", "lamb/mutton", "pork",
                                           "turkey", "unidentified meat"}
_SHARES_FISH_CAT_RX = re.compile(
    r"\bfish|bivalve|anchov|salmon|tuna|cod\b|haddock|pollock|sardine|halibut|mahi|catfish|tilapia|trout|bass|"
    r"hake|sole|flounder|whiting|carp|conger|milkfish|mullet|sauries|smelt|amberjack|unagi|barramundi|jacks|"
    r"pangasius|shrimp|crab|lobster|octopus|squid|mackerel|herring")

AIA_ATTRIBUTES = {   # column in the categories file -> heading in the report
    "aw": "Animal welfare (AW) certified",
    "env": "Environmental (Env) certified",
    "labor": "Labor certified",
}
# Anchors in Action (GFPP Standards 3.0) worksheet entries and targets
_GFPP_LEVEL_RX = re.compile(r"^\s*Level\s+(\d+)\s*:\s*(.+?)\s*$", re.I)     # "Level 3: USDA Organic"
GFPP_AW_GROUPS = ("Dairy", "Eggs", "Poultry", "Beef", "Pork")
GFPP_FTI = "fair trade international"     # Labor: counts only for coffee, sugar and cocoa (p. 28, note 39)
_GFPP_FTI_NAME_RX = re.compile(r"\b(?:coffee|espresso|cappuccino|latte|mocha|sugars?|cocoa|cacao|chocolate)\b")
_GFPP_FTI_CATEGORIES = {"coffee", "cocoa", "sugars and sweeteners"}
GFPP_SOCIAL_IMPACT = {"social/economic impact supplier": "impact",
                      "poc supplier (also counts as social/economic impact)": "poc"}
# No routine antibiotics (p. 38) counts animal products: a product whose rows in these (land animal) categories
# are at least half its weight; it is a chicken product when chicken is at least half of that.
GFPP_ABX_ANIMAL_SHARE = 0.5
# GFPP targets in percent: {measure: {first program year the target applies: percent}}
GFPP_TARGETS = {
    "abx_chicken": {1: 40, 5: 75},        # p. 38: chicken 40% initially, 75% by Year 5
    "abx_other": {1: 15, 5: 30},          #        other animal products 15%, then 30%
    "impact_local": {3: 10, 5: 15},       # p. 18: social / economic impact suppliers, % of local food
    "poc_local": {3: 5, 5: 10},           #        POC suppliers, % of local food
    "nutrition": {1: 51, 5: 80},          # pp. 39-40: Community Health & Nutrition checklist points
}

def _gfpp_split(v: str):
    """(level or None, name) of a worksheet entry such as 'Level 3: USDA Organic'."""
    m = _GFPP_LEVEL_RX.match(v or "")
    return (int(m.group(1)), m.group(2)) if m else (None, (v or "").strip())

def _gfpp_target(name: str, year):
    """The GFPP target (percent) for a program year, or None (no year chosen, or no target yet that year)."""
    if year is None:
        return None
    ok = [y for y in GFPP_TARGETS[name] if y <= year]
    return GFPP_TARGETS[name][max(ok)] if ok else None

GFPP_SETTING_COLUMNS = {"program_year": "Program year",                       # whole-file values, chosen once
                         "nutrition_checklist_points": "Nutrition checklist points (%)"}   # in the first data row

def _gfpp_settings():
    """(program year or None, nutrition checklist points % or None, notes) from the input worksheet's last two
    columns, Program year ('Year N') and Nutrition checklist points (%) (0-100). Each is one value for the whole
    file, chosen in the first data row: the first non-blank value is used, and later rows holding a different
    value are noted. notes: lines for the report (why a value is missing; differing rows)."""
    p = Path(str(globals().get("input_filename", "")))
    if not p.exists():
        return None, None, [f"the input worksheet ({p.name or 'not set'}) was not found"]
    try:
        raw = read_input_csv(p, remember_format=False)
    except Exception as e:
        return None, None, [f"the input worksheet could not be read ({type(e).__name__})"]
    names = dict(zip(_normalize_colnames(raw.columns), raw.columns))
    # row numbers as the user sees them: the web upload page's Worksheet row column, else headings in row 1
    srows = pd.to_numeric(raw[names[SHEET_ROW_HEADING]], errors="coerce") if SHEET_ROW_HEADING in names else None
    def rown(i):
        return str(int(srows[i])) if srows is not None and pd.notna(srows[i]) else str(i + 2)
    found, notes = {}, []
    for key, heading in GFPP_SETTING_COLUMNS.items():
        if key not in names:
            notes.append(f"the worksheet has no {heading} column")
            continue
        col = raw[names[key]].map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else _txt(v))
        filled = col[col.ne("")]
        if filled.empty:
            notes.append(f"no {heading} was chosen (row {rown(raw.index[0]) if len(raw) else 2} of that column is blank)")
            continue
        first = filled.iloc[0]
        found[key] = first
        rows = [rown(i) for i, v in filled.items() if v.casefold() != first.casefold()]
        if rows:
            notes.append(f"{heading}: {'row' if len(rows) == 1 else 'rows'} {', '.join(rows[:20])}"
                         f"{' ...' if len(rows) > 20 else ''} {'holds' if len(rows) == 1 else 'hold'} a value "
                         f"other than '{first}' (row {rown(filled.index[0])}); '{first}' was used")
    m = re.search(r"\d+", found.get("program_year", ""))
    year = int(m.group()) if m else None
    if "program_year" in found and year is None:
        notes.append(f"Program year '{found['program_year']}' is not a year (choose 'Year 1' .. 'Year 25')")
    nut = pd.to_numeric(pd.Series([found.get("nutrition_checklist_points", "").rstrip("%")]), errors="coerce").iloc[0]
    nut = float(nut) if pd.notna(nut) and 0 <= nut <= 100 else None
    if "nutrition_checklist_points" in found and nut is None:
        notes.append(f"Nutrition checklist points '{found['nutrition_checklist_points']}' is not a number 0-100")
    return year, nut, notes
# AASHE STARS columns (as add_certifier_dropdowns.ipynb names them): column in the categories file ->
# (heading, STARS sheet Section, STARS sheet category column)
AASHE_COLUMNS = {
    "aashe_organic": ("AASHE Organic", "Organic", "Organic"),
    "aashe_animal_welfare": ("AASHE Animal Welfare", "Animal welfare", "Animal welfare"),
    "aashe_environment": ("AASHE Environment", "Environment", "Environment"),
    "aashe_labor": ("AASHE Labor", "Labor", "Labor"),
    "aashe_small_producers": ("AASHE Small Producers", "Small producers", "Small producers"),
    "aashe_seafood": ("AASHE Seafood", "Seafood", "Sustainable seafood"),
}
STARS_CAMPUS_FARM = ("Campus farm or garden", "aashe_small_producers")   # no Section on the STARS sheet
STARS_POINTS_FACTOR = 0.06      # OP-07 7.1, Tables II and III: points = percent x 0.06 (up to 6 in all)
STARS_SOCIAL_IMPACT_POINTS = [(10, 2), (5, 1)]   # OP-07 7.2: percent of dining spend -> points
_STARS_AVOID_RX = re.compile(r"\bavoid\b|does not qualify", re.I)

def _txt(v) -> str:
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v).strip()

def _pb_value(v) -> str:
    """A plant_based cell as '1'-'4', 'y', 'review' or '0' (blank and anything else = not plant-based)."""
    t = _txt(v).lower()
    if _is_yes(t):
        return "y"
    if t == "review":
        return "review"
    c = _as_category(t)
    return str(c) if c else "0"

def _kcal_per_kg() -> dict:
    """{category (lower case): kcal per kg} from factors.csv (blank = 0), or {} if it cannot be read."""
    try:
        fac = pd.read_csv(globals().get("factors_filename") or "factors.csv")
        k = pd.to_numeric(fac["kcal"], errors="coerce").fillna(0.0)
        return {str(c).strip().lower(): float(v) for c, v in zip(fac["category"], k)}
    except Exception:
        return {}

def _is_cooking_oil(cat: str) -> bool:
    """Plant oil categories of factors.csv ('... (oil)', vegetable oils)."""
    return cat.endswith("(oil)") or cat == "vegetable oils"
COOKING_OIL_SHARE = 0.9     # a product at least this share oil by weight counts as a cooking oil...
_COOKING_OIL_NAME_RX = re.compile(r"\b(?:oils?|margarine|shortening)\b")   # ...if its name says so (not olives)

def _impacts_per_1000_kcal(base: str) -> dict:
    """Stage 3 totals per 1,000 kcal (metric units), from impacts_metric_ or impacts_lbs_{base}.csv; {} if none."""
    for name, metric in ((f"impacts_metric_{base}.csv", True), (f"impacts_lbs_{base}.csv", False)):
        p = result_out(name)
        if not p.exists():
            continue
        try:
            df = pd.read_csv(p, encoding="utf-8")
            lab = df["category"].astype(str)
            t = df[lab.str.startswith("TOTAL") & lab.str.endswith("All")].iloc[-1]
            per = 1000.0 / float(t["kcal"])
        except Exception:
            continue
        if metric:
            vals = {"kg CO2e": t["co2_kg"], "m2 land": t["land_sqm"], "L blue water": t["blue_water_liters"],
                    "L green water": t["green_water_liters"]}
        else:
            vals = {"kg CO2e": t["co2_lbs"] * LB_TO_KG, "m2 land": t["land_sqft"] * 0.09290304,
                    "L blue water": t["blue_water_gal"] * 3.785411784, "L green water": t["green_water_gal"] * 3.785411784}
        return {k: float(v) * per for k, v in vals.items()}
    return {}

def _pb_counted(v: str, max_category: int) -> bool:
    """True if a product's plant_based value counts when categories 1..max_category are counted."""
    return v == "y" or (v.isdigit() and 1 <= int(v) <= max_category)

def _certifiers_sheet(sheet: str, min_cols: int, needs: str):
    """(DataFrame, source) for a sheet of the first CERTIFIERS_FILES file that exists, or (None, why)."""
    for p in globals().get("CERTIFIERS_FILES", []):
        p = Path(p)
        if not p.exists():
            continue
        try:
            df = pd.read_excel(p, sheet_name=sheet, engine="odf" if p.suffix.lower() == ".ods" else None)
        except Exception as e:
            return None, f"{p} could not be read ({type(e).__name__}: {e})"
        if df.shape[1] < min_cols:
            return None, f"{p}: the {sheet} sheet needs columns {needs}"
        return df, str(p)
    return None, "certifiers.ods not found (CERTIFIERS_FILES in Cell 1)"

def _load_gfpp_levels():
    """({(code, certifier, food group): level}, {(code, certifier) marked Yes in No routine antibiotics} or None,
    source) from the AIA sheet of certifiers.ods, or (None, None, why)."""
    aia, src = _certifiers_sheet("AIA", 4, "Code | Certifier | Food group | GFPP Level")
    if aia is None:
        return None, None, src
    abx_col = aia.iloc[:, 4].map(_txt).str.lower().eq("yes") if aia.shape[1] >= 5 else None
    aia = aia.iloc[:, :4].copy()
    aia.columns = ["code", "certifier", "group", "level"]
    aia["abx"] = abx_col if abx_col is not None else False
    aia = aia[aia["code"].notna() & pd.to_numeric(aia["level"], errors="coerce").notna()]
    table = {(_txt(r.code).lower(), _txt(r.certifier).casefold(), _txt(r.group).lower()): int(float(r.level))
             for r in aia.itertuples(index=False)}
    abx = ({(_txt(r.code).lower(), _txt(r.certifier).casefold()) for r in aia.itertuples(index=False) if r.abx}
           if abx_col is not None else None)
    return table, abx, src

def _load_stars_options():
    """({AASHE column: {option casefolded: STARS name}}, source) from the STARS sheet, or (None, why). Options
    are named as in the dropdowns: the Dropdown option without its "Section: " prefix."""
    st, src = _certifiers_sheet("STARS", 3, "Section | Dropdown option | STARS name | ... category columns")
    if st is None:
        return None, src
    st = st.astype(object).where(st.notna(), "").apply(lambda s: s.map(_txt))
    st.columns = [_txt(c) for c in st.columns]
    sec_col, opt_col, name_col = st.columns[:3]
    table = {col: {} for col in AASHE_COLUMNS}
    for _, r in st.iterrows():
        opt = r[opt_col].removeprefix(f"{r[sec_col]}: ") if r[sec_col] else r[opt_col]
        if not opt:
            continue
        for col, (_, section, category) in AASHE_COLUMNS.items():
            if r[sec_col] == section or (category in st.columns and r[category].lower() == "yes"):
                table[col][opt.casefold()] = r[name_col] or opt
    farm, farm_col = STARS_CAMPUS_FARM
    table[farm_col].setdefault(farm.casefold(), farm)
    return table, src

def _stars_points(pct: float) -> float:
    return round(min(max(pct, 0.0), 100.0) * STARS_POINTS_FACTOR, 2) if pd.notna(pct) else np.nan

def _social_impact_points(pct: float) -> float:
    if pd.isna(pct):
        return np.nan
    return next((p for floor, p in STARS_SOCIAL_IMPACT_POINTS if pct >= floor), 0)

def run_shares_report(base: str):
    cats_path = result_in(f"categories_{base}.csv")
    if not cats_path.exists():
        print(f"[shares] {cats_path.name} not found; no shares report.")
        return None
    cats = pd.read_csv(cats_path)
    cats, _ = cooked_split(cats)    # canned / cooked beans, rice, pasta (a no-op for files Stage 2 has split)
    cols = set(cats.columns)

    # Weights (as Stage 3): ingredient weight, else weight, else product weight x percent
    suf = "kg" if any(f"{c}_kg" in cols for c in ("ingredient_weight", "weight", "product_weight")) else "lbs"
    w = None
    for c in (f"ingredient_weight_{suf}", f"weight_{suf}"):
        if c in cols and pd.to_numeric(cats[c], errors="coerce").notna().any():
            w = pd.to_numeric(cats[c], errors="coerce")
            break
    if w is None and f"product_weight_{suf}" in cols and "percent" in cols:
        w = pd.to_numeric(cats[f"product_weight_{suf}"], errors="coerce") * pd.to_numeric(cats["percent"], errors="coerce") / 100
    if w is None:
        print(f"[shares] {cats_path.name} has no usable weights; no shares report.")
        return None
    cats["__kg"] = w.fillna(0.0) * (1.0 if suf == "kg" else LB_TO_KG)
    cats["__cat"] = cats["category"].map(_txt).str.lower() if "category" in cols else ""
    drink = cats["drink_water"].map(_is_yes) if "drink_water" in cols else pd.Series(False, index=cats.index)
    cats["__drop"] = cats["__cat"].eq("water") & drink
    if "match_method" in cols:      # water from the canned / cooked / broth split is food, never drink water
        cats["__drop"] &= ~cats["match_method"].astype(str).isin(["rule_cooked", "rule_broth", "rule_base"])
    kcal_map = _kcal_per_kg()
    have_kcal = bool(kcal_map)
    cats["__kcal"] = cats["__kg"] * cats["__cat"].map(kcal_map).fillna(0.0) if have_kcal else 0.0
    cats["__oil"] = cats["__cat"].map(_is_cooking_oil)

    # Which system: Anchors in Action (AW / Env / Labor columns) or AASHE STARS (AASHE columns)
    has_aia, has_stars = any(a in cols for a in AIA_ATTRIBUTES), any(a in cols for a in AASHE_COLUMNS)
    system = "aia" if has_aia and not has_stars else "stars" if has_stars and not has_aia else ""
    aashe_cols = [a for a in AASHE_COLUMNS if a in cols]
    if system == "aia":
        system_line = "Scored for: Anchors in Action (GFPP) -- the worksheet has AW / Env / Labor columns"
    elif system == "stars":
        system_line = ("Scored for: AASHE STARS OP-07 v3.0 -- the worksheet has AASHE columns ("
                       + ", ".join(AASHE_COLUMNS[a][0] for a in aashe_cols) + ")")
    elif has_aia:
        system_line = ("No certification shares: the worksheet has both AW / Env / Labor and AASHE STARS columns, so "
                       "it is not clear which system to score. Make the worksheet again with one PREP_TYPE.")
    else:
        system_line = ("No certification shares: the worksheet has no AW / Env / Labor or AASHE STARS column "
                       "(make it with add_certifier_dropdowns.ipynb). Plant-based is still reported.")

    levels, abx_list, levels_src = _load_gfpp_levels() if system == "aia" else (None, None, "")
    year, nutrition, setting_notes = _gfpp_settings() if system == "aia" else (None, None, [])
    options, options_src = _load_stars_options() if system == "stars" else (None, "")
    max_cat = _as_category(globals().get("PLANT_BASED_MAX_CATEGORY", 3)) or 3
    have = {k: k in cols for k in list(AIA_ATTRIBUTES) + list(AASHE_COLUMNS) + [
        "aw_food_group", "antibiotics", "local", "social_impact", "plant_based", "explanation"]}

    def _level(code: str, entry: str, group: str) -> str:
        """GFPP level of a worksheet entry: 'Level N' or 'Other (no GFPP level)' (both count), or
        'not scored: ...' (does not count)."""
        if not entry:
            return ""
        shown, name = _gfpp_split(entry)
        if name.casefold() == "other":
            return "Other (no GFPP level)"
        if code == "aw" and not group:
            return "not scored: no AW food group"
        if levels is None:
            return (f"Level {shown} (from the worksheet; certifiers.ods not read)" if shown
                    else "counted; level not looked up (certifiers.ods not read)")
        lv = levels.get((code, name.casefold(), (group or "all").lower()))
        if lv is None:
            return f"not scored: not on the {group} AW list" if code == "aw" else "not scored: not on the list"
        return f"Level {lv}" + (f" (the worksheet said Level {shown})" if shown and shown != lv else "")
    _scored = lambda status: bool(status) and not status.startswith("not scored")

    def _stars_status(col: str, choice: str):
        """(status, STARS name) of an entry in an AASHE column; status 'qualifies...' or 'Other' counts."""
        if not choice:
            return "", ""
        if choice.casefold() == "other":
            return "Other", ""
        if _STARS_AVOID_RX.search(choice):
            return "does not qualify (Seafood Watch: Avoid)", ""
        if options is None:
            return "qualifies (STARS list not checked)", ""
        if choice.casefold() not in options[col]:
            return f"not on the {AASHE_COLUMNS[col][0]} list", ""
        return "qualifies", options[col][choice.casefold()]
    _counts = lambda status: status.startswith("qualifies") or status == "Other"

    key = ["row_sheet", "product"] if "row_sheet" in cols else ["product"]
    recs = []
    for k, g in cats.groupby(key, sort=False, dropna=False):
        first = g.iloc[0]
        kept = g[~g["__drop"]]
        kg = float(kept["__kg"].sum())
        # A product's rows should agree; if edits left them different, the most conservative value is used
        pb_vals = [_pb_value(v) for v in g["plant_based"]] if have["plant_based"] else ["0"]
        if len(set(pb_vals)) == 1:
            pb = pb_vals[0]
        elif "0" in pb_vals or "review" in pb_vals:
            pb = "0" if "0" in pb_vals else "review"
        else:
            pb = max((v for v in pb_vals if v.isdigit()), default="y")
        animal_cats = sorted({c for c in g["__cat"] if c in SHARES_ANIMAL_CATEGORIES or _SHARES_FISH_CAT_RX.search(c)})
        r = {
            "row_sheet": first["row_sheet"] if "row_sheet" in cols else "",
            "product": first["product"],
            "weight_kg": kg,
            "weight_lbs": kg * KG_TO_LB,
            "water_left_out_kg": float(g.loc[g["__drop"], "__kg"].sum()),
            "kcal": float(kept["__kcal"].sum()) if have_kcal else np.nan,
            "kcal_per_kg": float(kept["__kcal"].sum()) / kg if have_kcal and kg else np.nan,
            "cooking_oil": bool(kg) and float(kept.loc[kept["__oil"], "__kg"].sum()) >= COOKING_OIL_SHARE * kg
                           and bool(_COOKING_OIL_NAME_RX.search(norm_text(first["product"]))),
            "plant_based": pb,
            "plant_category": (PLANT_CATEGORIES[int(pb)] if pb.isdigit() and int(pb) in PLANT_CATEGORIES
                               else "marked y in the worksheet (every category)" if pb == "y" else ""),
            "pb_basis": _txt(first.get("pb_basis", "")),
            "pb_rows_differ": len(set(pb_vals)) > 1,
            "animal_categories": ", ".join(animal_cats),
        }
        for a in list(AIA_ATTRIBUTES) + list(AASHE_COLUMNS) + ["aw_food_group", "antibiotics", "local",
                                                                "social_impact", "explanation"]:
            r[a] = _txt(first[a]) if have[a] else ""
        if system == "aia":
            # Food group found from the product's rows (the group with the most weight), else from its name
            grp_w = {name: kept.loc[kept["__cat"].isin(cs), "__kg"].sum() for name, cs in AW_FOOD_GROUPS.items()}
            detected = max(grp_w, key=grp_w.get) if max(grp_w.values()) > 0 else ""
            if not detected and not _pb_counted(pb, 4):     # the name is no guide for "MILK SUB, SOY"
                nm = norm_text(first["product"])
                detected = next((gname for gname, rx in AW_FOOD_GROUP_NAME_RX if rx.search(nm)), "")
            r["aw_food_group_detected"] = detected
            # AW is scored with the worksheet's AW food group (the found group for worksheets without that column)
            if have["aw_food_group"]:
                group = next((gn for gn in GFPP_AW_GROUPS if gn.casefold() == r["aw_food_group"].casefold()), "")
            else:
                group = detected
            r["aw_level"] = _level("aw", r["aw"], group)
            if have["aw_food_group"] and r["aw_food_group"] and not group:
                r["aw_level"] = "not scored: unknown AW food group" if r["aw"] else ""
            r["aw_group_used"] = group or detected
            r["env_level"] = _level("env", r["env"], "All")
            r["labor_level"] = _level("labor", r["labor"], "All")
            # Fair Trade International: coffee, sugar or cocoa in the name, or as the product's main ingredient
            main_cat = g.groupby("__cat")["__kg"].sum().idxmax() if len(g) else ""
            fti_ok = bool(_GFPP_FTI_NAME_RX.search(norm_text(first["product"])) or main_cat in _GFPP_FTI_CATEGORIES)
            if _gfpp_split(r["labor"])[1].casefold() == GFPP_FTI and _scored(r["labor_level"]) and not fti_ok:
                r["labor_level"] = "not scored: Fair Trade International counts only for coffee, sugar and cocoa"
            r["aw_counts"], r["env_counts"], r["labor_counts"] = (_scored(r["aw_level"]), _scored(r["env_level"]),
                                                                  _scored(r["labor_level"]))
            certified = r["aw_counts"] or r["env_counts"] or r["labor_counts"]

            # No routine antibiotics: animal products only (chicken / other), y-cert worked out again
            animal_kg = float(kept.loc[kept["__cat"].isin(SHARES_ANIMAL_CATEGORIES), "__kg"].sum())
            chicken_kg = float(kept.loc[kept["__cat"].eq("chicken"), "__kg"].sum())
            r["animal_group"] = ("" if not kg or animal_kg < GFPP_ABX_ANIMAL_SHARE * kg else
                                 "chicken" if chicken_kg >= 0.5 * animal_kg else "other animal product")
            r["animal_share"] = animal_kg / kg if kg else 0.0
            cert_names = [("aw", _gfpp_split(r["aw"])[1]), ("env", _gfpp_split(r["env"])[1])]
            if abx_list is not None:
                by_cert = any(n and (c, n.casefold()) in abx_list for c, n in cert_names)
            else:
                by_cert = r["antibiotics"].casefold() == "y-cert"
            typed = r["antibiotics"].casefold()
            r["antibiotics_status"] = ("y-cert" if by_cert else "y (other verification)" if _is_yes(typed)
                                       else "" if typed in ("", "y-cert") else "not recognized")
            r["antibiotics_cached_differs"] = typed == "y-cert" and not by_cert
            r["antibiotics_counted"] = r["antibiotics_status"] in ("y-cert", "y (other verification)") \
                and bool(r["animal_group"])

            # Local by GFPP level; social / economic impact suppliers count as a share of local food
            lv_local, _ = _gfpp_split(r["local"])
            r["local_level"] = (f"Level {lv_local}" if lv_local else "y (no GFPP level)" if _is_yes(r["local"])
                                else "not recognized" if r["local"] else "")
            r["local_counts"] = r["local_level"].startswith(("Level", "y"))
            kind = GFPP_SOCIAL_IMPACT.get(r["social_impact"].casefold(),
                                          "impact" if _is_yes(r["social_impact"]) else
                                          "not recognized" if r["social_impact"] else "")
            r["social_impact_kind"] = {"impact": "social / economic impact", "poc": "POC (also social / economic "
                                       "impact)"}.get(kind, kind)
            r["social_impact_counted"] = kind in ("impact", "poc") and r["local_counts"]
            r["poc_counted"] = kind == "poc" and r["local_counts"]
        elif system == "stars":
            # Every filled AASHE column: categories and STARS names that count; entries that do not
            cats_ok, names, other, problems = [], [], [], []
            for a in aashe_cols:
                status, name = _stars_status(a, r[a])
                label = AASHE_COLUMNS[a][0].removeprefix("AASHE ")
                r[f"{a}_counts"] = _counts(status)
                if _counts(status):
                    cats_ok.append(label)
                    if status == "Other":
                        other.append(label)
                    elif name and name not in names:
                        names.append(name)
                elif status:
                    problems.append(f"{label}: {r[a]} ({status})")
            r["stars_categories"] = "; ".join(cats_ok)
            r["stars_names"] = "; ".join(names + [f"Other ({o})" for o in other])
            r["stars_other"] = "; ".join(other)
            r["stars_not_counted"] = "; ".join(problems)
            certified = bool(cats_ok)
        else:
            certified = False
        r["certified"] = "y" if certified else ""
        r["certified_or_plant_based"] = "y" if certified or _pb_counted(pb, max_cat) else ""
        recs.append(r)
    prod = pd.DataFrame(recs)
    total = prod["weight_kg"].sum()
    total_kcal = prod["kcal"].sum() if have_kcal else 0.0

    def _measure(label, mask, available=True, missing="column not in the worksheet"):
        if not available:
            return {"measure": label, "weight_kg": np.nan, "weight_lbs": np.nan, "percent_of_total": np.nan,
                    "products": np.nan, "kcal": np.nan, "percent_of_kcal": np.nan, "note": missing}
        kg = prod.loc[mask, "weight_kg"].sum()
        kc = prod.loc[mask, "kcal"].sum() if have_kcal else np.nan
        return {"measure": label, "weight_kg": kg, "weight_lbs": kg * KG_TO_LB,
                "percent_of_total": 100 * kg / total if total else np.nan, "products": int(mask.sum()),
                "kcal": kc, "percent_of_kcal": 100 * kc / total_kcal if total_kcal else np.nan, "note": ""}

    nonblank = lambda c: prod[c].ne("")
    old_files = "not judged: files from an older version; re-run from Stage 1"
    pb_mask = prod["plant_based"].map(lambda v: _pb_counted(v, max_cat))
    adds = {1: "category 1: whole or minimally processed", 2: "categories 1-2: + cooking ingredients",
            3: "categories 1-3: + simple processed and alternatives", 4: "categories 1-4: + ultra-processed"}
    pb_label = f"Plant-based (categories 1-{max_cat}, the PLANT_BASED_MAX_CATEGORY setting)"

    rows = [_measure("Total purchased weight (without water in bottled water and drinks)", prod["weight_kg"].notna())]
    if system == "aia":
        rows += [_measure(lbl, prod[f"{a}_counts"].astype(bool), have[a]) for a, lbl in AIA_ATTRIBUTES.items()]
    elif system == "stars":
        rows.append(_measure("Sustainably or ethically produced (OP-07 7.1, Table II; each product once)",
                             prod["certified"].eq("y")))
        for a in aashe_cols:
            rows.append(_measure(f"    {AASHE_COLUMNS[a][0]} (a product may count in several categories)",
                                 prod[f"{a}_counts"].astype(bool)))
        pb_label = f"Plant-based (OP-07 7.1, Table III; categories 1-{max_cat})"
    rows.append(_measure(pb_label, pb_mask, have["plant_based"], old_files))
    for k in PLANT_CATEGORIES:
        rows.append(_measure(f"    plant-based, {adds[k]}",
                             prod["plant_based"].map(lambda v, k=k: _pb_counted(v, k)), have["plant_based"], old_files))
    if system == "aia":
        rows.append(_measure("Certified (AW, Env or Labor) or plant-based, each product once",
                             prod["certified_or_plant_based"].eq("y")))
        rows.append(_measure("Local (any GFPP level; levels in the table below)", prod["local_counts"].astype(bool),
                             have["local"]))
    elif system == "stars" and have["local"]:
        rows.append(_measure("Local (tracked only; not part of the STARS v3.0 score)", prod["local"].map(_is_yes)))
        rows[-1]["note"] = "no STARS points for proximity alone"
    elif system == "" and have["local"]:
        rows.append(_measure("Local", prod["local"].map(_is_yes)))
    si_label = "Social impact supplier (OP-07 7.2)" if system == "stars" else "Social impact supplier"
    targets = pd.DataFrame()
    if system == "aia":
        # Shares with a GFPP target are of a part of the purchases (local food; chicken; other animal products)
        def _part(label, mask, base_mask, base_label, available, target, missing="column not in the worksheet"):
            m = _measure(label, mask & base_mask, available, missing)
            base_kg = prod.loc[base_mask, "weight_kg"].sum()
            pct = 100 * m["weight_kg"] / base_kg if available and base_kg else np.nan
            if available:
                m["note"] = (f"{pct:.1f}% of {base_label} ({base_kg:,.1f} kg)" if base_kg
                             else f"no {base_label} in the purchases")
            t = _gfpp_target(target, year)
            tgt.append({"measure": label.strip(), "percent": pct, "of": base_label,
                        "target_initial": GFPP_TARGETS[target].get(1, np.nan),
                        "target_year_3": GFPP_TARGETS[target].get(3, GFPP_TARGETS[target].get(1, np.nan)),
                        "target_year_5": GFPP_TARGETS[target][5], "program_year": year,
                        "target_this_year": t if t is not None else np.nan,
                        "meets_target": ("" if t is None or pd.isna(pct) else "yes" if pct >= t else "no")})
            return m
        tgt = []
        local_mask = prod["local_counts"].astype(bool)
        rows.append(_part("Social / economic impact suppliers (POC included)",
                          prod["social_impact_counted"].astype(bool), local_mask, "local food", have["social_impact"]
                          and have["local"], "impact_local",
                          "Social Impact Supplier or Local (GFPP level) column not in the worksheet"))
        rows.append(_part("    POC suppliers", prod["poc_counted"].astype(bool), local_mask, "local food",
                          have["social_impact"] and have["local"], "poc_local",
                          "Social Impact Supplier or Local (GFPP level) column not in the worksheet"))
        rows.append(_part("No routine antibiotics: chicken", prod["antibiotics_counted"].astype(bool),
                          prod["animal_group"].eq("chicken"), "chicken purchases", have["antibiotics"], "abx_chicken"))
        rows.append(_part("No routine antibiotics: other meat, dairy and eggs", prod["antibiotics_counted"].astype(bool),
                          prod["animal_group"].eq("other animal product"), "other meat, dairy and egg purchases",
                          have["antibiotics"], "abx_other"))
        t = _gfpp_target("nutrition", year)
        tgt.append({"measure": "Community Health & Nutrition checklist points", "percent": nutrition,
                    "of": "checklist points (worksheet column)", "target_initial": 51, "target_year_3": 51,
                    "target_year_5": 80, "program_year": year, "target_this_year": t if t is not None else np.nan,
                    "meets_target": "" if t is None or nutrition is None else "yes" if nutrition >= t else "no"})
        targets = pd.DataFrame(tgt)
        targets["percent"] = pd.to_numeric(targets["percent"], errors="coerce").round(1)
        targets["program_year"] = targets["program_year"].astype("Int64")
    else:
        rows.append(_measure(si_label, prod["social_impact"].map(_is_yes), have["social_impact"]))
    shares = pd.DataFrame(rows)
    shares["products"] = shares["products"].astype("Int64")

    points = {}
    if system == "stars":
        pct = dict(zip(shares["measure"], shares["percent_of_total"]))
        sust = _stars_points(pct[rows[1]["measure"]])
        plant = _stars_points(pct[pb_label])
        p71 = min(6.0, (0 if pd.isna(sust) else sust) + (0 if pd.isna(plant) else plant))
        p72 = _social_impact_points(pct[si_label])
        points = {"7.1 sustainably or ethically produced (Table II)": sust, "7.1 plant-based (Table III)": plant,
                  "7.1 total (Table IV, up to 6)": p71, "7.2 social impact suppliers (up to 2)": p72,
                  "OP-07 total (up to 8)": p71 + (0 if pd.isna(p72) else p72)}
        shares["estimated_points"] = shares["measure"].map({rows[1]["measure"]: sust, pb_label: plant,
                                                            si_label: p72})
        shares = shares[["measure", "weight_kg", "weight_lbs", "percent_of_total", "products", "kcal",
                         "percent_of_kcal", "estimated_points", "note"]]

    lvl = pd.DataFrame(columns=["certification", "food_group", "gfpp_level", "weight_kg", "weight_lbs",
                                "percent_of_total", "percent_of_food_group", "products"])
    if system == "aia":
        lvl_rows = []
        _as_tuple = lambda k: k if isinstance(k, tuple) else (k,)
        _lv_order = lambda x: (_as_tuple(x[0])[0], not _as_tuple(x[0])[-1].startswith("Level"), _as_tuple(x[0])[-1])
        for code, label in (("aw", "AW"), ("env", "Env"), ("labor", "Labor"), ("local", "Local")):
            if not have[code]:
                continue
            sub = prod[prod[code].ne("")]
            by = ["aw_group_used", "aw_level"] if code == "aw" else [f"{code}_level"]
            for key_, g in sorted(sub.groupby(by), key=_lv_order):
                key_ = _as_tuple(key_)
                grp, lv = (key_[0], key_[1]) if code == "aw" else ("", key_[0])
                kg = g["weight_kg"].sum()
                grp_kg = prod.loc[prod["aw_group_used"].eq(grp), "weight_kg"].sum() if grp else 0
                lvl_rows.append({"certification": label, "food_group": grp, "gfpp_level": lv, "weight_kg": kg,
                                 "weight_lbs": kg * KG_TO_LB, "percent_of_total": 100 * kg / total if total else np.nan,
                                 "percent_of_food_group": 100 * kg / grp_kg if grp_kg else np.nan, "products": len(g)})
        lvl = pd.DataFrame(lvl_rows, columns=lvl.columns)
        lvl["percent_of_food_group"] = pd.to_numeric(lvl["percent_of_food_group"], errors="coerce").round(1)

    # ---- Write the tables ----
    def _rounded(df):
        out = df.copy()
        for c in ("weight_kg", "weight_lbs", "water_left_out_kg", "percent_of_total", "percent_of_kcal"):
            if c in out.columns:
                out[c] = out[c].round(1)
        for c in ("kcal", "kcal_per_kg"):
            if c in out.columns:
                out[c] = out[c].round(0)
        return out
    shares_out, lvl_out = _rounded(shares), _rounded(lvl)
    pb_cols = ["plant_based", "plant_category", "pb_basis"]
    if system == "aia":
        prod_cols = (["row_sheet", "product", "weight_kg", "weight_lbs", "water_left_out_kg", "kcal", "kcal_per_kg", "cooking_oil", "certified_or_plant_based",
                      "aw_food_group", "aw_food_group_detected", "aw", "aw_level", "env", "env_level", "labor",
                      "labor_level", "antibiotics", "antibiotics_status", "animal_group", "antibiotics_counted"]
                     + pb_cols + ["local", "local_level", "social_impact", "social_impact_kind",
                                  "social_impact_counted", "explanation"])
    elif system == "stars":
        prod_cols = (["row_sheet", "product", "weight_kg", "weight_lbs", "water_left_out_kg", "kcal", "kcal_per_kg", "cooking_oil", "certified",
                      "stars_categories", "stars_names"] + aashe_cols + ["stars_not_counted"] + pb_cols
                     + (["local"] if have["local"] else []) + ["social_impact", "explanation"])
    else:
        prod_cols = (["row_sheet", "product", "weight_kg", "weight_lbs", "water_left_out_kg", "kcal", "kcal_per_kg", "cooking_oil"] + pb_cols
                     + [c for c in ("local", "social_impact", "explanation") if have[c]])
    prod_out = _rounded(prod)[prod_cols]
    prod_out["cooking_oil"] = prod_out["cooking_oil"].map(lambda v: "y" if v else "")
    if system == "aia":
        for c in ("antibiotics_counted", "social_impact_counted"):
            prod_out[c] = prod_out[c].map(lambda v: "y" if v else "")
    shares_out.to_csv(result_out(f"shares_{base}.csv"), index=False, encoding="utf-8")
    prod_out.to_csv(result_out(f"product_shares_{base}.csv"), index=False, encoding="utf-8")
    written = [f"shares_{base}.csv", f"product_shares_{base}.csv"]
    if system == "aia":
        lvl_out.to_csv(result_out(f"gfpp_levels_{base}.csv"), index=False, encoding="utf-8")
        targets.to_csv(result_out(f"gfpp_targets_{base}.csv"), index=False, encoding="utf-8")
        written += [f"gfpp_levels_{base}.csv", f"gfpp_targets_{base}.csv"]
    inv = pd.DataFrame()
    if system == "stars":
        inv = _rounded(prod[prod["certified"].eq("y") | pb_mask]).copy()
        inv["sustainable_or_ethical"] = inv["certified"]
        inv["plant_based_counted"] = inv["plant_based"].map(lambda v: "y" if _pb_counted(v, max_cat) else "")
        inv = inv[["row_sheet", "product", "weight_kg", "weight_lbs", "sustainable_or_ethical", "stars_categories",
                   "stars_names"] + aashe_cols + ["plant_based_counted", "plant_based", "plant_category",
                                                  "social_impact", "explanation"]]
        inv.to_csv(result_out(f"stars_inventory_{base}.csv"), index=False, encoding="utf-8")
        written.append(f"stars_inventory_{base}.csv")

    # ---- Text report ----
    unit = "kg" if suf == "kg" else "lb"
    wcol = "weight_kg" if suf == "kg" else "weight_lbs"
    def _list(title, df, extra=()):
        if df.empty:
            return []
        out = ["", f"{title} ({len(df)})"]
        for _, r in df.iterrows():
            parts = [f"row {r['row_sheet']}" if _txt(r["row_sheet"]) else "", str(r["product"]),
                     f"{r[wcol]:,.1f} {unit}"] + [f"{lbl}: {r[c]}" for c, lbl in extra if _txt(r[c])]
            out.append("  - " + " | ".join(p for p in parts if p))
        return out

    show_cols = ["measure", wcol, "percent_of_total", "products"] + (["estimated_points"] if points else []) + ["note"]
    show = shares_out[show_cols].astype(object)
    show = show.where(show.notna(), "")
    review = prod[prod["plant_based"].eq("review")]
    water_kg = prod["water_left_out_kg"].sum()
    lines = [f"FoodImpacts: shares of purchased weight -- {base}", "", system_line, "",
             f"Total purchased weight: {shares.loc[0, wcol]:,.1f} {unit}"
             f" ({shares.loc[0, 'weight_kg']:,.1f} kg / {shares.loc[0, 'weight_lbs']:,.1f} lb)", "",
             show.to_string(index=False)]
    if points:
        lines += ["", "Estimated AASHE STARS OP-07 points (from weight shares; STARS scores spend -- see below)", ""]
        lines += [f"  {k}: {v:.2f}" for k, v in points.items()]
    if not lvl_out.empty:
        lv_show = lvl_out[["certification", "food_group", "gfpp_level", wcol, "percent_of_total",
                           "percent_of_food_group", "products"]].astype(object)
        lines += ["", "Weight by GFPP level (percent of total purchased weight; AW also percent of that food "
                      "group's purchases)", "", lv_show.where(lv_show.notna(), "").to_string(index=False)]
    if system == "aia":
        t_show = targets[["measure", "percent", "of", "target_initial", "target_year_3", "target_year_5",
                          "target_this_year", "meets_target"]].astype(object)
        lines += ["", "GFPP targets (Standards 3.0) -- Program year: "
                  + (f"Year {year}" if year is not None else "not known"), ""]
        lines += [f"  Note: {n}" for n in setting_notes] + ([""] if setting_notes else [])
        lines += [t_show.where(t_show.notna(), "").to_string(index=False)]
    if have_kcal and total_kcal:
        cal = shares_out.iloc[1:][["measure", "percent_of_total", "percent_of_kcal"]].dropna(subset=["percent_of_kcal"])
        cal = cal.assign(difference=(cal["percent_of_kcal"] - cal["percent_of_total"]).round(1))
        cal.columns = ["measure", "% of weight", "% of calories", "difference"]
        no_oil = ~prod["cooking_oil"].astype(bool)
        w_no, k_no = prod.loc[no_oil, "weight_kg"].sum(), prod.loc[no_oil, "kcal"].sum()
        pb_w = 100 * prod.loc[pb_mask & no_oil, "weight_kg"].sum() / w_no if w_no else np.nan
        pb_k = 100 * prod.loc[pb_mask & no_oil, "kcal"].sum() / k_no if k_no else np.nan
        oil_k = 100 * prod.loc[~no_oil, "kcal"].sum() / total_kcal
        n_oil = int((~no_oil).sum())
        kept_all = cats[~cats["__drop"]]
        animal = kept_all["__cat"].map(lambda c: c in SHARES_ANIMAL_CATEGORIES or bool(_SHARES_FISH_CAT_RX.search(c)))
        an_w = 100 * kept_all.loc[animal, "__kg"].sum() / kept_all["__kg"].sum() if kept_all["__kg"].sum() else np.nan
        an_k = 100 * kept_all.loc[animal, "__kcal"].sum() / total_kcal
        per = _impacts_per_1000_kcal(base)
        lines += ["", "Shares of calories (for information only: AASHE STARS and Anchors in Action do not use calories)",
                  "", f"Total: {total_kcal / 1e6:,.1f} million kcal ({total_kcal / total:,.0f} kcal per kg purchased)"
                  if total else "", "", cal.to_string(index=False), "",
                  f"Plant-based without cooking oils: {pb_w:.1f}% of weight, {pb_k:.1f}% of calories. Cooking oils "
                  f"(products at least {COOKING_OIL_SHARE:.0%} oil; {n_oil} product{'' if n_oil == 1 else 's'}) are "
                  f"{oil_k:.1f}% of calories; much frying oil is thrown away rather than eaten.",
                  f"Animal-derived ingredients (meat, fish, seafood, dairy, eggs, animal fats): {an_w:.1f}% of weight, "
                  f"{an_k:.1f}% of calories."]
        if per:
            lines.append("Impacts per 1,000 kcal (Stage 3 totals): "
                         + ", ".join(f"{v:,.2f} {u}" for u, v in per.items()) + ".")
    lines += ["", "How the shares were calculated and how unclear cases were handled", ""]
    if system == "stars":
        lines += [
            "- Shares are of purchased weight, not spend. OP-07 7.1 uses the percentage of food and beverage spend, "
            "and 7.2 the percentage of all spend managed by dining services (which may also include supplies, "
            "equipment and services). The points above put the weight shares in place of those spend shares, so "
            "they are estimates only.",
            "- 7.1 points: percent sustainably or ethically produced x 0.06 plus percent plant-based x 0.06, up to 6 "
            "in all. A product can count in both. 7.2: 2 points at 10% or more, 1 point at 5% to under 10%.",
            "- A product counts as sustainably or ethically produced when at least one of its AASHE columns has an "
            "entry on that column's list (the STARS sheet of " + (options_src if options is not None else
            f"certifiers.ods -- not checked: {options_src}; every entry except Avoid was counted") + ": the "
            "certifications in that category's section plus those STARS marks as meeting the category). It counts "
            "once in the 7.1 share however many AASHE columns are filled in. The indented lines show each "
            "category's share; a product meeting several categories counts in each, so they add up to more than "
            "the 7.1 share and earn no points of their own. 'Other' counts and is listed below with its "
            "explanation; 'Seafood Watch - Avoid' and entries not on the column's list do not count and are listed "
            "below.",
            "- A product marked Social Impact Supplier counts once, whichever kind of supplier it is (owned by "
            "people from marginalized groups, employee owned, a social enterprise or a Certified B Corporation).",
            ("- Local: starting with version 3.0 of its standards, AASHE STARS gives no credit for purchases just "
             "because they are local (proximity alone), so the local total does not count toward the estimated "
             "points. It is reported for tracking only." if have["local"] else
             "- Local: starting with version 3.0 of its standards, AASHE STARS gives no credit for purchases just "
             "because they are local (proximity alone). This worksheet has no Local Credit column, so local "
             "purchases are not reported (to track them, make the worksheet with STARS_INCLUDE_LOCAL = True in "
             "add_certifier_dropdowns.ipynb)."),
            f"- stars_inventory_{base}.csv lists every product that counts for 7.1. The STARS inventory reports "
            "spend: add each product's spend from the invoices before uploading it.",
        ]
    else:
        lines.append("- Shares are of purchased weight, not spend. The Anchors in Action (GFPP) framework normally uses "
                     "spend, so these shares are estimates." if system == "aia" else
                     "- Shares are of purchased weight, not spend (AASHE STARS and Anchors in Action normally use spend).")
    lines += [
        (f"- The total leaves out only the water in bottled water and diluted drinks such as soda "
         f"({water_kg:,.1f} kg / {water_kg * KG_TO_LB:,.1f} lb here). Water in soups, juices, plant milks and "
         f"other foods stays in." if "drink_water" in cols else
         "- Water in drinks could not be left out of the total: the files come from an older version "
         "(no drink_water column); re-run from Stage 1."),
        "- A product with an attribute counts with its whole weight; there is no partial credit.",
        ("- Calories: each ingredient's purchased weight times its category's kcal per kg in factors.csv. Bones "
         "and shells are not taken off, and categories with no kcal figure (water, flavoring, additive, unknown) "
         "count as 0. Canned and cooked beans, rice and pasta count as their dry equivalent plus water "
         "(COOKED_DRY_SHARE in Cell 2), broth or stock as a small share of its meat, seafood or vegetable "
         "category plus water (BROTH_EQUIVALENT_SHARE; demi-glace DEMI_GLACE_STOCK_FACTOR times that), and the meat "
         "and seafood in soup bases as at most BASE_ANIMAL_MAX_PERCENT of the base, as in the impacts. Impacts per "
         "1,000 kcal use Stage 3's totals."
         if have_kcal else "- Calories: not reported (factors.csv could not be read)."),
    ]
    if system == "aia":
        lines += [
            "- A product with several attributes counts in each of them. 'Certified (AW, Env or Labor) or "
            "plant-based' is a summary added by FoodImpacts, not a GFPP measure (GFPP scores each value separately): "
            "the share of purchased weight that has at least one of these, with each product counted once, so a "
            "product that is both certified and plant-based is not counted twice. Local, social impact and no "
            "routine antibiotics are not part of it: local and social impact describe the supplier rather than how "
            "the food was produced, and GFPP measures them, and antibiotics, against their own targets.",
            "- A certifier entry counts when certifiers.ods lists it (for AW, in the worksheet's AW food group); "
            "'Other' counts too and is listed below with its explanation. Entries that are not on the list, AW "
            "entries with no AW food group, and Fair Trade International on products other than coffee, sugar "
            "and cocoa (GFPP Standards 3.0 p. 28) are not counted and are listed below.",
            "- Local counts any GFPP level (Level 3 = the smallest suppliers). Social / economic impact and POC "
            "suppliers are shares of local food (p. 18): a supplier marked on a product that is not marked "
            "local is not counted and is listed below. POC suppliers also count as social / economic impact.",
            "- No routine antibiotics (p. 38): y-cert is worked out again from the AW and Env certifiers (column E "
            "of the AIA sheet), because the worksheet's formulas may have no saved value; a y typed in the "
            "column counts as other third-party verification, which the Explanation should name. Only animal "
            "products count: a product whose meat, dairy and egg ingredients are at least half its weight; it "
            "is a chicken product when chicken is at least half of those. Each share is of that group's "
            "purchases. Seafood is not included.",
            "- GFPP targets are compared for the Program year chosen in the worksheet's Program year column; the "
            "Community Health & Nutrition checklist points come from its Nutrition checklist points (%) column. "
            "Each is one value for the whole file, chosen in the first data row (row 2): the first value filled "
            "in is used, and other rows holding a different value are noted above the targets table.",
        ]
    lines += [
        "- Plant-based is judged for each purchased product as a whole, so no part of a product such as "
        "pepperoni pizza counts. A product is plant-based when it has no animal-derived ingredient; minor "
        "additives that vegan products commonly contain from non-animal sources (e.g. vitamin D3) are allowed; "
        "honey, gelatin and the sodium caseinate in most 'non-dairy' creamers are not.",
        "- Each plant-based product also has a processing category, following the NOVA groups used by AASHE "
        "STARS OP-7: 1 = " + PLANT_CATEGORIES[1] + " (produce, plain beans, grains, nuts, spices and spice blends, "
        "tea, coffee); 2 = " + PLANT_CATEGORIES[2] + "; 3 = " + PLANT_CATEGORIES[3] + " (category 1 foods with "
        "only salt, sugar or oil added, e.g. baked beans, sweetened dried fruit, plain frozen fries; 100% juice, "
        "pasta, plain bread, tofu, hummus, nut butters, plant milks, vegan meats and cheeses); 4 = "
        + PLANT_CATEGORIES[4] + ", with additives such as emulsifiers, flavorings, colors or modified starch (most "
        "breakfast cereals, commercial snack chips (chips of plain dried fruit or vegetables are category 1), snack bars, sweets, condiments and sauces, margarine-type spreads, soup bases, "
        "coated fries). "
        f"'Plant-based' here counts categories 1-{max_cat} (PLANT_BASED_MAX_CATEGORY in Cell 1). Categories 1-3 "
        "match STARS v3.0, which recommends counting only foods that clearly qualify; 1-4 also counts all-plant "
        "ultra-processed foods. The category comes from GPT's answer for the whole product; single staples "
        "settled without GPT use a fixed table, and a product with no answer is put in category 4 (3 if labeled "
        "vegan) so that it is counted only under the broadest definition.",
        "- Products made mostly of things that are neither plant nor animal (bottled water, soft drinks, salt, "
        "baking soda) are not plant-based.",
        "- A y in the worksheet's Plant Based (y or 1-4) column counts in every category, and a number 1-4 there sets the "
        "category. Otherwise a vegan / plant-based label means yes, an animal word in the product name means no unless the name marks it as a plant version "
        "(vegan, plant-based, meatless, veggie, '... substitute' or '... alternative', oat milk, peanut butter and so on: "
        "these, such as meatless chicken, are judged like other products and can count), "
        "and other products are judged from their ingredient list and GPT's answer for the whole product.",
        f"- When those two disagree the product is marked 'review' and is NOT counted ({len(review)} products, "
        f"{review[wcol].sum():,.1f} {unit}; listed below). To change a decision, edit plant_based (a category 1-4, "
        f"0 for not plant-based, review, or y) on all rows of the product in ingredients_{base}.csv and re-run with "
        f"START_MODE = \"after_ingredients\" (or in categories_{base}.csv with \"after_categories\").",
        "- If a product's rows have different plant_based values, the most conservative one is used.",
    ]
    if system == "aia":
        lines.append(
            "- GFPP levels come from " + (levels_src if levels is not None else f"nowhere: {levels_src}") + ". "
            "AW levels depend on the food group (Dairy, Eggs, Poultry, Beef or Pork) chosen in the worksheet's "
            "AW food group column; the food group found from the product's ingredients is shown beside it "
            "(aw_food_group_detected) and a difference is listed below.")
        _is_other = lambda c: prod[c].map(lambda v: _gfpp_split(v)[1].casefold() == "other")
        other_rows = prod[_is_other("aw") | _is_other("env") | _is_other("labor")]
        lines += _list("Products with certifier 'Other'", other_rows,
                       [("aw", "AW"), ("env", "Env"), ("labor", "Labor"), ("explanation", "Explanation")])
    elif system == "stars":
        other_rows = prod[prod["stars_other"].ne("")]
        lines += _list("Products with AASHE entry 'Other' (counted; check the explanation)", other_rows,
                       [("stars_other", "Column"), ("explanation", "Explanation")])
        lines += _list("Seafood Watch: Avoid purchases (not counted)",
                       prod[prod["stars_not_counted"].str.contains("Seafood Watch: Avoid", regex=False)],
                       [("explanation", "Explanation")])
        lines += _list("AASHE entries not on their column's list (not counted; check)",
                       prod[prod["stars_not_counted"].str.contains("list)", regex=False)],
                       [("stars_not_counted", "Entry"), ("explanation", "Explanation")])
    else:
        other_rows = prod.iloc[0:0]
    lines += _list("Plant-based 'review' products (not counted)", review, [("pb_basis", "Why")])
    lines += _list("Marked plant-based but an ingredient is in an animal category (check)",
                   prod[prod["plant_based"].map(lambda v: _pb_counted(v, 4)) & prod["animal_categories"].ne("")],
                   [("animal_categories", "Categories"), ("pb_basis", "Basis")])
    lines += _list("Marked in the worksheet's Plant Based (y or 1-4) column although an animal word was found (counted)",
                   prod[prod["pb_basis"].str.contains("animal word", na=False)
                        & prod["plant_based"].map(lambda v: _pb_counted(v, 4))],
                   [("pb_basis", "Note")])
    lines += _list("Products whose rows have different plant_based values (the most conservative value is used)",
                   prod[prod["pb_rows_differ"]], [])
    if system == "aia":
        ns = lambda c: prod[c].str.startswith("not scored")
        lines += _list("Certifier entries not counted (check)", prod[ns("aw_level") | ns("env_level") | ns("labor_level")],
                       [("aw_food_group", "AW food group"), ("aw", "AW"), ("aw_level", "AW"), ("env", "Env"),
                        ("env_level", "Env"), ("labor", "Labor"), ("labor_level", "Labor")])
        if have["aw_food_group"]:
            lines += _list("AW food group differs from the food group found in the ingredients (scored with the "
                           "worksheet's group; check)",
                           prod[prod["aw"].ne("") & prod["aw_food_group"].ne("") & prod["aw_food_group_detected"].ne("")
                                & prod["aw_food_group"].str.casefold().ne(prod["aw_food_group_detected"].str.casefold())],
                           [("aw_food_group", "Worksheet"), ("aw_food_group_detected", "Found"), ("aw", "AW")])
        if have["antibiotics"]:
            typed_y = prod[prod["antibiotics_status"].eq("y (other verification)")].copy()
            typed_y["abx_group"] = typed_y["animal_group"].replace("", "not an animal product (not counted)")
            lines += _list("No routine antibiotics typed y (counted for animal products; the Explanation should "
                           "name the verification)", typed_y, [("abx_group", "Group"), ("explanation", "Explanation")])
            lines += _list("No routine antibiotics marked on a product that is partly animal (under half its weight; "
                           "not counted)", prod[prod["antibiotics_status"].ne("") & prod["animal_group"].eq("")
                                                & prod["animal_share"].gt(0)],
                           [("antibiotics_status", "Entry")])
            lines += _list("No routine antibiotics entries not recognized (not counted)",
                           prod[prod["antibiotics_status"].eq("not recognized")], [("antibiotics", "Entry")])
            lines += _list("Worksheet shows y-cert but neither the AW nor the Env certifier is on the no routine "
                           "antibiotics list (not counted)", prod[prod["antibiotics_cached_differs"]],
                           [("aw", "AW"), ("env", "Env")])
        if have["local"] or have["social_impact"]:
            lines += _list("Local or Social Impact Supplier entries not recognized (not counted)",
                           prod[prod["local_level"].eq("not recognized") | prod["social_impact_kind"].eq("not recognized")],
                           [("local", "Local"), ("social_impact", "Social impact")])
            lines += _list("Social Impact Supplier marked on a product not marked local (not counted; GFPP counts "
                           "these as shares of local food)",
                           prod[prod["social_impact_kind"].isin(["social / economic impact",
                                                                 "POC (also social / economic impact)"])
                                & ~prod["local_counts"].astype(bool)], [("social_impact", "Supplier")])
        lines += ["", f"Product list: product_shares_{base}.csv (filter certified_or_plant_based = y for the "
                      f"purchases that are certified or plant-based)."]
    else:
        lines += ["", f"Product list: product_shares_{base}.csv."]
    rep = result_out(f"shares_report_{base}.txt")
    rep.write_text("\n".join(lines) + "\n", encoding="utf-8")

    print("\n[shares] " + system_line)
    print("[shares] Shares of purchased weight:")
    print(show.to_string(index=False))
    if have_kcal and total_kcal:
        pb_kc = 100 * prod.loc[pb_mask, "kcal"].sum() / total_kcal
        print(f"[shares] Calories: {total_kcal / 1e6:,.1f} million kcal; plant-based {pb_kc:.1f}% of calories "
              f"({100 * prod.loc[pb_mask, 'weight_kg'].sum() / total:.1f}% of weight) -- more in {rep.name}")
    if points:
        print(f"[shares] Estimated OP-07 points (weight in place of spend): "
              f"{points['OP-07 total (up to 8)']:.2f} of 8")
    if len(other_rows):
        print(f"[shares] {len(other_rows)} product(s) with certification 'Other' -- listed with explanations in {rep.name}")
    if len(review):
        print(f"[shares] {len(review)} product(s) marked plant-based 'review' (not counted) -- listed in {rep.name}")
    print(f"[shares] wrote {rep.name}, " + ", ".join(written))
    return shares_out


In [ ]:
# === Cell 9 — Runner / Workflow Controller ===
# Purpose: Orchestrate execution of all pipeline stages based on START_MODE.
# Inputs:
#   - input_filename and configuration from Cell 1
#   - intermediate files (ingredients_, categories_)
# Dependencies:
#   - run_stage_1, run_stage_2, run_stage_3 functions
# Outputs:
#   - Console/log summary of executed stages; the shares report (Cell 8b) after Stage 3
# Behavior:
#   - START_MODE='full' → runs all stages sequentially
#   - START_MODE='after_ingredients' → skips Stage 1
#   - START_MODE='after_categories' → skips Stage 1 and 2
#   - Uses file_type to route to correct Stage 1 function
# Error Handling:
#   - Raises clear messages if required input files or functions are missing.

from pathlib import Path
import sys
import pandas as pd
import inspect  # for safe routing to type03 function

def _die(msg: str):
    print(f"[ERROR] {msg}")
    raise SystemExit("Fix errors above and re-run this cell.")

# --- Normalize/ensure BASE & base_filename consistency
try:
    input_filename
except NameError:
    _die("`input_filename` is not defined in the config cell.")
try:
    base_filename
except NameError:
    base_filename = Path(input_filename).stem
BASE = base_filename

# --- Ensure file_type is resolved (handle 'auto' here before any Stage 1 routing)
try:
    _ft_lower = str(file_type).lower()
except NameError:
    _die("'file_type' is not defined. Run the config/setup cells first.")

if _ft_lower == "auto":
    # We rely on Cell 3's helpers:
    if 'safe_read_csv' not in globals() or 'autodetect_type' not in globals():
        _die("Cannot resolve file_type='auto' because safe_read_csv/autodetect_type are undefined. Run Cell 3.")
    _raw_for_detect = safe_read_csv(input_filename)
    file_type = autodetect_type(_raw_for_detect)
    print(f"[runner] file_type resolved from 'auto' → '{file_type}'")

# --- Provide robust fallbacks for run_stage_1 / run_stage_2 / run_stage_3
if 'run_stage_1' not in globals():
    def run_stage_1(input_filename: str, file_type: str, base_filename: str):
    # Policy (v100g):
    # - Within the SAME kernel session, skip re-running Stage 1 if ingredients_{BASE}.csv was already created.
    # - On a NEW session (after kernel restart), allow overwriting older outputs.
    # - If 'qty' exists in the returned DataFrame, ensure it is preserved to disk.
        if 'load_and_standardize' not in globals():
            _die("`load_and_standardize` is not defined. Ensure the standardization cell ran.")
        if 'extract_ingredients' not in globals():
            _die("`extract_ingredients` is not defined. Ensure the ingredient extraction cell ran.")
        std_df = load_and_standardize(input_filename, file_type)
        ing_df = extract_ingredients(std_df, base_filename)
        return ing_df

if 'run_stage_2' not in globals():
    def run_stage_2(base: str):
        ing_path = result_in(f"ingredients_{base}.csv")
        if not ing_path.exists():
            _die(f"`{ing_path.name}` not found. Run Stage 1 first or check `base_filename`/`BASE`.")
        df_ing = pd.read_csv(ing_path)
        if 'categorize_ingredients' not in globals():
            _die("`categorize_ingredients` is not defined. Ensure the categorization cell ran.")
        df_cat = categorize_ingredients(df_ing, base)
        return df_cat

if 'run_stage_3' not in globals():
    def run_stage_3(base: str, factors_csv: str):
        cats_path = result_in(f"categories_{base}.csv")
        if not cats_path.exists():
            _die(f"`{cats_path.name}` not found. Run Stage 2 first or check `base_filename`/`BASE`.")
        df_cat = pd.read_csv(cats_path)
        if 'calculate_impacts' not in globals():
            _die("`calculate_impacts` is not defined. Ensure the impacts cell ran.")
        return calculate_impacts(df_cat, factors_csv, base)

# --- Strict Stage 1 router (type03 requires Cell 5's GPT path)
def run_stage_1_router(input_filename: str, file_type: str, base_filename: str):
    ft = (file_type or "").lower()
    if ft == "type03":
        if 'run_stage_1_type03_gpt' not in globals():
            _die("type03 file but Cell 5 function not available. Run Cell 5 first.")
        fn = globals()['run_stage_1_type03_gpt']
        try:
            nparams = len(inspect.signature(fn).parameters)
        except Exception:
            nparams = 0
        print("[runner] Stage 1: using type03 GPT path (Cell 5).")
        return fn(input_filename, file_type, base_filename) if nparams >= 3 else fn(input_filename, base_filename)
    print("[runner] Stage 1: using standard path.")
    return run_stage_1(input_filename, file_type, base_filename)

# --- Banner
print("FoodImpacts — runner")
print(f"  input_filename: {Path(input_filename).resolve()}")
print(f"  file_type: {file_type}")
print(f"  base_filename: {base_filename}")
print(f"  results folder: {globals().get('output_dir', '(notebook folder)')}")
print(f"  CATEGORY_SOURCE: {CATEGORY_SOURCE if 'CATEGORY_SOURCE' in globals() else '(unspecified)'}")
print(f"  START_MODE: {START_MODE}")
print(f"  INPUT_WEIGHT_UNIT: {globals().get('INPUT_WEIGHT_UNIT', 'auto')}   OUTPUT_FILES: {globals().get('OUTPUT_FILES', 'both')}")

# --- Execute requested start mode
if START_MODE == "full":
    print("[runner] Stage 1: force recompute ingredients (will overwrite any existing ingredients file).")
    df_ing = run_stage_1_router(input_filename, file_type, base_filename)
    df_cat = run_stage_2(base_filename)
    df_imp_lbs, df_imp_metric = run_stage_3(base_filename, factors_filename)

elif START_MODE == "after_ingredients":
    df_cat = run_stage_2(base_filename)
    df_imp_lbs, df_imp_metric = run_stage_3(base_filename, factors_filename)

elif START_MODE == "after_categories":
    df_imp_lbs, df_imp_metric = run_stage_3(base_filename, factors_filename)

else:
    _die("START_MODE must be 'full', 'after_ingredients', or 'after_categories'")

# --- Shares of purchased weight (Cell 8b); a problem here never stops the run
try:
    run_shares_report(base_filename)
except Exception as e:
    print(f"[shares] Could not write the shares report: {type(e).__name__}: {e}")

print("\n[runner] Done.")